In [362]:
### Creating preliminary functions and dictionaries + package loading 
cell1_start_time = time(); start_load_pkgs = time()
    using Pkg;  
#    Pkg.add("Plots"); using Plots;  
    Pkg.add("JSON3"); using JSON3;
    Pkg.add("Dates"); using Dates;
    Pkg.add("JLD2"); using JLD2
    Pkg.add("HypothesisTests"); using HypothesisTests
    Pkg.add("DataFrames"); using DataFrames
    Pkg.add("CSV"); using CSV
    Pkg.add("XLSX"); using XLSX
    Pkg.add("FASTX"); using FASTX
    Pkg.add("Combinatorics"); using Combinatorics
    Pkg.add("StatsBase"); using StatsBase
    using Statistics
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
println(pwd()); cd("/Users/ryhisner"); println(pwd())
#    Pkg.add("LaTeXStrings");  using LaTeXStrings
#    Pkg.add("PyPlot"); using PyPlot
#    Pkg.add("PlotlyJS"); using PlotlyJS
#    Pkg.add("PGFPlotsX"); using PGFPlotsX
#    Pkg.add("UnicodePlots"); using UnicodePlots
#    Pkg.add("InspectDR"); using InspectDR
#    Pkg.add("GLMakie"); using GLMakie
#    Pkg.add("CairoMakie"); using CairoMakie
#    Pkg.add("WGLMakie"); using WGLMakie
#    Pkg.add("GMT"); using GMT
#####################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_fin:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
######################################################################################################################################
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
function add_zeros_to_rounded(number::Float64, digits_rd::Int)
    if number ≥ 1
        num_final = 0.0
        num_whole = number*10^digits_rd
        num_whole_str = split(string(num_whole), ".")[1]
        len = length(num_whole_str)
        if len ≤ 2
            num_final = "."*"0"^(digits_rd-len)*num_whole_str
        else
            num_final = num_whole_str[1:end-digits_rd]*"."*num_whole_str[end-digits_rd+1:end]
        end
    return num_final
    end
end
#####################################################################################################################################
## Makes all digits go to 2 decimal places so they line up nicely
function number_to_string_to_two_decimal_places(number::Float64, decimals::Int)  
    num_str = string(number)
    before_dec = split(num_str, ".")[1]
    before_dec_str = string(before_dec)
    after_dec = split(num_str, ".")[2]
    after_dec_str = ""
    if length(after_dec) ≥ decimals
        after_dec_str = after_dec[1:decimals]
    else
        after_dec_len = length(after_dec)
        zero_array = Vector{String}()
        missing_zeros_to_fill = decimals - after_dec_len
        for i in 1:missing_zeros_to_fill
            push!(zero_array, "0")
        end
        zeros_str = join(zero_array)
        after_dec_str = after_dec[1:after_dec_len]*zeros_str
    end
    final_num_string = before_dec_str*"."*after_dec_str
    return final_num_string
end
#####################################################################################################################################
load_pkgs_runtime = time() - start_load_pkgs
load_pkgs_hms1, load_pkgs_hms2 = seconds_to_hrs_min_sec(load_pkgs_runtime)
println("Time to Load Packages = ", load_pkgs_hms1); println("Time to Load Packages = ", load_pkgs_hms2)
#####################################################################################################################################
hydrophobic_index_dict = Dict{String, Int}("L"=>97, "I"=>99, "F"=>100, "W"=>97, "V"=>76, "M"=>74, "C"=>49, "Y"=>63, "A"=>41, "T"=>13, "E"=>-31, "H"=>8, "G"=>0, "S"=>-5, "Q"=>-10, "D"=>-55, "R"=>-14, "K"=>-23, "N"=>-28, "P"=>5, "*"=>0)
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
#####################################################################################################################################
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
mut_gene(n) = split(string(n), ":")[1]
######################################################################################################################################
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int, string(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int, string(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int, string(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
####################################################################################
function pango_unaliased_minus_X_fx(unaliased::String, minus::Int)
    dot_ct = count(".", unaliased)
    if dot_ct ≥ minus 
        dotsplits = split(unaliased, ".")
        println(dotsplits)
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X
    else
        return minus_X_pango
    end
end
##########################################################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
wuhan_ref_seq = "..."
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
#ref_AA_dict = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String, String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int,Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int, Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String, Int}()
aa_gene_comprehensive_dict = Dict{String, String}()
aa_gene_and_pos_comprehensive_dict = Dict{String, String}()
refAA_comprehensive_dict = Dict{String, String}()
qryAA_comprehensive_dict = Dict{String, String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
aa_gene_comprehensive_dict["NTD:disulfide"] = "S"
aa_pos_comprehensive_dict["NTD:disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD:disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD:disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD:disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String, String}()
qry_nuc_comprehensive_dict = Dict{String, String}()
nuc_mut_int_comprehensive_dict = Dict{String, Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String, String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_muts_pos_dict = Dict{String, Int}()
NSP_muts_gene_dict = Dict{String, String}()
NSP_ref_AA_dict = Dict{String, String}()
NSP_qry_AA_dict = Dict{String, String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function AAmut_fto_AApos(m)
    gn = split(m, ":")[1]
    mt = split(m, ":")[2]
    pos = mt[2:end-1]
    AApos = gn*":"*pos
    return AApos
end
########################################################################################################################################################################
########################################################################################################################################################################
########################################################################################################################################################################
function mixed2nuc(mix_mut)
    mut = mix_mut
    qry_nuc = qry_nuc_comprehensive_dict[mut]
    ref_nuc = ref_nuc_comprehensive_dict[mut]
    nuc_int = nuc_mut_int_comprehensive_dict[mut]
    ref_n_pos = "$(ref_nuc)$(nuc_int)"
    if length(mix_mut) ≥ 4
        if qry_nuc == "T"
            mut = mix_mut
        elseif qry_nuc == "C"
            mut = mix_mut
        elseif qry_nuc == "A"
            mut = mix_mut
        elseif qry_nuc == "G"
            mut = mix_mut
        elseif qry_nuc == "N"
            mut = mix_mut
        end   
        if ref_nuc == "T"
            if qry_nuc == "Y"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "C"
            if qry_nuc == "Y"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "A"
            if qry_nuc == "R"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "G"
            if qry_nuc == "R"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return mut
end
#####################################################################################################################################
####################### Making gene AA & nuc references for all designated variants #################################################
#####################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String, String}}()
nuc_genome_pango_dict = Dict{String, String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String, String}()
end
################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    if length(nuc_seq) ≥ 28000
        nuc_genome_pango_dict[pango] = nuc_seq
    end
end
seqs_in_nuc_genome_pango_dict = length(nuc_genome_pango_dict)
println("seqs_in_nuc_genome_pango_dict = $(seqs_in_nuc_genome_pango_dict)")
######################################################################################################################################
######################################################################################################################################
gene_hydrophobic_dict = Dict{String, Float64}()
for (gene, aaseq) in gene_AA_dict
    gene_hydrophobe_sum = 0
    for aa in aaseq
        hydrophobe_score = hydrophobic_index_dict[string(aa)]
        gene_hydrophobe_sum += hydrophobe_score
    end
    gene_hydrophobe_score = gene_hydrophobe_sum/length(aaseq)
    gene_hydrophobic_dict[gene] = gene_hydrophobe_score
end 
######################################################################################################################################
AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*"])
AA_res_set_noDel = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "*"])
AA_res_pairs = Set{Tuple{String, String}}()
for res1 in AA_res_set_noDel
    for res2 in AA_res_set
        push!(AA_res_pairs, (res1, res2) )
    end
end
###########################################################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
############################################################################################################################################################################
############################################################################################################################################################################
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int,Int, Int}}()
tuple_to_index = Dict{Tuple{Int,Int, Int}, Int}()        
###########################################################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
############################################################################################################################################################################
############################################################################################################################################################################
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{String, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{String, Int}()
        end
    end
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function nuc_mut_pos_only_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{Int, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{Int, Int}()
        end
    end
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function ORF1abMut_to_NSP(mut::String)
    NSPmut = ""
#    NSP_muts = Dict("NSP$(i)" => Dict{String, Int}() for i in 1:16 if i ≠ 11)
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    refAA = refAA_comprehensive_dict[mut]
    qryAA = qryAA_comprehensive_dict[mut]
    if gene == "ORF1a"
        for (NSP, range) in NSP_ranges1a
            if pos in range
                NSPpos = pos - NSP1a_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    if gene == "ORF1b"
        for (NSP, range) in NSP_ranges1b
            if pos in range
                NSPpos = pos - NSP1b_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    return NSPmut
end
#####################################################################################################################################
function NSPmut_to_ORF1ab(NSPmut::String)
    ORFmut = ""
    ORFpos = ""
    NSP_num = parse(Int, split(NSPmut, ":")[1][4:end])
    NSP_pos = NSP_muts_pos_dict[NSPmut]
    refAA = NSP_ref_AA_dict[NSPmut]
    qryAA = NSP_qry_AA_dict[NSPmut]
    if NSPnum in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
        ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSPnum in [13, 14, 15, 16]
        ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
        ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSP_num == 12
        if NSP_pos in [1:8...]
            ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
            ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
        end
        if NSP_pos in [10:932...]
            ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
            ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
        end
    end
    ORF1pos = parse(Int, ORFpos)
    return ORFmut
end
#####################################################################################################################################
function NSP_range_to_ORF1ab_range(NSP_range::String)
    NSP = split(NSP_range, ":")[1]
    if length(NSP) < 3
        return ""
    end
    if NSP[1:3] ≠ "NSP"
        return ""
    end
    NSP_int = parse(Int, NSP[4:end])
    range = split(NSP_range, ":")[2]
    NSPpos1_int = parse(Int, split(range, "-")[1])
    NSPpos2_int = parse(Int, split(range, "-")[2])
    ORF_int1 = 0
    ORF_int2 = 0
    ORF1_AorB_pos1 = ""
    ORF1_AorB_pos2 = ""
    if NSP_int in [1:10...]
        ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1a"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int in [13:16...]
        ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1b"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int == 12
        if NSPpos1_int in [1:9...]
            ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos1 = "1a"
        else
            ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
            ORF1_AorB_pos1 = "ORF1b"
        end
        if NSPpos2_int in [1:9...]
            ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos2 = "1a"
        end
        if !(NSPpos2_int in [1:9...])
            ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
            if ORF1_AorB_pos1 == "1a"
                ORF1_AorB_pos2 = "1b:"
            else
            ORF1_AorB_pos2 = ""
            end
        end
    end
    ORF_int1_str = string(ORF_int1)
    ORF_int2_str = string(ORF_int2)
    ORF_range = ORF1_AorB_pos1*":"*ORF_int1_str*"-"*ORF1_AorB_pos2*ORF_int2_str
    return ORF_range
end
#####################################################################################################################################
function ORF1ab_range_to_NSP_range(ab_range::String)
    ab = split(ab_range, ":")[1]
    range = split(ab_range, ":")[2]
    pos1 = split(range, "-")[1]
    pos2 = split(range, "-")[2]
    pos1_int = parse(Int, pos1)
    pos2_int = parse(Int, pos2)
    NSPint1 = 0
    NSPint2 = 0
    NSPpos1 = ""
    NSPpos2 = ""
    NSPrange = ""
    NSPrange_pt1 = ""
    top_ct = 0
    if ab == "ORF1a"
        ct = 0
        for (NSP, rng) in NSP_ranges1a
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPint2 = pos2_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1a_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end
    if ab == "ORF1b"
        ct = 0
        for (NSP, rng) in NSP_ranges1b
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPint2 = pos2_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1b_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end 
    return NSPrange
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
######################################################################################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))    
    return sort_arr_of_strings2
end
######################################################################################################################################
function stringlist_to_strings_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end  
######################################################################################################################################
function stringlist_to_set(txt::String)
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end   
    return set_of_strings
end
#######################################################################################################################################
function list_to_strings(list::String)
    string_vec = string.(split(list, ", "))
    return string_vec
end
##################################################
function list_to_strings_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
#####################################################################################################################################
function cross_check_set(old_seq_set::Set{String}, new_seq_set::Set{String})
    new_set2 = Set{String}()
    old_set2 = Set{String}()
    old_seq_set_len = length(old_seq_set)
    new_seq_set_len = length(new_seq_set)
    difference = new_seq_set_len - old_seq_set_len
    println("Number of Sequences in Old List = $(old_seq_set_len)")
    println("Number of Sequences in New List = $(new_seq_set_len)")
    println("Difference between Old & New Sets = $(difference)")
    println()
    for seq in new_seq_set
        if !(seq in old_seq_set)
            push!(new_set2, seq)
        end
        if seq in old_seq_set
            push!(old_set2, seq)
        end
    end
    new_len = length(new_set2)
    old_len = length(old_set2)
    if difference == new_len
        println("The numbers check out: difference between old & new sets = Number of new seqs")
    else
        println("The numbers don't check out: difference between old & new sets not equal to Number of new seqs")
    end
    println()
    println("Number of New Sequences = $(new_len)")
    println("Number of Old Sequences = $(old_len)")
    new_set2_sort = sort(collect(new_set2), by = x -> (length(x), x))
    return new_set2, new_set2_sort
end
############################################################################################################################################################################
############################################################################################################################################################################
function mut_to_mutpos_surround_int(mut::String, plus_or_minus::Int)
    gene = aa_gene_comprehensive_dict[mut]
    pos_int = aa_pos_comprehensive_dict[mut]
    mutpos = aa_gene_and_pos_comprehensive_dict[mut]
#    gene = split(mut, ":")[1]
#    pos = split(mut, ":")[2][2:end-1]
#    mutpos = gene*":"*pos
#    pos_int = parse(Int, pos)
    mut_pos_close_vec = String[]
    for i in -plus_or_minus:plus_or_minus
#        if i ≠ 0
            pos_close = pos_int + i
            mutpos_close = gene*":"*string(pos_close)
            push!(mut_pos_close_vec, mutpos_close)
#        end
    end
    return mut_pos_close_vec
end
############################################################################################################################################################################
############################################################################################################################################################################
function round_addDecimal_pad(number::Float64, rd_digits::Int, zero_ct::Int, pad::Int)
    number_rd = round(digits=rd_digits, number)
    number_2dec = add_zeros_to_rounded(number_rd, rd_digits)
    number_pad = lpad(number_2dec, pad)
    return number_pad
end
function round_addDecimal_pad_ANTI(number::Float64, rd_digits::Int, zero_ct::Int, pad::Int)
    number_rd = round(digits=rd_digits, number)
    number_2dec = add_zeros_to_rounded(number_rd, rd_digits)
    number_pad = lpad(number_2dec, pad)
    return number_pad
end
############################################################################################################################################################################
############################################################################################################################################################################
### Fx: get_pango_WT_muts_sort
function get_pango_WT_muts_sort(pango::String)
    fx_pango_AAsub_WT_sort = sort(collect(pango_AAsub_WT[pango]), by = x -> gene___AAnum_sortkey(x))
    fx_pango_AAsub_WT_sort_join = join(fx_pango_AAsub_WT_sort, ", ")
    println(fx_pango_AAsub_WT_sort_join)
    print("\n"^1)
    fx_pango_AAdel_private_sort = sort(collect(pango_AAdel_private[pango]), by = x -> gene___AAnum_sortkey(x))
    fx_pango_AAdel_private_sort_join = join(fx_pango_AAdel_private_sort, ", ")
    println(fx_pango_AAdel_private_sort_join)
end
############################################################################################################################################################################
############################################################################################################################################################################
cell1_runtime = round(digits=1, time() - cell1_start_time)
c1runtime1, c1runtime2 = seconds_to_hrs_min_sec(cell1_runtime)
println("Cell1 Runtime v0 = $(cell1_runtime) seconds")
println("Cell1 Runtime v1 = $(c1runtime1)")
println("Cell1 Runtime v2 = $(c1runtime2)"); println()
######################################################################################################################################
######################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

2026_03_04__612PM
/Users/ryhisner
/Users/ryhisner
Time to Load Packages = 0:00:13.00
Time to Load Packages = 0 hr, 0 min, 13.00 sec
seqs_in_nuc_genome_pango_dict = 3586
Cell1 Runtime v0 = 45.8 seconds
Cell1 Runtime v1 = 0:00:45.80
Cell1 Runtime v2 = 0 hr, 0 min, 45.80 sec



In [363]:
### Fx: load_all_seq_dicts, 2026_03_08   |   Loads HQCS Dataset   | many large dictionaries not needed & therefore commented out  
load_all_start = time();   date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function load_all_seq_dicts(date::String, fx_name::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int)
    start_all = time()
    filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
############################################################################################################################################################################    
    global seq_ct_by_year_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
############################################################################################################################################################################
    global avg_private_AA_per_circ_seq = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", "avg_private_AA_per_circ_seq")
    global all_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", "all_seq_ct")
    global qualifying_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", "qualifying_seq_ct")
    global nuc_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
    global AA_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct")
    global AA_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
    global AA_muts_ct_pos_only_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_muts_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global gene_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density")
    global nuc_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", "nuc_muts_ct_sort")
    global nuc_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global nuc_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", "nuc_muts_ct_no_dels_sort")
    global nuc_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", "AA_muts_ct_sort")
    global AA_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_pos_only_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", "AA_muts_ct_pos_only_sort")
    global AA_muts_ct_pos_only_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", "AA_muts_ct_no_dels_sort")
    global AA_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", "AA_muts_ct_pos_only_no_dels_sort")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")
    global gene_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global domain_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global domain_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global AA_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq")
    global nuc_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct")
############################################################################################################################################################################
    global AA_no_dels_sub_count_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", "AA_no_dels_sub_count")
    global total_AA_subs_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", "total_AA_subs")
#    global date_nuc_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
#    global date_nuc_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
#    global date_AA_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct")
#    global date_AA_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
#    global date_AA_mut_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
#    global seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
    global clade_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct")
############################################################################################################################################################################
######################################################## Below: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
## Changed mind about pango_date_index_ct--better to have high qc filters in place for those to weed out shitty misassigned seqs
#    global pango_date_index_ct = load("2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut7_maxRevs2_qcMax10_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global pango_date_index_ct = load("2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_pango_all = load("dictionaries/2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_seq_pango.jld2", "seq_pango")



    
#####################################################################################################################################
    global country_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_country_set.jld2", "country_set")
    global clade_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_clade_set.jld2", "clade_set")
    global pango_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_pango_set.jld2", "pango_set")
#    global seq_lab_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################
    
    
    
    
    #    date_temp = "2025-08-03"; max_AA_mut_temp = 5; revs_thresh_temp = 1; qc_max_temp = 5; ndjson_name_temp = "EPI_ISL_400000_20080000"
#    global country_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_set.jld2", "country_set")
#    global clade_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_set.jld2", "clade_set")
#    global pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_set.jld2", "pango_set")
#    global pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_set.jld2", "pango_unaliased_set")
#    global pango_to_pango_unaliased = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_to_pango_unaliased.jld2", "pango_to_pango_unaliased")
#    global clade_pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_set.jld2", "clade_pango_set")
#    global clade_pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_unaliased_set.jld2", "clade_pango_unaliased_set")
#####################################################################################################################################
#    global seq_country_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_country.jld2", "seq_country")
#    global seq_US_state_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_US_state.jld2", "seq_US_state")
#    global seq_clade_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_clade.jld2", "seq_clade")
############ Currently using low-qc seq_pango_all as it's more comprehensive ###########
#    global seq_pango_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango.jld2", "seq_pango")
#    global seq_pango_unaliased_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango_unaliased.jld2", "seq_pango_unaliased")
############ Currently using low-qc seq_date_index_all as it's more comprehensive ########################
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_collection_date_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_collection_date.jld2", "seq_collection_date")
#    global seq_lab_dict_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_dict.jld2", "seq_lab_dict")
#    global seq_nuc_del_ranges_ct_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_nuc_del_ranges_ct.jld2", "seq_nuc_del_ranges_ct")
#    global seq_lab_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################    
#    global pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
#    global country_clade_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct")
#    global country_pango_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct")
#    global country_pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct")
############################################################################################################################################################################
######################################################## Above: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
#    global ORF9b_CTD_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq")
    global multi_ORF9b_CTD_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD")
    global multi_ORF9b_CTD_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", "multi_ORF9b_CTD_ct")
#    global qualifying_ORF9b_double_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", "qualifying_ORF9b_double_ct")
    
#    global ORF9b_CTD_muts_seq_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_ORF9b_CTD_muts_seq_relaxed_qc_10_1_30.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_ct_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_10_1_30")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_qualifying_ORF9b_double_ct_relaxed_qc_10_1_30.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_10_1_30")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_ORF9b_CTD_muts_seq_relaxed_qc_15_1_50.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_ct_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_50_1_15")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_qualifying_ORF9b_double_ct_relaxed_qc_15_1_50.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_50_1_15")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_ORF9b_CTD_muts_seq_relaxed_qc_25_3_200.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_ct_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_200_3_25")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_qualifying_ORF9b_double_ct_relaxed_qc_25_3_200.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_200_3_25")
#    global avg_private_AA_per_circ_seq2 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", "avg_private_AA_per_circ_seq2")
    ##################### Below: Dicts for all seqs using higher (less restrictive) QC filters ##########################################
#    global AA_muts_seq_all_8000qc_filter = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_muts_seq_all_999qc_filter = load("dictionaries/2025-08-24_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs4_qcMax999_AA_muts_seq.jld2", "AA_muts_seq")
#####################################################################################################################################
    println("Dictionaries loaded!");   println()
    finish_all = time() - start_all;    finish_all_rd = round(digits=1, finish_all)
    load_hms1, load_hms2 = seconds_to_hrs_min_sec(finish_all)
    println("Total Time to Load ALL Dictionaries = $(finish_all_rd) seconds")  # println("Total Time to Load ALL Dictionaries = $(load_hms1)")
    println("Total Time to Load ALL Dictionaries = $(load_hms2)")
end
######################################################################################################################################
date = "2026_02_08"
clade_pct_date_index = load("dictionaries/$(date)__clade_pct_date_index.jld2", "clade_pct_date_index")
clade_pct_date_tuple = load("dictionaries/$(date)__clade_pct_date_tuple.jld2", "clade_pct_date_tuple")
clade_pct_date_string = load("dictionaries/$(date)__clade_pct_date_string.jld2", "clade_pct_date_string")
pango_pct_date_index = load("dictionaries/$(date)__pango_pct_date_index.jld2", "pango_pct_date_index")
pango_pct_date_tuple = load("dictionaries/$(date)__pango_pct_date_tuple.jld2", "pango_pct_date_tuple")
pango_pct_date_string = load("dictionaries/$(date)__pango_pct_date_string.jld2", "pango_pct_date_string")
pango_unaliased_pct_date_index = load("dictionaries/$(date)__pango_unaliased_pct_date_index.jld2", "pango_unaliased_pct_date_index")
pango_unaliased_pct_date_tuple = load("dictionaries/$(date)__pango_unaliased_pct_date_tuple.jld2", "pango_unaliased_pct_date_tuple")
pango_unaliased_pct_date_string = load("dictionaries/$(date)__pango_unaliased_pct_date_string.jld2", "pango_unaliased_pct_date_string")
pango_to_pango_unaliased_v2 = load("dictionaries/$(date)__pango_to_pango_unaliased_v2.jld2", "pango_to_pango_unaliased_v2")
pango_unaliased_to_pango_prefix = load("dictionaries/$(date)__pango_unaliased_to_pango_prefix.jld2", "pango_unaliased_to_pango_prefix")
pango_unaliased_to_pango = load("dictionaries/$(date)__pango_unaliased_to_pango.jld2", "pango_unaliased_to_pango")
pango_unaliased_predecessor_meta_dict = load("dictionaries/$(date)__pango_unaliased_predecessor_meta_dict.jld2", "pango_unaliased_predecessor_meta_dict")
pango_predecessor_meta_dict = load("dictionaries/$(date)__pango_predecessor_meta_dict.jld2", "pango_predecessor_meta_dict")
pango_unaliased_set = load("dictionaries/$(date)__pango_unaliased_set.jld2", "pango_unaliased_set")
pango_unaliased_date_index_ct = load("dictionaries/$(date)__pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
clade_date_index_cumul = load("dictionaries/$(date)__clade_date_index_cumul.jld2", "clade_date_index_cumul")
pango_date_index_cumul = load("dictionaries/$(date)__pango_date_index_cumul.jld2", "pango_date_index_cumul")
pango_unaliased_date_index_cumul = load("dictionaries/$(date)__pango_unaliased_date_index_cumul.jld2", "pango_unaliased_date_index_cumul")
clade_total = load("dictionaries/$(date)__clade_total.jld2", "clade_total")
pango_total = load("dictionaries/$(date)__pango_total.jld2", "pango_total")
pango_unaliased_total = load("dictionaries/$(date)__pango_unaliased_total.jld2", "pango_unaliased_total")
AAmut_date_index_cumul = load("dictionaries/$(date)__AAmut_date_index_cumul.jld2", "AAmut_date_index_cumul")
nucmut_date_index_cumul = load("dictionaries/$(date)__nucmut_date_index_cumul.jld2", "nucmut_date_index_cumul")
pango_nuc_sub_WT = load("dictionaries/$(date)__pango_nuc_sub_WT.jld2", "pango_nuc_sub_WT")
pango_nuc_del_WT = load("dictionaries/$(date)__pango_nuc_del_WT.jld2", "pango_nuc_del_WT")
pango_nuc_sub_private = load("dictionaries/$(date)__pango_nuc_sub_private.jld2", "pango_nuc_sub_private")
pango_nuc_del_private = load("dictionaries/$(date)__pango_nuc_del_private.jld2", "pango_nuc_del_private")
pango_nuc_sub_revs = load("dictionaries/$(date)__pango_nuc_sub_revs.jld2", "pango_nuc_sub_revs")
pango_nuc_del_revs = load("dictionaries/$(date)__pango_nuc_del_revs.jld2", "pango_nuc_del_revs")
pango_AAsub_WT = load("dictionaries/$(date)__pango_AAsub_WT.jld2", "pango_AAsub_WT")
pango_AAsub_WT_pos_only = load("dictionaries/$(date)__pango_AAsub_WT_pos_only.jld2", "pango_AAsub_WT_pos_only")
pango_AAdel_WT = load("dictionaries/$(date)__pango_AAdel_WT.jld2", "pango_AAdel_WT")
pango_AAsub_private = load("dictionaries/$(date)__pango_AAsub_private.jld2", "pango_AAsub_private")
pango_AAdel_private = load("dictionaries/$(date)__pango_AAdel_private.jld2", "pango_AAdel_private")
pango_AAsub_revs = load("dictionaries/$(date)__pango_AAsub_revs.jld2", "pango_AAsub_revs")
pango_AAdel_revs = load("dictionaries/$(date)__pango_AAdel_revs.jld2", "pango_AAdel_revs")
pango_frameshifts_WT = load("dictionaries/$(date)__pango_frameshifts_WT.jld2", "pango_frameshifts_WT")
pango_designation_date = load("dictionaries/$(date)__pango_designation_date.jld2", "pango_designation_date")
clade_pango_set = load("dictionaries/$(date)__clade_pango_set.jld2", "clade_pango_set")
delete!(pango_AAsub_WT["B.55"], "")
delete!(pango_AAsub_WT["B"], "")
println("Done!")
load_all_fx_runtime = time() - load_all_start; load_all_fx_runtime_rd = round(digits=3, load_all_fx_runtime)
load_all_fx_hms1, load_all_fx_hms2 = seconds_to_hrs_min_sec(load_all_fx_runtime)
println("Total Time to Run load_all Fx = $(load_all_fx_runtime_rd) seconds")  # println("Total Time to Run load_all Fx = $(load_all_fx_hms1)") 
println("Total Time to Run load_all Fx = $(load_all_fx_hms2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_04__612PM
Done!
Total Time to Run load_all Fx = 7.448 seconds
Total Time to Run load_all Fx = 0 hr, 0 min, 07.45 sec



In [364]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--5-1-5 | Runtime = 38 sec #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_load_all_dicts = time()
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 5
HQCS_revs_thresh = 1
HQCS_qc_max = 5
HQCS_qc_str = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int, Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int, Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_mut_int_comprehensive_dict[nuc_mut]
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
#############################################__Create   pango_index_date_set    ###################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println(length(pango_index_date_set))
#################################################################################################################################
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
#################################################################################################################################
#################################################################################################################################

2026_03_04__612PM
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 49.6 seconds
Total Time to Load ALL Dictionaries = 0 hr, 0 min, 49.61 sec
Total Time to Load ALL Dictionaries = 49.7
4297
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.08 seconds



In [365]:
### Fx: chronic_load_dicts2_DQ, 2026_03_01 version |  Loads EPCI dataset  | ###############################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_chr_load_fx = time()
function chronic_load_dicts2_DQ(ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Int, abs_min_mut_thresh::Int)
    start_date = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(start_date)
######################################################################################################################################
#    global cumulative_AA_mut_ct_by_date_index
#        cumulative_AA_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_AA_mut_ct_by_date_index.jld2", "cumulative_AA_mut_ct_by_date_index")
#    global cumulative_nuc_mut_ct_by_date_index
#        cumulative_nuc_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_nuc_mut_ct_by_date_index.jld2", "cumulative_nuc_mut_ct_by_date_index")
######################################################################################################################################
    global seq_AA_insertions_WT
        seq_AA_insertions_WT = load("$(folder_name)/$(folder_name)__seq_AA_insertions_WT.jld2", "seq_AA_insertions_WT")
    global seq_nuc_insertions_WT
        seq_nuc_insertions_WT = load("$(folder_name)/$(folder_name)__seq_nuc_insertions_WT.jld2", "seq_nuc_insertions_WT")    
######################################################################################################################################
    global total_chr_AA_subs
        total_chr_AA_subs = load("$(folder_name)/$(folder_name)__total_chr_AA_subs.jld2", "total_chr_AA_subs")
######################################################################################################################################
    global total_nuc_revs_seq
        total_nuc_revs_seq = load("$(folder_name)/$(folder_name)__total_nuc_revs_seq.jld2", "total_nuc_revs_seq")
    global seq_nuc_total_revs
        seq_nuc_total_revs = load("$(folder_name)/$(folder_name)__seq_nuc_total_revs.jld2", "seq_nuc_total_revs")
    global total_AA_revs_seq
        total_AA_revs_seq = load("$(folder_name)/$(folder_name)__total_AA_revs_seq.jld2", "total_AA_revs_seq")
    global seq_AA_total_revs
        seq_AA_total_revs = load("$(folder_name)/$(folder_name)__seq_AA_total_revs.jld2", "seq_AA_total_revs")
    global seq_AA_revs
        seq_AA_revs = load("$(folder_name)/$(folder_name)__seq_AA_revs.jld2", "seq_AA_revs")
######################################################################################################################################
    global AA_muts_ct_chr_all_ratio
        AA_muts_ct_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio.jld2", "AA_muts_ct_chr_all_ratio")
    global AA_muts_ct_no_dels_chr_all_ratio
        AA_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio.jld2", "AA_muts_ct_no_dels_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio")
    global AA_muts_ct_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_no_dels_no_revs_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
    global nuc_muts_ct_no_dels_chr_all_ratio
        nuc_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio")
    global nuc_muts_ct_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_no_revs_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
####################################################################################################################################
    global AA_muts_ct_pos_only_adj_score
        AA_muts_ct_pos_only_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score.jld2", "AA_muts_ct_pos_only_adj_score")
    global AA_muts_ct_adj
        AA_muts_ct_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj.jld2", "AA_muts_ct_adj")
    global AA_muts_ct_pos_only_adj_score_no_dels
        AA_muts_ct_pos_only_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels.jld2", "AA_muts_ct_pos_only_adj_score_no_dels")
    global nuc_muts_ct_adj
        nuc_muts_ct_adj = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj.jld2", "nuc_muts_ct_adj")
    global AA_muts_ct_adj_score
        AA_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score.jld2", "AA_muts_ct_adj_score")
    global AA_muts_ct_adj_score_no_dels
        AA_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels.jld2", "AA_muts_ct_adj_score_no_dels")
    global AA_muts_ct_pos_only_adj
        AA_muts_ct_pos_only_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj.jld2", "AA_muts_ct_pos_only_adj")
    global AA_muts_ct_pos_only_adj_no_dels
        AA_muts_ct_pos_only_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels.jld2", "AA_muts_ct_pos_only_adj_no_dels")
    global nuc_muts_ct_adj_score_no_dels
        nuc_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels.jld2", "nuc_muts_ct_adj_score_no_dels")
    global nuc_muts_ct_adj_score
        nuc_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score.jld2", "nuc_muts_ct_adj_score")
    global nuc_muts_ct_adj_no_dels
        nuc_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels.jld2", "nuc_muts_ct_adj_no_dels")
    global AA_muts_ct_adj_no_dels
        AA_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels.jld2", "AA_muts_ct_adj_no_dels")
####################################################################################################################################
####################################################################################################################################
    global seq_ct_by_year
        seq_ct_by_year = load("$(folder_name)/$(folder_name)__seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month
        seq_ct_by_year_month = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day
        seq_ct_by_year_month_day = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global seq_date_tuple
        seq_date_tuple = load("$(folder_name)/$(folder_name)__seq_date_tuple.jld2", "seq_date_tuple")
    global date_nuc_mut_ct_no_dels
        date_nuc_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
    global date_AA_mut_ct_no_dels
        date_AA_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
    global date_AA_mut_ct
        date_AA_mut_ct = load("$(folder_name)/$(folder_name)__date_AA_mut_ct.jld2", "date_AA_mut_ct")
    global date_AA_mut_ct_pos_only_no_dels
        date_AA_mut_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
####################################################################################################################################
    global seq_collection_date
        seq_collection_date = load("$(folder_name)/$(folder_name)__seq_collection_date.jld2", "seq_collection_date")
    global seq_date_index
        seq_date_index = load("$(folder_name)/$(folder_name)__seq_date_index.jld2", "seq_date_index")
    global date_nuc_mut_ct
        date_nuc_mut_ct = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
####################################################################################################################################
    global seq_AA_dels
        seq_AA_dels = load("$(folder_name)/$(folder_name)__seq_AA_dels.jld2", "seq_AA_dels")
    global seq_AA_muts_pos_only_no_dels
        seq_AA_muts_pos_only_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only_no_dels.jld2", "seq_AA_muts_pos_only_no_dels")
    global seq_AA_muts_WT_pos_only
        seq_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT_pos_only.jld2", "seq_AA_muts_WT_pos_only")
    global seq_AA_dels_WT
        seq_AA_dels_WT = load("$(folder_name)/$(folder_name)__seq_AA_dels_WT.jld2", "seq_AA_dels_WT")
    global seq_AA_muts_WT
        seq_AA_muts_WT = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT.jld2", "seq_AA_muts_WT")
    global seq_AA_muts
        seq_AA_muts = load("$(folder_name)/$(folder_name)__seq_AA_muts.jld2", "seq_AA_muts")
    global seq_AA_muts_no_dels
        seq_AA_muts_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_no_dels.jld2", "seq_AA_muts_no_dels")
    global seq_AA_muts_pos_only
        seq_AA_muts_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only.jld2", "seq_AA_muts_pos_only")
    global seq_mixed_AA_muts
        seq_mixed_AA_muts = load("$(folder_name)/$(folder_name)__seq_mixed_AA_muts.jld2", "seq_mixed_AA_muts")
    global seq_unknown_AA
        seq_unknown_AA = load("$(folder_name)/$(folder_name)__seq_unknown_AA.jld2", "seq_unknown_AA")
    global seq_unknown_AA_ranges
        seq_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__seq_unknown_AA_ranges.jld2", "seq_unknown_AA_ranges")
####################################################################################################################################
    global seq_nuc_muts
        seq_nuc_muts = load("$(folder_name)/$(folder_name)__seq_nuc_muts.jld2", "seq_nuc_muts")
    global seq_nuc_muts_WT
        seq_nuc_muts_WT = load("$(folder_name)/$(folder_name)__seq_nuc_muts_WT.jld2", "seq_nuc_muts_WT")
    global seq_nuc_del_ranges_WT
        seq_nuc_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges_WT.jld2", "seq_nuc_del_ranges_WT")
    global seq_nuc_dropout
        seq_nuc_dropout = load("$(folder_name)/$(folder_name)__seq_nuc_dropout.jld2", "seq_nuc_dropout")
    global seq_nuc_del_ranges
        seq_nuc_del_ranges = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges.jld2", "seq_nuc_del_ranges")
    global seq_mixed_nucs
        seq_mixed_nucs = load("$(folder_name)/$(folder_name)__seq_mixed_nucs.jld2", "seq_mixed_nucs")
###################################################################################################################################
    global seq_clade
        seq_clade = load("$(folder_name)/$(folder_name)__seq_clade.jld2", "seq_clade")
    global seq_pango
        seq_pango = load("$(folder_name)/$(folder_name)__seq_pango.jld2", "seq_pango")
    global seq_pango_unaliased
        seq_pango_unaliased = load("$(folder_name)/$(folder_name)__seq_pango_unaliased.jld2", "seq_pango_unaliased")    
    global seq_clade_display
        seq_clade_display = load("$(folder_name)/$(folder_name)__seq_clade_display.jld2", "seq_clade_display")
    global seq_clade_ct
        seq_clade_ct = load("$(folder_name)/$(folder_name)__seq_clade_ct.jld2", "seq_clade_ct")
    global seq_pango_ct
        seq_pango_ct = load("$(folder_name)/$(folder_name)__seq_pango_ct.jld2", "seq_pango_ct")
    global seq_pango_unaliased_ct
        seq_pango_unaliased_ct = load("$(folder_name)/$(folder_name)__seq_pango_unaliased_ct.jld2", "seq_pango_unaliased_ct")
####################################################################################################################################
    global seq_US_state
        seq_US_state = load("$(folder_name)/$(folder_name)__seq_US_state.jld2", "seq_US_state")
    global seq_country
        seq_country = load("$(folder_name)/$(folder_name)__seq_country.jld2", "seq_country")
    global seq_lab_dict
        seq_lab_dict = load("$(folder_name)/$(folder_name)__seq_lab_dict.jld2", "seq_lab_dict")
####################################################################################################################################
    global gene_mut_density
        gene_mut_density = load("$(folder_name)/$(folder_name)__gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density
        domain_mut_density = load("$(folder_name)/$(folder_name)__domain_mut_density.jld2", "domain_mut_density")
####################################################################################################################################
    global nuc_muts_ct
        nuc_muts_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_dels_ct
        nuc_dels_ct = load("$(folder_name)/$(folder_name)__nuc_dels_ct.jld2", "nuc_dels_ct")
    global nuc_muts_ct_no_dels
        nuc_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
######################
    global nuc_dels_seq
        nuc_dels_seq = load("$(folder_name)/$(folder_name)__nuc_dels_seq.jld2", "nuc_dels_seq")
    global nuc_muts_seq
        nuc_muts_seq = load("$(folder_name)/$(folder_name)__nuc_muts_seq.jld2", "nuc_muts_seq")
    global nuc_dels_seq_WT
        nuc_dels_seq_WT = load("$(folder_name)/$(folder_name)__nuc_dels_seq_WT.jld2", "nuc_dels_seq_WT")
    global nuc_muts_seq_WT
        nuc_muts_seq_WT = load("$(folder_name)/$(folder_name)__nuc_muts_seq_WT.jld2", "nuc_muts_seq_WT")
##################################################################
    global AA_muts_ct
        AA_muts_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct.jld2", "AA_muts_ct")
    global AA_muts_ct_no_dels_no_revs
        AA_muts_ct_no_dels_no_revs = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs.jld2", "AA_muts_ct_no_dels_no_revs")
    global AA_muts_ct_pos_only
        AA_muts_ct_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_dels_ct
        AA_dels_ct = load("$(folder_name)/$(folder_name)__AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_pos_only_no_dels
        AA_muts_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global AA_muts_ct_no_dels
        AA_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
############################## 
    global AA_muts_seq
        AA_muts_seq = load("$(folder_name)/$(folder_name)__AA_muts_seq.jld2", "AA_muts_seq")
    global AA_muts_seq_pos_only
        AA_muts_seq_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only.jld2", "AA_muts_seq_pos_only")
    global AA_muts_seq_pos_only_no_dels
        AA_muts_seq_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only_no_dels.jld2", "AA_muts_seq_pos_only_no_dels")
    global AA_dels_seq
        AA_dels_seq = load("$(folder_name)/$(folder_name)__AA_dels_seq.jld2", "AA_dels_seq")
##############################
    global AA_muts_seq_WT
        AA_muts_seq_WT = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT.jld2", "AA_muts_seq_WT")
    global AA_muts_seq_WT_pos_only
        AA_muts_seq_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT_pos_only.jld2", "AA_muts_seq_WT_pos_only")
    global AA_dels_seq_WT
        AA_dels_seq_WT = load("$(folder_name)/$(folder_name)__AA_dels_seq_WT.jld2", "AA_dels_seq_WT")
#####################################################################################################################################
#####################################################################################################################################
    global NSP_muts
        NSP_muts = load("$(folder_name)/$(folder_name)__NSP_muts.jld2", "NSP_muts")
    global NSP_muts_no_dels
        NSP_muts_no_dels = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels.jld2", "NSP_muts_no_dels")
#####################################################################################################################################
    global non_rep_seqs_AA
        non_rep_seqs_AA = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA.jld2", "non_rep_seqs_AA")
    global non_rep_seqs_AA_pos_only
        non_rep_seqs_AA_pos_only = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only.jld2", "non_rep_seqs_AA_pos_only")
    global non_rep_seqs_AA_pos_only_no_dels
        non_rep_seqs_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only_no_dels.jld2", "non_rep_seqs_AA_pos_only_no_dels")
#####################################################################################################################################
    global rep_seq_grps_AA_muts_WT
        rep_seq_grps_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT.jld2", "rep_seq_grps_AA_muts_WT")
    global rep_seq_grps_AA_muts_WT_pos_only
        rep_seq_grps_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT_pos_only.jld2", "rep_seq_grps_AA_muts_WT_pos_only")
    global rep_seq_grps_AA_dels_WT
        rep_seq_grps_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels_WT.jld2", "rep_seq_grps_AA_dels_WT")
    global rep_seq_grps_nuc_muts_WT
        rep_seq_grps_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_muts_WT.jld2", "rep_seq_grps_nuc_muts_WT")
    global rep_seq_grps_nuc_dels_WT
        rep_seq_grps_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dels_WT.jld2", "rep_seq_grps_nuc_dels_WT")
    global nuc_muts_rep_seq_grps_WT
        nuc_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_WT.jld2", "nuc_muts_rep_seq_grps_WT")
    global nuc_dels_rep_seq_grps_WT
        nuc_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps_WT.jld2", "nuc_dels_rep_seq_grps_WT")
    global AA_dels_rep_seq_grps_WT
        AA_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps_WT.jld2", "AA_dels_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT
        AA_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT.jld2", "AA_muts_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT_pos_only
        AA_muts_rep_seq_grps_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT_pos_only.jld2", "AA_muts_rep_seq_grps_WT_pos_only")
#####################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_maxmut_pango
        rep_seq_grps_maxmut_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango.jld2", "rep_seq_grps_maxmut_pango")
    global rep_seq_grps_maxmut_clade
        rep_seq_grps_maxmut_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_clade.jld2", "rep_seq_grps_maxmut_clade")
    global rep_seq_grps_maxmut_pango_unaliased
        rep_seq_grps_maxmut_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango_unaliased.jld2", "rep_seq_grps_maxmut_pango_unaliased")
#####################################################################################################################################
    global rep_seq_grps_maxmut_dels
        rep_seq_grps_maxmut_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_dels.jld2", "rep_seq_grps_maxmut_dels")
    global rep_seq_grps_maxmut_AA_pos_only_no_dels
        rep_seq_grps_maxmut_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only_no_dels.jld2", "rep_seq_grps_maxmut_AA_pos_only_no_dels")
    global rep_seq_grps_maxmut_nuc
        rep_seq_grps_maxmut_nuc = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc.jld2", "rep_seq_grps_maxmut_nuc")
    global rep_seq_grps_maxmut_AA_pos_only
        rep_seq_grps_maxmut_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only.jld2", "rep_seq_grps_maxmut_AA_pos_only")
    global rep_seq_grps_maxmut_nuc_dropout
        rep_seq_grps_maxmut_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dropout.jld2", "rep_seq_grps_maxmut_nuc_dropout")
    global rep_seq_grps_maxmut_nuc_no_dels
        rep_seq_grps_maxmut_nuc_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_no_dels.jld2", "rep_seq_grps_maxmut_nuc_no_dels")
    global rep_seq_grps_maxmut_mixed_nucs
        rep_seq_grps_maxmut_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_nucs.jld2", "rep_seq_grps_maxmut_mixed_nucs")
    global rep_seq_grps_maxmut_AA_dels
        rep_seq_grps_maxmut_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels.jld2", "rep_seq_grps_maxmut_AA_dels")
    global rep_seq_grps_maxmut_del_ranges_ct
        rep_seq_grps_maxmut_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_del_ranges_ct.jld2", "rep_seq_grps_maxmut_del_ranges_ct")
    global rep_seq_grps_maxmut_AA
        rep_seq_grps_maxmut_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA.jld2", "rep_seq_grps_maxmut_AA")
    global rep_seq_grps_maxmut_mixed_AA_muts
        rep_seq_grps_maxmut_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_AA_muts.jld2", "rep_seq_grps_maxmut_mixed_AA_muts")
    global rep_seq_grps_maxmut_unknown_AA
        rep_seq_grps_maxmut_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA.jld2", "rep_seq_grps_maxmut_unknown_AA")
    global rep_seq_grps_maxmut_unknown_AA_ranges
        rep_seq_grps_maxmut_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA_ranges.jld2", "rep_seq_grps_maxmut_unknown_AA_ranges")
    global rep_seq_grps_maxmut_seqs
        rep_seq_grps_maxmut_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_seqs.jld2", "rep_seq_grps_maxmut_seqs")
    global rep_seq_grps_maxmut_AA_no_dels
        rep_seq_grps_maxmut_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_no_dels.jld2", "rep_seq_grps_maxmut_AA_no_dels")
    global rep_seq_grps_maxmut_nuc_muts_WT
        rep_seq_grps_maxmut_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_muts_WT.jld2", "rep_seq_grps_maxmut_nuc_muts_WT")
    global rep_seq_grps_maxmut_nuc_dels_WT
        rep_seq_grps_maxmut_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dels_WT.jld2", "rep_seq_grps_maxmut_nuc_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT
        rep_seq_grps_maxmut_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT.jld2", "rep_seq_grps_maxmut_AA_muts_WT")
    global rep_seq_grps_maxmut_AA_dels_WT
        rep_seq_grps_maxmut_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels_WT.jld2", "rep_seq_grps_maxmut_AA_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT_pos_only
        rep_seq_grps_maxmut_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT_pos_only.jld2", "rep_seq_grps_maxmut_AA_muts_WT_pos_only")
################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_seqs
        rep_seq_grps_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_seqs.jld2", "rep_seq_grps_seqs")
    global rep_seq_grps_pango
        rep_seq_grps_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango.jld2", "rep_seq_grps_pango")
    global rep_seq_grps_clade
        rep_seq_grps_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_clade.jld2", "rep_seq_grps_clade")
    global rep_seq_grps_pango_unaliased
        rep_seq_grps_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango_unaliased.jld2", "rep_seq_grps_pango_unaliased")
    global rep_seq_grps_muts
        rep_seq_grps_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts.jld2", "rep_seq_grps_muts")
    global rep_seq_grps_mixed_nucs
        rep_seq_grps_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_nucs.jld2", "rep_seq_grps_mixed_nucs")
    global rep_seq_grps_muts_no_dels
        rep_seq_grps_muts_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts_no_dels.jld2", "rep_seq_grps_muts_no_dels")
    global rep_seq_grps_AA_pos_only
        rep_seq_grps_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only.jld2", "rep_seq_grps_AA_pos_only")
    global rep_seq_grps_AA_no_dels
        rep_seq_grps_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_no_dels.jld2", "rep_seq_grps_AA_no_dels")
    global rep_seq_grps_AA_dels
        rep_seq_grps_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels.jld2", "rep_seq_grps_AA_dels")
    global rep_seq_grps_dels
        rep_seq_grps_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_dels.jld2", "rep_seq_grps_dels")
    global rep_seq_grps_AA_pos_only_no_dels
        rep_seq_grps_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only_no_dels.jld2", "rep_seq_grps_AA_pos_only_no_dels")
    global rep_seq_grps_del_ranges_ct
        rep_seq_grps_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_del_ranges_ct.jld2", "rep_seq_grps_del_ranges_ct")
    global rep_seq_grps_AA
        rep_seq_grps_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA.jld2", "rep_seq_grps_AA")
    global rep_seq_grps_nuc_dropout
        rep_seq_grps_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dropout.jld2", "rep_seq_grps_nuc_dropout")
#    global rep_seq_grps_unknown_AA
#        rep_seq_grps_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_unknown_AA.jld2", "rep_seq_grps_unknown_AA")
#    global rep_seq_grps_mixed_AA_muts
#        rep_seq_grps_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_AA_muts.jld2", "rep_seq_grps_mixed_AA_muts")
    global AA_dels_rep_seq_grps
        AA_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps.jld2", "AA_dels_rep_seq_grps")
    global AA_muts_rep_seq_grps
        AA_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps.jld2", "AA_muts_rep_seq_grps")
    global AA_muts_rep_seq_grps_pos_only
        AA_muts_rep_seq_grps_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_pos_only.jld2", "AA_muts_rep_seq_grps_pos_only")
    global AA_muts_rep_seq_grps_no_dels
        AA_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_no_dels.jld2", "AA_muts_rep_seq_grps_no_dels")
    global nuc_muts_rep_seq_grps
        nuc_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps.jld2", "nuc_muts_rep_seq_grps")
    global nuc_muts_rep_seq_grps_no_dels
        nuc_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_no_dels.jld2", "nuc_muts_rep_seq_grps_no_dels")
    global nuc_dels_rep_seq_grps
        nuc_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps.jld2", "nuc_dels_rep_seq_grps")    
##########################################################################################################################################################################
##########################################################################################################################################################################
################################################ Below: Used to be in ungodly @save form, now changed ####################################################################
##########################################################################################################################################################################
##########################################################################################################################################################################
    global AA_muts_ct_pos_only_adj_sort_by_site
        AA_muts_ct_pos_only_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_sort_by_site")
    global nuc_muts_ct_adj_score_no_dels_sort_by_site
        nuc_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_no_dels_sort_by_site
        nuc_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_no_dels_sort_by_site")
    global nuc_muts_ct_adj_sort_by_seq_ct
        nuc_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_sort_by_seq_ct")
    global nuc_muts_ct_adj_score_sort_by_score
        nuc_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_score.jld2", "nuc_muts_ct_adj_score_sort_by_score")
    global AA_muts_ct_adj_score_sort_by_site
        AA_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_site.jld2", "AA_muts_ct_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_sort_by_site
        AA_muts_ct_pos_only_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_score_no_dels_sort_by_site
        AA_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_site")
    global AA_muts_ct_adj_sort_by_seq_ct
        AA_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_seq_ct.jld2", "AA_muts_ct_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_no_dels_sort_by_site
        AA_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_adj_score_sort_by_score
        AA_muts_ct_pos_only_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_score")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_no_dels_sort_by_seq_ct
        nuc_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_adj_sort_by_site
        AA_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_site.jld2", "AA_muts_ct_adj_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_site")
    global AA_muts_ct_adj_score_sort_by_score
        AA_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_score.jld2", "AA_muts_ct_adj_score_sort_by_score")
    global nuc_muts_ct_adj_score_no_dels_sort_by_score
        nuc_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_score.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_sort_by_site
        nuc_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_site.jld2", "nuc_muts_ct_adj_sort_by_site")
    global AA_muts_ct_adj_score_no_dels_sort_by_score
        AA_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_score")
    global AA_muts_ct_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_score_sort_by_site
        nuc_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_site.jld2", "nuc_muts_ct_adj_score_sort_by_site")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_sort_by_site
        AA_muts_ct_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_site.jld2", "AA_muts_ct_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_seq_ct
        AA_muts_ct_pos_only_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global nuc_muts_ct_sort_by_site
        nuc_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_site.jld2", "nuc_muts_ct_sort_by_site")
    global excluded_AA
        excluded_AA = load("$(folder_name)/$(folder_name)__excluded_AA.jld2", "excluded_AA")
    global chronic_search_muts
        chronic_search_muts = load("$(folder_name)/$(folder_name)__chronic_search_muts.jld2", "chronic_search_muts")
    global excluded_pos
        excluded_pos = load("$(folder_name)/$(folder_name)__excluded_pos.jld2", "excluded_pos")
    global rep_seqs
        rep_seqs = load("$(folder_name)/$(folder_name)__rep_seqs.jld2", "rep_seqs")
    global non_rep_seqs
        non_rep_seqs = load("$(folder_name)/$(folder_name)__non_rep_seqs.jld2", "non_rep_seqs")
    global all_seqs
        all_seqs = load("$(folder_name)/$(folder_name)__all_seqs.jld2", "all_seqs")
    global domain_mut_density_sort_by_gene
        domain_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density
        gene_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global nuc_muts_ct_sort_by_seq_ct
        nuc_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_sort_by_seq_ct
        AA_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_by_seq_ct
        AA_muts_ct_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_by_site
        AA_muts_ct_pos_only_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_site")
    global gene_mut_density_sort_by_gene
        gene_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global AA_muts_ct_sort_by_site
        AA_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_site.jld2", "AA_muts_ct_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_site
        AA_muts_ct_pos_only_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_site.jld2", "AA_muts_ct_pos_only_sort_by_site")
    global domain_mut_density_sort_by_density
        domain_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global too_many_reversions
        too_many_reversions = load("$(folder_name)/$(folder_name)__too_many_reversions.jld2", "too_many_reversions")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")

    global AA_muts_ct_chr_all_ratio_ct_sort
        AA_muts_ct_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_chr_all_ratio_ct_sort")
    global AA_muts_ct_chr_all_ratio_pos_sort
        AA_muts_ct_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_chr_all_ratio_pos_sort")    

    global AA_muts_ct_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global AA_muts_ct_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_pos_sort")
    
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    
    global avg_AA_subs_per_chr_seq
        avg_AA_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq.jld2", "avg_AA_subs_per_chr_seq")
    global avg_AA_subs_per_circ_seq
        avg_AA_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_circ_seq.jld2", "avg_AA_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global NSP_muts_sortByCt_Arr
        NSP_muts_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByCt_Arr.jld2", "NSP_muts_sortByCt_Arr")
    global NSP_muts_sortByPos_Arr
        NSP_muts_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByPos_Arr.jld2", "NSP_muts_sortByPos_Arr")
    global NSP_muts_no_dels_sortByCt_Arr
        NSP_muts_no_dels_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByCt_Arr.jld2", "NSP_muts_no_dels_sortByCt_Arr")
    global NSP_muts_no_dels_sortByPos_Arr
        NSP_muts_no_dels_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByPos_Arr.jld2", "NSP_muts_no_dels_sortByPos_Arr")
    global NSP1_muts_sortByCt
        NSP1_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByCt.jld2", "NSP1_muts_sortByCt")
    global NSP1_muts_sortByPos
        NSP1_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByPos.jld2", "NSP1_muts_sortByPos")
    global NSP2_muts_sortByCt
        NSP2_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByCt.jld2", "NSP2_muts_sortByCt")
    global NSP2_muts_sortByPos
        NSP2_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByPos.jld2", "NSP2_muts_sortByPos")
    global NSP3_muts_sortByCt
        NSP3_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByCt.jld2", "NSP3_muts_sortByCt")
    global NSP3_muts_sortByPos
        NSP3_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByPos.jld2", "NSP3_muts_sortByPos")
    global NSP4_muts_sortByCt
        NSP4_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByCt.jld2", "NSP4_muts_sortByCt")
    global NSP4_muts_sortByPos
        NSP4_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByPos.jld2", "NSP4_muts_sortByPos")
    global NSP5_muts_sortByCt
        NSP5_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByCt.jld2", "NSP5_muts_sortByCt")
    global NSP5_muts_sortByPos
        NSP5_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByPos.jld2", "NSP5_muts_sortByPos")
    global NSP6_muts_sortByCt
        NSP6_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByCt.jld2", "NSP6_muts_sortByCt")
    global NSP6_muts_sortByPos
        NSP6_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByPos.jld2", "NSP6_muts_sortByPos")
    global NSP7_muts_sortByCt
        NSP7_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByCt.jld2", "NSP7_muts_sortByCt")
    global NSP7_muts_sortByPos
        NSP7_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByPos.jld2", "NSP7_muts_sortByPos")
    global NSP8_muts_sortByCt
        NSP8_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByCt.jld2", "NSP8_muts_sortByCt")
    global NSP8_muts_sortByPos
        NSP8_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByPos.jld2", "NSP8_muts_sortByPos")
    global NSP9_muts_sortByCt
        NSP9_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByCt.jld2", "NSP9_muts_sortByCt")
    global NSP9_muts_sortByPos
        NSP9_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByPos.jld2", "NSP9_muts_sortByPos")
    global NSP10_muts_sortByCt
        NSP10_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByCt.jld2", "NSP10_muts_sortByCt")
    global NSP10_muts_sortByPos
        NSP10_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByPos.jld2", "NSP10_muts_sortByPos")
    global NSP11_muts_sortByCt
        NSP11_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByCt.jld2", "NSP11_muts_sortByCt")
    global NSP11_muts_sortByPos
        NSP11_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByPos.jld2", "NSP11_muts_sortByPos")
    global NSP12_muts_sortByCt
        NSP12_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByCt.jld2", "NSP12_muts_sortByCt")
    global NSP12_muts_sortByPos
        NSP12_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByPos.jld2", "NSP12_muts_sortByPos")
    global NSP13_muts_sortByCt
        NSP13_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByCt.jld2", "NSP13_muts_sortByCt")
    global NSP13_muts_sortByPos
        NSP13_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByPos.jld2", "NSP13_muts_sortByPos")
    global NSP14_muts_sortByCt
        NSP14_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByCt.jld2", "NSP14_muts_sortByCt")
    global NSP14_muts_sortByPos
        NSP14_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByPos.jld2", "NSP14_muts_sortByPos")
    global NSP15_muts_sortByCt
        NSP15_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByCt.jld2", "NSP15_muts_sortByCt")
    global NSP15_muts_sortByPos
        NSP15_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByPos.jld2", "NSP15_muts_sortByPos")
    global NSP16_muts_sortByCt
        NSP16_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByCt.jld2", "NSP16_muts_sortByCt")
    global NSP16_muts_sortByPos
        NSP16_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByPos.jld2", "NSP16_muts_sortByPos")
    global NSP1_muts_no_dels_sortByCt
        NSP1_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByCt.jld2", "NSP1_muts_no_dels_sortByCt")
    global NSP1_muts_no_dels_sortByPos
        NSP1_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByPos.jld2", "NSP1_muts_no_dels_sortByPos")
    global NSP2_muts_no_dels_sortByCt
        NSP2_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByCt.jld2", "NSP2_muts_no_dels_sortByCt")
    global NSP2_muts_no_dels_sortByPos
        NSP2_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByPos.jld2", "NSP2_muts_no_dels_sortByPos")
    global NSP3_muts_no_dels_sortByCt
        NSP3_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByCt.jld2", "NSP3_muts_no_dels_sortByCt")
    global NSP3_muts_no_dels_sortByPos
        NSP3_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByPos.jld2", "NSP3_muts_no_dels_sortByPos")
    global NSP4_muts_no_dels_sortByCt
        NSP4_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByCt.jld2", "NSP4_muts_no_dels_sortByCt")
    global NSP4_muts_no_dels_sortByPos
        NSP4_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByPos.jld2", "NSP4_muts_no_dels_sortByPos")
    global NSP5_muts_no_dels_sortByCt
        NSP5_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByCt.jld2", "NSP5_muts_no_dels_sortByCt")
    global NSP5_muts_no_dels_sortByPos
        NSP5_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByPos.jld2", "NSP5_muts_no_dels_sortByPos")
    global NSP6_muts_no_dels_sortByCt
        NSP6_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByCt.jld2", "NSP6_muts_no_dels_sortByCt")
    global NSP6_muts_no_dels_sortByPos
        NSP6_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByPos.jld2", "NSP6_muts_no_dels_sortByPos")
    global NSP7_muts_no_dels_sortByCt
        NSP7_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByCt.jld2", "NSP7_muts_no_dels_sortByCt")
    global NSP7_muts_no_dels_sortByPos
        NSP7_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByPos.jld2", "NSP7_muts_no_dels_sortByPos")
    global NSP8_muts_no_dels_sortByCt
        NSP8_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByCt.jld2", "NSP8_muts_no_dels_sortByCt")
    global NSP8_muts_no_dels_sortByPos
        NSP8_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByPos.jld2", "NSP8_muts_no_dels_sortByPos")
    global NSP9_muts_no_dels_sortByCt
        NSP9_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByCt.jld2", "NSP9_muts_no_dels_sortByCt")
    global NSP9_muts_no_dels_sortByPos
        NSP9_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByPos.jld2", "NSP9_muts_no_dels_sortByPos")
    global NSP10_muts_no_dels_sortByCt
        NSP10_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByCt.jld2", "NSP10_muts_no_dels_sortByCt")
    global NSP10_muts_no_dels_sortByPos
        NSP10_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByPos.jld2", "NSP10_muts_no_dels_sortByPos")
    global NSP11_muts_no_dels_sortByCt
        NSP11_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByCt.jld2", "NSP11_muts_no_dels_sortByCt")
    global NSP11_muts_no_dels_sortByPos
        NSP11_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByPos.jld2", "NSP11_muts_no_dels_sortByPos")
    global NSP12_muts_no_dels_sortByCt
        NSP12_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByCt.jld2", "NSP12_muts_no_dels_sortByCt")
    global NSP12_muts_no_dels_sortByPos
        NSP12_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByPos.jld2", "NSP12_muts_no_dels_sortByPos")
    global NSP13_muts_no_dels_sortByCt
        NSP13_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByCt.jld2", "NSP13_muts_no_dels_sortByCt")
    global NSP13_muts_no_dels_sortByPos
        NSP13_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByPos.jld2", "NSP13_muts_no_dels_sortByPos")
    global NSP14_muts_no_dels_sortByCt
        NSP14_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByCt.jld2", "NSP14_muts_no_dels_sortByCt")
    global NSP14_muts_no_dels_sortByPos
        NSP14_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByPos.jld2", "NSP14_muts_no_dels_sortByPos")
    global NSP15_muts_no_dels_sortByCt
        NSP15_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByCt.jld2", "NSP15_muts_no_dels_sortByCt")
    global NSP15_muts_no_dels_sortByPos
        NSP15_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByPos.jld2", "NSP15_muts_no_dels_sortByPos")
    global NSP16_muts_no_dels_sortByCt
        NSP16_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByCt.jld2", "NSP16_muts_no_dels_sortByCt")
    global NSP16_muts_no_dels_sortByPos
        NSP16_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByPos.jld2", "NSP16_muts_no_dels_sortByPos")
####################################################################################################
    global all_seqs_set
        all_seqs_set = load("$(folder_name)/$(folder_name)__all_seqs_set.jld2", "all_seqs_set")
    global all_qualifying_seqs
        all_qualifying_seqs = load("$(folder_name)/$(folder_name)__all_qualifying_seqs.jld2", "all_qualifying_seqs")
    global all_qualifying_seqs_set
        all_qualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_qualifying_seqs_set.jld2", "all_qualifying_seqs_set")
    global all_nonqualifying_seqs
        all_nonqualifying_seqs = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs.jld2", "all_nonqualifying_seqs")
    global all_nonqualifying_seqs_set
        all_nonqualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs_set.jld2", "all_nonqualifying_seqs_set")
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_no_revs_sort_by_site
        AA_muts_ct_no_dels_no_revs_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_site.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_site")
    global AA_muts_ct_no_dels_no_revs_sort_by_seq_ct
        AA_muts_ct_no_dels_no_revs_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_seq_ct")
    global chronic_search_muts_v2
        chronic_search_muts_v2 = load("$(folder_name)/$(folder_name)__chronic_search_muts_v2.jld2", "chronic_search_muts_v2")
    global avg_AA_subs_per_chr_seq_no_revs
        avg_AA_subs_per_chr_seq_no_revs = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq_no_revs.jld2", "avg_AA_subs_per_chr_seq_no_revs")
###########################################################################################################################################################################
    global nuc_muts_ct_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global nuc_muts_ct_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    global avg_nuc_subs_per_chr_seq
        avg_nuc_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_chr_seq.jld2", "avg_nuc_subs_per_chr_seq")
    global avg_nuc_subs_per_circ_seq
        avg_nuc_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_circ_seq.jld2", "avg_nuc_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    return date_nuc_mut_ct, date_nuc_mut_ct_no_dels, date_AA_mut_ct, date_AA_mut_ct_no_dels, date_AA_mut_ct_pos_only_no_dels, 
    seq_ct_by_year, seq_ct_by_year_month, seq_ct_by_year_month_day, 
    seq_collection_date, seq_date_index, seq_date_tuple, 
###################################################################################################################################
    seq_clade, seq_clade_display, seq_pango, 
    seq_pango_unaliased, seq_clade_ct, seq_pango_ct, seq_pango_unaliased_ct, 
###################################################################################################################################
    too_many_reversions, 
###################################################################################################################################
    seq_nuc_muts, seq_nuc_del_ranges, seq_nuc_dropout, seq_nuc_muts_WT, seq_nuc_del_ranges_WT, 
################################
    seq_AA_insertions_WT, seq_nuc_insertions_WT,
################################
    seq_AA_muts, seq_AA_muts_no_dels, seq_AA_muts_pos_only, seq_AA_dels, seq_mixed_AA_muts, 
    seq_AA_muts_WT, seq_AA_muts_WT_pos_only, seq_AA_dels_WT, seq_AA_muts_pos_only_no_dels, 
########################################################
    nuc_muts_seq, nuc_muts_seq_WT, 
    nuc_dels_seq, nuc_dels_seq_WT, 
    AA_muts_seq, AA_muts_seq_pos_only, AA_muts_seq_WT, AA_muts_seq_WT_pos_only, AA_muts_seq_pos_only_no_dels, 
    AA_dels_seq, AA_dels_seq_WT, 
######################################################## 
    seq_unknown_AA, seq_unknown_AA_ranges, seq_mixed_nucs, 
############################
    nuc_dels_ct, nuc_muts_ct, nuc_muts_ct_no_dels, 
###############
    AA_dels_ct, AA_muts_ct, AA_muts_ct_no_dels, AA_muts_ct_pos_only, AA_muts_ct_pos_only_no_dels, AA_muts_ct_no_dels_no_revs, 
############################
    nuc_muts_ct_sort_by_site, nuc_muts_ct_sort_by_seq_ct, 
############################ 
    AA_muts_ct_sort_by_site, AA_muts_ct_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_site, AA_muts_ct_no_dels_no_revs_sort_by_site, 
    AA_muts_ct_no_dels_no_revs_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_seq_ct,  AA_muts_ct_pos_only_sort_by_site, 
    AA_muts_ct_pos_only_sort_by_seq_ct, AA_muts_ct_pos_only_no_dels_sort_by_site, AA_muts_ct_pos_only_no_dels_sort_by_seq_ct, 
###################################################################################################################################
################################################################################################################################### 
    nuc_muts_ct_adj, nuc_muts_ct_adj_no_dels, nuc_muts_ct_adj_score, nuc_muts_ct_adj_score_no_dels,   
    nuc_muts_ct_adj_sort_by_seq_ct, nuc_muts_ct_adj_no_dels_sort_by_site, nuc_muts_ct_adj_no_dels_sort_by_seq_ct,
    nuc_muts_ct_adj_sort_by_site, nuc_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score, 
    nuc_muts_ct_adj_score_sort_by_site, nuc_muts_ct_adj_score_sort_by_score, nuc_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_adj, AA_muts_ct_adj_no_dels, AA_muts_ct_pos_only_adj, AA_muts_ct_pos_only_adj_no_dels, 
    AA_muts_ct_adj_score, AA_muts_ct_adj_score_no_dels, AA_muts_ct_pos_only_adj_score, AA_muts_ct_pos_only_adj_score_no_dels,
    AA_muts_ct_adj_sort_by_site, AA_muts_ct_adj_sort_by_seq_ct, AA_muts_ct_adj_no_dels_sort_by_site,
    AA_muts_ct_adj_no_dels_sort_by_seq_ct, AA_muts_ct_adj_score_sort_by_score, AA_muts_ct_adj_score_sort_by_site, 
    AA_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_sort_by_site, AA_muts_ct_pos_only_adj_sort_by_seq_ct, AA_muts_ct_pos_only_adj_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct, AA_muts_ct_pos_only_adj_score_sort_by_site, 
    AA_muts_ct_pos_only_adj_score_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site, 
####################################################################################################################################
    AA_muts_ct_chr_all_ratio, AA_muts_ct_no_dels_chr_all_ratio, AA_muts_ct_no_dels_no_revs_chr_all_ratio, AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
########################
    AA_muts_ct_chr_all_ratio_ct_sort, AA_muts_ct_chr_all_ratio_pos_sort, 
    AA_muts_ct_no_dels_chr_all_ratio_ct_sort, AA_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
###################################################### 
    nuc_muts_ct_no_dels_chr_all_ratio, nuc_muts_ct_no_dels_no_revs_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
#######################
    nuc_muts_ct_no_dels_chr_all_ratio_ct_sort, nuc_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
####################################################################################################################################
    NSP_muts, NSP_muts_sortByCt_Arr, NSP_muts_sortByPos_Arr, NSP_muts_no_dels_sortByCt_Arr, NSP_muts_no_dels_sortByPos_Arr, 
    NSP1_muts_sortByCt, NSP2_muts_sortByCt, NSP3_muts_sortByCt, NSP4_muts_sortByCt, NSP5_muts_sortByCt, NSP6_muts_sortByCt,
    NSP7_muts_sortByCt, NSP8_muts_sortByCt, NSP9_muts_sortByCt, NSP10_muts_sortByCt, NSP11_muts_sortByCt, NSP12_muts_sortByCt, 
    NSP13_muts_sortByCt, NSP14_muts_sortByCt, NSP15_muts_sortByCt, NSP16_muts_sortByCt, NSP1_muts_sortByPos, NSP2_muts_sortByPos, 
    NSP3_muts_sortByPos, NSP4_muts_sortByPos, NSP5_muts_sortByPos, NSP6_muts_sortByPos, NSP7_muts_sortByPos, NSP8_muts_sortByPos, 
    NSP9_muts_sortByPos, NSP10_muts_sortByPos, NSP11_muts_sortByPos, NSP12_muts_sortByPos, NSP13_muts_sortByPos, NSP14_muts_sortByPos, 
    NSP15_muts_sortByPos, NSP16_muts_sortByPos, NSP1_muts_no_dels_sortByCt, NSP2_muts_no_dels_sortByCt, NSP3_muts_no_dels_sortByCt, 
    NSP4_muts_no_dels_sortByCt, NSP5_muts_no_dels_sortByCt, NSP6_muts_no_dels_sortByCt, NSP7_muts_no_dels_sortByCt, 
    NSP8_muts_no_dels_sortByCt, NSP9_muts_no_dels_sortByCt, NSP10_muts_no_dels_sortByCt, NSP11_muts_no_dels_sortByCt, 
    NSP12_muts_no_dels_sortByCt, NSP13_muts_no_dels_sortByCt, NSP14_muts_no_dels_sortByCt, NSP15_muts_no_dels_sortByCt, 
    NSP16_muts_no_dels_sortByCt, NSP1_muts_no_dels_sortByPos, NSP2_muts_no_dels_sortByPos, NSP3_muts_no_dels_sortByPos, 
    NSP4_muts_no_dels_sortByPos, NSP5_muts_no_dels_sortByPos, NSP6_muts_no_dels_sortByPos, NSP7_muts_no_dels_sortByPos, 
    NSP8_muts_no_dels_sortByPos, NSP9_muts_no_dels_sortByPos, NSP10_muts_no_dels_sortByPos, NSP11_muts_no_dels_sortByPos, 
    NSP12_muts_no_dels_sortByPos, NSP13_muts_no_dels_sortByPos, NSP14_muts_no_dels_sortByPos, NSP15_muts_no_dels_sortByPos, 
    NSP16_muts_no_dels_sortByPos,
####################################################################################################################################
    chronic_search_muts, chronic_search_muts_v2, 
####################################################################################################################################
    avg_AA_subs_per_circ_seq, avg_nuc_subs_per_circ_seq, avg_AA_subs_per_chr_seq, avg_nuc_subs_per_chr_seq,
####################################################################################################################################
    nuc_muts_rep_seq_grps, nuc_dels_rep_seq_grps, 
    nuc_muts_rep_seq_grps_no_dels, rep_seq_grps_dels, rep_seq_grps_muts_no_dels, rep_seq_grps_del_ranges_ct, rep_seq_grps_nuc_dropout, 
    rep_seq_grps_mixed_nucs, rep_seq_grps_AA_dels, rep_seq_grps_AA_no_dels, AA_muts_rep_seq_grps, AA_dels_rep_seq_grps, 
    AA_muts_rep_seq_grps_no_dels, AA_muts_rep_seq_grps_pos_only,  
    rep_seq_grps_nuc_muts_WT, rep_seq_grps_nuc_dels_WT, rep_seq_grps_AA_muts_WT, rep_seq_grps_AA_dels_WT, nuc_muts_rep_seq_grps_WT, 
    nuc_dels_rep_seq_grps_WT, AA_muts_rep_seq_grps_WT, AA_dels_rep_seq_grps_WT, rep_seq_grps_AA_muts_WT_pos_only, 
    AA_muts_rep_seq_grps_WT_pos_only, rep_seq_grps_pango, rep_seq_grps_pango_unaliased, excluded_pos, excluded_AA, all_seqs, rep_seqs, 
    non_rep_seqs, rep_seq_grps_seqs, rep_seq_grps_muts, rep_seq_grps_clade, rep_seq_grps_AA, rep_seq_grps_AA_pos_only, 
    rep_seq_grps_AA_pos_only_no_dels, non_rep_seqs_AA, non_rep_seqs_AA_pos_only, non_rep_seqs_AA_pos_only_no_dels, 
####################################################################################################################################
    rep_seq_grps_maxmut_nuc_muts_WT, rep_seq_grps_maxmut_nuc_dels_WT, 
    rep_seq_grps_maxmut_AA_muts_WT, rep_seq_grps_maxmut_AA_dels_WT, rep_seq_grps_maxmut_AA_muts_WT_pos_only,
    rep_seq_grps_maxmut_seqs, rep_seq_grps_maxmut_nuc, rep_seq_grps_maxmut_nuc_no_dels, rep_seq_grps_maxmut_dels, 
    rep_seq_grps_maxmut_nuc_dropout, rep_seq_grps_maxmut_mixed_nucs, rep_seq_grps_maxmut_AA, rep_seq_grps_maxmut_AA_no_dels, 
    rep_seq_grps_maxmut_AA_dels, rep_seq_grps_maxmut_AA_pos_only, rep_seq_grps_maxmut_AA_pos_only_no_dels, 
    rep_seq_grps_maxmut_unknown_AA, rep_seq_grps_maxmut_unknown_AA_ranges, rep_seq_grps_maxmut_mixed_AA_muts, rep_seq_grps_maxmut_del_ranges_ct,
####################################################################################################################################
    gene_mut_density_sort_by_gene, gene_mut_density_sort_by_density, domain_mut_density_sort_by_gene, domain_mut_density_sort_by_density, 
    gene_mut_density, domain_mut_density, 
####################################################################################################################################
    all_seqs_set, all_qualifying_seqs, all_qualifying_seqs_set, all_nonqualifying_seqs, all_nonqualifying_seqs_set
####################################################################################################################################
    total_chr_AA_subs, total_nuc_revs_seq, seq_nuc_total_revs, total_AA_revs_seq, seq_AA_total_revs, seq_AA_revs
end
# REMOVED: rep_seq_grps_unknown_AA, rep_seq_grps_mixed_AA_muts,
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_chr(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    ORF1a_pos_ct = Dict{String, Int}()
    ORF1b_pos_ct = Dict{String, Int}()
    S_pos_ct = Dict{String, Int}()
    ORF3a_pos_ct = Dict{String, Int}()
    E_pos_ct = Dict{String, Int}()
    M_pos_ct = Dict{String, Int}()
    ORF6_pos_ct = Dict{String, Int}()
    ORF7a_pos_ct = Dict{String, Int}()
    ORF7b_pos_ct = Dict{String, Int}()
    ORF8_pos_ct = Dict{String, Int}()
    N_pos_ct = Dict{String, Int}()
    ORF9b_pos_ct = Dict{String, Int}()
    gene_pos_ct_dict_arr = [ORF1a_pos_ct, ORF1b_pos_ct, S_pos_ct, ORF3a_pos_ct, E_pos_ct, M_pos_ct, ORF6_pos_ct, ORF7a_pos_ct, ORF7b_pos_ct, ORF8_pos_ct, N_pos_ct, ORF9b_pos_ct]
######################################################
    ORF1a_pos_ct_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_v1 = Dict{Int, Int}()
    S_pos_ct_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_v1 = Dict{Int, Int}()
    E_pos_ct_v1 = Dict{Int, Int}()
    M_pos_ct_v1 = Dict{Int, Int}()
    ORF6_pos_ct_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_v1 = Dict{Int, Int}()
    ORF8_pos_ct_v1 = Dict{Int, Int}()
    N_pos_ct_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_v1 = Dict{Int, Int}()
    gene_pos_ct_v1_dict_arr = [ORF1a_pos_ct_v1, ORF1b_pos_ct_v1, S_pos_ct_v1, ORF3a_pos_ct_v1, E_pos_ct_v1, M_pos_ct_v1, ORF6_pos_ct_v1, ORF7a_pos_ct_v1, ORF7b_pos_ct_v1, ORF8_pos_ct_v1, N_pos_ct_v1, ORF9b_pos_ct_v1]
######################################################
    for i in 1:length(gene_pos_ct_dict_arr)
        dict = gene_pos_ct_dict_arr[i]
        dict_v1 = gene_pos_ct_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct[mut] = ct
            ORF1a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct[mut] = ct
            ORF1b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct[mut] = ct
            S_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct[mut] = ct
            ORF3a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct[mut] = ct
            E_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct[mut] = ct
            M_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct[mut] = ct
            ORF6_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct[mut] = ct
            ORF7a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct[mut] = ct
            ORF7b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct[mut] = ct
            ORF8_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct[mut] = ct
            N_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct[mut] = ct
            ORF9b_pos_ct_v1[pos] = ct
        end
    end
######################### 
    global ORF1a_pos_ct_sort_by_pos = sort(collect(ORF1a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_sort_by_pos = sort(collect(ORF1b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_sort_by_pos = sort(collect(S_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_sort_by_pos = sort(collect(ORF3a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_sort_by_pos = sort(collect(E_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global M_pos_ct_sort_by_pos = sort(collect(M_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_sort_by_pos = sort(collect(ORF6_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_sort_by_pos = sort(collect(ORF7a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_sort_by_pos = sort(collect(ORF7b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_sort_by_pos = sort(collect(ORF8_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_sort_by_pos = sort(collect(N_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF9b_pos_ct_sort_by_pos = sort(collect(ORF9b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
#########################
    global ORF1a_pos_ct_sort_by_ct = sort(collect(ORF1a_pos_ct), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct = sort(collect(ORF1b_pos_ct), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct = sort(collect(S_pos_ct), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct = sort(collect(ORF3a_pos_ct), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct = sort(collect(E_pos_ct), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct = sort(collect(M_pos_ct), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct = sort(collect(ORF6_pos_ct), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct = sort(collect(ORF7a_pos_ct), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct = sort(collect(ORF7b_pos_ct), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct = sort(collect(ORF8_pos_ct), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct = sort(collect(N_pos_ct), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct = sort(collect(ORF9b_pos_ct), by = x -> x[2], rev=true)
#########################
    global ORF1a_pos_ct_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[1])
    global ORF1b_pos_ct_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[1])
    global S_pos_ct_sort_by_pos_v1 = sort(collect(S_pos_ct_v1), by = x -> x[1])
    global ORF3a_pos_ct_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[1])
    global E_pos_ct_sort_by_pos_v1 = sort(collect(E_pos_ct_v1), by = x -> x[1])
    global M_pos_ct_sort_by_pos_v1 = sort(collect(M_pos_ct_v1), by = x -> x[1])
    global ORF6_pos_ct_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[1])
    global ORF7a_pos_ct_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[1])
    global ORF7b_pos_ct_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[1])
    global ORF8_pos_ct_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[1])
    global N_pos_ct_sort_by_pos_v1 = sort(collect(N_pos_ct_v1), by = x -> x[1])
    global ORF9b_pos_ct_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[1])
#########################
    global ORF1a_pos_ct_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct_v1 = sort(collect(S_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct_v1 = sort(collect(E_pos_ct_v1), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct_v1 = sort(collect(M_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct_v1 = sort(collect(N_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[2], rev=true)
end
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
function gene_sub_ranks_chr(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    ORF1a_ct = Dict{String, Int}()
    ORF1b_ct = Dict{String, Int}()
    S_ct = Dict{String, Int}()
    ORF3a_ct = Dict{String, Int}()
    E_ct = Dict{String, Int}()
    M_ct = Dict{String, Int}()
    ORF6_ct = Dict{String, Int}()
    ORF7a_ct = Dict{String, Int}()
    ORF7b_ct = Dict{String, Int}()
    ORF8_ct = Dict{String, Int}()
    N_ct = Dict{String, Int}()
    ORF9b_ct = Dict{String, Int}()
#############################################################
    gene_ct_dict_arr = [ORF1a_ct, ORF1b_ct, S_ct, ORF3a_ct, E_ct, M_ct, ORF6_ct, ORF7a_ct, ORF7b_ct, ORF8_ct, N_ct, ORF9b_ct] 
#############################################################
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
########################################## 
    global ORF1a_ct_sort_by_pos = sort(collect(ORF1a_ct), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_sort_by_pos = sort(collect(ORF1b_ct), by = x -> fin_sortkey(x[1]))
    global S_ct_sort_by_pos = sort(collect(S_ct), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_sort_by_pos = sort(collect(ORF3a_ct), by = x -> fin_sortkey(x[1]))
    global E_ct_sort_by_pos = sort(collect(E_ct), by = x -> fin_sortkey(x[1]))
    global M_ct_sort_by_pos = sort(collect(M_ct), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_sort_by_pos = sort(collect(ORF6_ct), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_sort_by_pos = sort(collect(ORF7a_ct), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_sort_by_pos = sort(collect(ORF7b_ct), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_sort_by_pos = sort(collect(ORF8_ct), by = x -> fin_sortkey(x[1]))
    global N_ct_sort_by_pos = sort(collect(N_ct), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_sort_by_pos = sort(collect(ORF9b_ct), by = x -> fin_sortkey(x[1]))
##########################################
    global ORF1a_ct_sort_by_ct = sort(collect(ORF1a_ct), by = x -> x[2], rev=true)
    global ORF1b_ct_sort_by_ct = sort(collect(ORF1b_ct), by = x -> x[2], rev=true)
    global S_ct_sort_by_ct = sort(collect(S_ct), by = x -> x[2], rev=true)
    global ORF3a_ct_sort_by_ct = sort(collect(ORF3a_ct), by = x -> x[2], rev=true)
    global E_ct_sort_by_ct = sort(collect(E_ct), by = x -> x[2], rev=true)
    global M_ct_sort_by_ct = sort(collect(M_ct), by = x -> x[2], rev=true)
    global ORF6_ct_sort_by_ct = sort(collect(ORF6_ct), by = x -> x[2], rev=true)
    global ORF7a_ct_sort_by_ct = sort(collect(ORF7a_ct), by = x -> x[2], rev=true)
    global ORF7b_ct_sort_by_ct = sort(collect(ORF7b_ct), by = x -> x[2], rev=true)
    global ORF8_ct_sort_by_ct = sort(collect(ORF8_ct), by = x -> x[2], rev=true)
    global N_ct_sort_by_ct = sort(collect(N_ct), by = x -> x[2], rev=true)
    global ORF9b_ct_sort_by_ct = sort(collect(ORF9b_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = aa_pos_comprehensive_dict[sub]
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
#############################
    global NSP1_ct_sort_by_pos = sort(collect(NSP1_ct), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_sort_by_pos = sort(collect(NSP2_ct), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_sort_by_pos = sort(collect(NSP3_ct), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_sort_by_pos = sort(collect(NSP4_ct), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_sort_by_pos = sort(collect(NSP5_ct), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_sort_by_pos = sort(collect(NSP6_ct), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_sort_by_pos = sort(collect(NSP7_ct), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_sort_by_pos = sort(collect(NSP8_ct), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_sort_by_pos = sort(collect(NSP9_ct), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_sort_by_pos = sort(collect(NSP10_ct), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_sort_by_pos = sort(collect(NSP11_ct), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_sort_by_pos = sort(collect(NSP12_ct), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_sort_by_pos = sort(collect(NSP13_ct), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_sort_by_pos = sort(collect(NSP14_ct), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_sort_by_pos = sort(collect(NSP15_ct), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_sort_by_pos = sort(collect(NSP16_ct), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_sort_by_ct = sort(collect(NSP1_ct), by = x -> x[2], rev=true)
    global NSP2_ct_sort_by_ct = sort(collect(NSP2_ct), by = x -> x[2], rev=true)
    global NSP3_ct_sort_by_ct = sort(collect(NSP3_ct), by = x -> x[2], rev=true)
    global NSP4_ct_sort_by_ct = sort(collect(NSP4_ct), by = x -> x[2], rev=true)
    global NSP5_ct_sort_by_ct = sort(collect(NSP5_ct), by = x -> x[2], rev=true)
    global NSP6_ct_sort_by_ct = sort(collect(NSP6_ct), by = x -> x[2], rev=true)
    global NSP7_ct_sort_by_ct = sort(collect(NSP7_ct), by = x -> x[2], rev=true)
    global NSP8_ct_sort_by_ct = sort(collect(NSP8_ct), by = x -> x[2], rev=true)
    global NSP9_ct_sort_by_ct = sort(collect(NSP9_ct), by = x -> x[2], rev=true)
    global NSP10_ct_sort_by_ct = sort(collect(NSP10_ct), by = x -> x[2], rev=true)
    global NSP11_ct_sort_by_ct = sort(collect(NSP11_ct), by = x -> x[2], rev=true)
    global NSP12_ct_sort_by_ct = sort(collect(NSP12_ct), by = x -> x[2], rev=true)
    global NSP13_ct_sort_by_ct = sort(collect(NSP13_ct), by = x -> x[2], rev=true)
    global NSP14_ct_sort_by_ct = sort(collect(NSP14_ct), by = x -> x[2], rev=true)
    global NSP15_ct_sort_by_ct = sort(collect(NSP15_ct), by = x -> x[2], rev=true)
    global NSP16_ct_sort_by_ct = sort(collect(NSP16_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_pos_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    NSP1_pos_ct = Dict{String, Int}()
    NSP2_pos_ct = Dict{String, Int}()
    NSP3_pos_ct = Dict{String, Int}()
    NSP4_pos_ct = Dict{String, Int}()
    NSP5_pos_ct = Dict{String, Int}()
    NSP6_pos_ct = Dict{String, Int}()
    NSP7_pos_ct = Dict{String, Int}()
    NSP8_pos_ct = Dict{String, Int}()
    NSP9_pos_ct = Dict{String, Int}()
    NSP10_pos_ct = Dict{String, Int}()
    NSP11_pos_ct = Dict{String, Int}()
    NSP12_pos_ct = Dict{String, Int}()
    NSP13_pos_ct = Dict{String, Int}()
    NSP14_pos_ct = Dict{String, Int}()
    NSP15_pos_ct = Dict{String, Int}()
    NSP16_pos_ct = Dict{String, Int}()
    NSP_pos_ct_array = [NSP1_pos_ct, NSP2_pos_ct, NSP3_pos_ct, NSP4_pos_ct, NSP5_pos_ct, NSP6_pos_ct, NSP7_pos_ct, NSP8_pos_ct, NSP9_pos_ct, NSP10_pos_ct, NSP11_pos_ct, NSP12_pos_ct, NSP13_pos_ct, NSP14_pos_ct, NSP15_pos_ct, NSP16_pos_ct]
##########################################
    NSP1_pos_ct_v1 = Dict{Int, Int}()
    NSP2_pos_ct_v1 = Dict{Int, Int}()
    NSP3_pos_ct_v1 = Dict{Int, Int}()
    NSP4_pos_ct_v1 = Dict{Int, Int}()
    NSP5_pos_ct_v1 = Dict{Int, Int}()
    NSP6_pos_ct_v1 = Dict{Int, Int}()
    NSP7_pos_ct_v1 = Dict{Int, Int}()
    NSP8_pos_ct_v1 = Dict{Int, Int}()
    NSP9_pos_ct_v1 = Dict{Int, Int}()
    NSP10_pos_ct_v1 = Dict{Int, Int}()
    NSP11_pos_ct_v1 = Dict{Int, Int}()
    NSP12_pos_ct_v1 = Dict{Int, Int}()
    NSP13_pos_ct_v1 = Dict{Int, Int}()
    NSP14_pos_ct_v1 = Dict{Int, Int}()
    NSP15_pos_ct_v1 = Dict{Int, Int}()
    NSP16_pos_ct_v1 = Dict{Int, Int}()
    NSP_pos_ct_v1_array = [NSP1_pos_ct_v1, NSP2_pos_ct_v1, NSP3_pos_ct_v1, NSP4_pos_ct_v1, NSP5_pos_ct_v1, NSP6_pos_ct_v1, NSP7_pos_ct_v1, NSP8_pos_ct_v1, NSP9_pos_ct_v1, NSP10_pos_ct_v1, NSP11_pos_ct_v1, NSP12_pos_ct_v1, NSP13_pos_ct_v1, NSP14_pos_ct_v1, NSP15_pos_ct_v1, NSP16_pos_ct_v1]
#######################################################
    NSP_pos_ct = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_array)
        NSP_dict = NSP_pos_ct_array[i]
        NSP_dict_v1 = NSP_pos_ct_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_dict[NSP_pos] = 0
            NSP_dict_v1[j] = 0
            NSP_pos_ct[NSP_pos] = 0
        end
    end
##########################################################################################################################
##########################################################################################################################
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            NSP_num = parse(Int, split(NSP, "P")[2])
            NSP_dict = NSP_ct_dict_arr[NSP_num]
            NSP_dict[NSP_sub] = ct
        end
    end
##########################################################################################################################
##########################################################################################################################
    for i in 1:length(NSP_ct_dict_arr)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = NSP_muts_pos_dict[sub]
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct[sub_pos] += ct
        end
    end
    global NSP1_pos_ct_sort_by_pos = sort(collect(NSP1_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_sort_by_pos = sort(collect(NSP2_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_sort_by_pos = sort(collect(NSP3_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_sort_by_pos = sort(collect(NSP4_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_sort_by_pos = sort(collect(NSP5_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_sort_by_pos = sort(collect(NSP6_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_sort_by_pos = sort(collect(NSP7_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_sort_by_pos = sort(collect(NSP8_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_sort_by_pos = sort(collect(NSP9_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_sort_by_pos = sort(collect(NSP10_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_sort_by_pos = sort(collect(NSP11_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_sort_by_pos = sort(collect(NSP12_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_sort_by_pos = sort(collect(NSP13_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_sort_by_pos = sort(collect(NSP14_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_sort_by_pos = sort(collect(NSP15_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_sort_by_pos = sort(collect(NSP16_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
#########################
    global NSP1_pos_ct_sort_by_ct = sort(collect(NSP1_pos_ct), by = x -> x[2], rev=true)
    global NSP2_pos_ct_sort_by_ct = sort(collect(NSP2_pos_ct), by = x -> x[2], rev=true)
    global NSP3_pos_ct_sort_by_ct = sort(collect(NSP3_pos_ct), by = x -> x[2], rev=true)
    global NSP4_pos_ct_sort_by_ct = sort(collect(NSP4_pos_ct), by = x -> x[2], rev=true)
    global NSP5_pos_ct_sort_by_ct = sort(collect(NSP5_pos_ct), by = x -> x[2], rev=true)
    global NSP6_pos_ct_sort_by_ct = sort(collect(NSP6_pos_ct), by = x -> x[2], rev=true)
    global NSP7_pos_ct_sort_by_ct = sort(collect(NSP7_pos_ct), by = x -> x[2], rev=true)
    global NSP8_pos_ct_sort_by_ct = sort(collect(NSP8_pos_ct), by = x -> x[2], rev=true)
    global NSP9_pos_ct_sort_by_ct = sort(collect(NSP9_pos_ct), by = x -> x[2], rev=true)
    global NSP10_pos_ct_sort_by_ct = sort(collect(NSP10_pos_ct), by = x -> x[2], rev=true)
    global NSP11_pos_ct_sort_by_ct = sort(collect(NSP11_pos_ct), by = x -> x[2], rev=true)
    global NSP12_pos_ct_sort_by_ct = sort(collect(NSP12_pos_ct), by = x -> x[2], rev=true)
    global NSP13_pos_ct_sort_by_ct = sort(collect(NSP13_pos_ct), by = x -> x[2], rev=true)
    global NSP14_pos_ct_sort_by_ct = sort(collect(NSP14_pos_ct), by = x -> x[2], rev=true)
    global NSP15_pos_ct_sort_by_ct = sort(collect(NSP15_pos_ct), by = x -> x[2], rev=true)
    global NSP16_pos_ct_sort_by_ct = sort(collect(NSP16_pos_ct), by = x -> x[2], rev=true)
#########################
    global NSP1_pos_ct_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_v1), by = x -> x[1])
    global NSP2_pos_ct_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_v1), by = x -> x[1])
    global NSP3_pos_ct_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_v1), by = x -> x[1])
    global NSP4_pos_ct_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_v1), by = x -> x[1])
    global NSP5_pos_ct_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_v1), by = x -> x[1])
    global NSP6_pos_ct_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_v1), by = x -> x[1])
    global NSP7_pos_ct_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_v1), by = x -> x[1])
    global NSP8_pos_ct_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_v1), by = x -> x[1])
    global NSP9_pos_ct_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_v1), by = x -> x[1])
    global NSP10_pos_ct_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_v1), by = x -> x[1])
    global NSP11_pos_ct_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_v1), by = x -> x[1])
    global NSP12_pos_ct_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_v1), by = x -> x[1])
    global NSP13_pos_ct_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_v1), by = x -> x[1])
    global NSP14_pos_ct_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_v1), by = x -> x[1])
    global NSP15_pos_ct_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_v1), by = x -> x[1])
    global NSP16_pos_ct_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_v1), by = x -> x[1])
end
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_ranks_all(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    ORF1a_ct_all = Dict{String, Int}()
    ORF1b_ct_all = Dict{String, Int}()
    S_ct_all = Dict{String, Int}()
    ORF3a_ct_all = Dict{String, Int}()
    E_ct_all = Dict{String, Int}()
    M_ct_all = Dict{String, Int}()
    ORF6_ct_all = Dict{String, Int}()
    ORF7a_ct_all = Dict{String, Int}()
    ORF7b_ct_all = Dict{String, Int}()
    ORF8_ct_all = Dict{String, Int}()
    N_ct_all = Dict{String, Int}()
    ORF9b_ct_all = Dict{String, Int}()
#############################################################
    #                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    gene_ct_dict_all_arr = [ORF1a_ct_all, ORF1b_ct_all, S_ct_all, ORF3a_ct_all, E_ct_all, M_ct_all, ORF6_ct_all, ORF7a_ct_all, ORF7b_ct_all, ORF8_ct_all, N_ct_all, ORF9b_ct_all]
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_all_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels_all
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct_all[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
    global ORF1a_ct_all_sort_by_pos = sort(collect(ORF1a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_all_sort_by_pos = sort(collect(ORF1b_ct_all), by = x -> fin_sortkey(x[1]))
    global S_ct_all_sort_by_pos = sort(collect(S_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_all_sort_by_pos = sort(collect(ORF3a_ct_all), by = x -> fin_sortkey(x[1]))
    global E_ct_all_sort_by_pos = sort(collect(E_ct_all), by = x -> fin_sortkey(x[1]))
    global M_ct_all_sort_by_pos = sort(collect(M_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_all_sort_by_pos = sort(collect(ORF6_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_all_sort_by_pos = sort(collect(ORF7a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_all_sort_by_pos = sort(collect(ORF7b_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_all_sort_by_pos = sort(collect(ORF8_ct_all), by = x -> fin_sortkey(x[1]))
    global N_ct_all_sort_by_pos = sort(collect(N_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_all_sort_by_pos = sort(collect(ORF9b_ct_all), by = x -> fin_sortkey(x[1]))

    global ORF1b_ct_all_sort_by_ct = sort(collect(ORF1b_ct_all), by = x -> x[2], rev=true)
    global ORF1a_ct_all_sort_by_ct = sort(collect(ORF1a_ct_all), by = x -> x[2], rev=true)
    global S_ct_all_sort_by_ct = sort(collect(S_ct_all), by = x -> x[2], rev=true)
    global ORF3a_ct_all_sort_by_ct = sort(collect(ORF3a_ct_all), by = x -> x[2], rev=true)
    global E_ct_all_sort_by_ct = sort(collect(E_ct_all), by = x -> x[2], rev=true)
    global M_ct_all_sort_by_ct = sort(collect(M_ct_all), by = x -> x[2], rev=true)
    global ORF6_ct_all_sort_by_ct = sort(collect(ORF6_ct_all), by = x -> x[2], rev=true)
    global ORF7a_ct_all_sort_by_ct = sort(collect(ORF7a_ct_all), by = x -> x[2], rev=true)
    global ORF7b_ct_all_sort_by_ct = sort(collect(ORF7b_ct_all), by = x -> x[2], rev=true)
    global ORF8_ct_all_sort_by_ct = sort(collect(ORF8_ct_all), by = x -> x[2], rev=true)
    global N_ct_all_sort_by_ct = sort(collect(N_ct_all), by = x -> x[2], rev=true)
    global ORF9b_ct_all_sort_by_ct = sort(collect(ORF9b_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_all(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int},  AA_muts_ct_pos_only_no_dels_all::Dict{String, Int})
    ORF1a_pos_ct_all = Dict{String, Int}()
    ORF1b_pos_ct_all = Dict{String, Int}()
    S_pos_ct_all = Dict{String, Int}()
    ORF3a_pos_ct_all = Dict{String, Int}()
    E_pos_ct_all = Dict{String, Int}()
    M_pos_ct_all = Dict{String, Int}()
    ORF6_pos_ct_all = Dict{String, Int}()
    ORF7a_pos_ct_all = Dict{String, Int}()
    ORF7b_pos_ct_all = Dict{String, Int}()
    ORF8_pos_ct_all = Dict{String, Int}()
    N_pos_ct_all = Dict{String, Int}()
    ORF9b_pos_ct_all = Dict{String, Int}()
    gene_ct_pos_all_dict_arr = [ORF1a_pos_ct_all, ORF1b_pos_ct_all, S_pos_ct_all, ORF3a_pos_ct_all, E_pos_ct_all, M_pos_ct_all, ORF6_pos_ct_all, ORF7a_pos_ct_all, ORF7b_pos_ct_all, ORF8_pos_ct_all, N_pos_ct_all, ORF9b_pos_ct_all] 
###################################################### 
    ORF1a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_all_v1 = Dict{Int, Int}()
    S_pos_ct_all_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_all_v1 = Dict{Int, Int}()
    E_pos_ct_all_v1 = Dict{Int, Int}()
    M_pos_ct_all_v1 = Dict{Int, Int}()
    ORF6_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_all_v1 = Dict{Int, Int}()
    ORF8_pos_ct_all_v1 = Dict{Int, Int}()
    N_pos_ct_all_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_all_v1 = Dict{Int, Int}()
    gene_pos_ct_all_v1_dict_arr = [ORF1a_pos_ct_all_v1, ORF1b_pos_ct_all_v1, S_pos_ct_all_v1, ORF3a_pos_ct_all_v1, E_pos_ct_all_v1, M_pos_ct_all_v1, ORF6_pos_ct_all_v1, ORF7a_pos_ct_all_v1, ORF7b_pos_ct_all_v1, ORF8_pos_ct_all_v1, N_pos_ct_all_v1, ORF9b_pos_ct_all_v1]
###################################################### 
    for i in 1:length(gene_ct_pos_all_dict_arr)
        dict = gene_ct_pos_all_dict_arr[i]
        dict_v1 = gene_pos_ct_all_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels_all
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct_all[mut] = ct
            ORF1a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct_all[mut] = ct
            ORF1b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct_all[mut] = ct
            S_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct_all[mut] = ct
            ORF3a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct_all[mut] = ct
            E_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct_all[mut] = ct
            M_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct_all[mut] = ct
            ORF6_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct_all[mut] = ct
            ORF7a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct_all[mut] = ct
            ORF7b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct_all[mut] = ct
            ORF8_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct_all[mut] = ct
            N_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct_all[mut] = ct
            ORF9b_pos_ct_all_v1[pos] = ct
        end
    end
    global ORF1a_pos_ct_all_sort_by_pos = sort(collect(ORF1a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_all_sort_by_pos = sort(collect(ORF1b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_all_sort_by_pos = sort(collect(S_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_all_sort_by_pos = sort(collect(ORF3a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_all_sort_by_pos = sort(collect(E_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]]) 
    global M_pos_ct_all_sort_by_pos = sort(collect(M_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_all_sort_by_pos = sort(collect(ORF6_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_all_sort_by_pos = sort(collect(ORF7a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_all_sort_by_pos = sort(collect(ORF7b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_all_sort_by_pos = sort(collect(ORF8_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_all_sort_by_pos = sort(collect(N_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])   
    global ORF9b_pos_ct_all_sort_by_pos = sort(collect(ORF9b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct = sort(collect(ORF1a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct = sort(collect(ORF1b_pos_ct_all), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct = sort(collect(S_pos_ct_all), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct = sort(collect(ORF3a_pos_ct_all), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct = sort(collect(E_pos_ct_all), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct = sort(collect(M_pos_ct_all), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct = sort(collect(ORF6_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct = sort(collect(ORF7a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct = sort(collect(ORF7b_pos_ct_all), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct = sort(collect(ORF8_pos_ct_all), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct = sort(collect(N_pos_ct_all), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct = sort(collect(ORF9b_pos_ct_all), by = x -> x[2], rev=true)
##################################
    global ORF1a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[1])
    global ORF1b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[1])
    global S_pos_ct_all_sort_by_pos_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[1])
    global ORF3a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[1])
    global E_pos_ct_all_sort_by_pos_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[1]) 
    global M_pos_ct_all_sort_by_pos_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[1])
    global ORF6_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[1])
    global ORF7a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[1])
    global ORF7b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[1])
    global ORF8_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[1])
    global N_pos_ct_all_sort_by_pos_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[1])   
    global ORF9b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[1])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function NSP_sub_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
#############################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            gene = aa_gene_comprehensive_dict[sub]
            if gene == "ORF1a" || gene == "ORF1b"
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
#############################
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
    global NSP1_ct_all_sort_by_pos = sort(collect(NSP1_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_all_sort_by_pos = sort(collect(NSP2_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_all_sort_by_pos = sort(collect(NSP3_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_all_sort_by_pos = sort(collect(NSP4_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_all_sort_by_pos = sort(collect(NSP5_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_all_sort_by_pos = sort(collect(NSP6_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_all_sort_by_pos = sort(collect(NSP7_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_all_sort_by_pos = sort(collect(NSP8_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_all_sort_by_pos = sort(collect(NSP9_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_all_sort_by_pos = sort(collect(NSP10_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_all_sort_by_pos = sort(collect(NSP11_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_all_sort_by_pos = sort(collect(NSP12_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_all_sort_by_pos = sort(collect(NSP13_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_all_sort_by_pos = sort(collect(NSP14_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_all_sort_by_pos = sort(collect(NSP15_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_all_sort_by_pos = sort(collect(NSP16_ct_all), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_all_sort_by_ct = sort(collect(NSP1_ct_all), by = x -> x[2], rev=true)
    global NSP2_ct_all_sort_by_ct = sort(collect(NSP2_ct_all), by = x -> x[2], rev=true)
    global NSP3_ct_all_sort_by_ct = sort(collect(NSP3_ct_all), by = x -> x[2], rev=true)
    global NSP4_ct_all_sort_by_ct = sort(collect(NSP4_ct_all), by = x -> x[2], rev=true)
    global NSP5_ct_all_sort_by_ct = sort(collect(NSP5_ct_all), by = x -> x[2], rev=true)
    global NSP6_ct_all_sort_by_ct = sort(collect(NSP6_ct_all), by = x -> x[2], rev=true)
    global NSP7_ct_all_sort_by_ct = sort(collect(NSP7_ct_all), by = x -> x[2], rev=true)
    global NSP8_ct_all_sort_by_ct = sort(collect(NSP8_ct_all), by = x -> x[2], rev=true)
    global NSP9_ct_all_sort_by_ct = sort(collect(NSP9_ct_all), by = x -> x[2], rev=true)
    global NSP10_ct_all_sort_by_ct = sort(collect(NSP10_ct_all), by = x -> x[2], rev=true)
    global NSP11_ct_all_sort_by_ct = sort(collect(NSP11_ct_all), by = x -> x[2], rev=true)
    global NSP12_ct_all_sort_by_ct = sort(collect(NSP12_ct_all), by = x -> x[2], rev=true)
    global NSP13_ct_all_sort_by_ct = sort(collect(NSP13_ct_all), by = x -> x[2], rev=true)
    global NSP14_ct_all_sort_by_ct = sort(collect(NSP14_ct_all), by = x -> x[2], rev=true)
    global NSP15_ct_all_sort_by_ct = sort(collect(NSP15_ct_all), by = x -> x[2], rev=true)
    global NSP16_ct_all_sort_by_ct = sort(collect(NSP16_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
# Removed from parameters: NSP_AA_size::Dict{String, Int}
function NSP_sub_pos_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_pos_ct_all = Dict{String, Int}()
    NSP2_pos_ct_all = Dict{String, Int}()
    NSP3_pos_ct_all = Dict{String, Int}()
    NSP4_pos_ct_all = Dict{String, Int}()
    NSP5_pos_ct_all = Dict{String, Int}()
    NSP6_pos_ct_all = Dict{String, Int}()
    NSP7_pos_ct_all = Dict{String, Int}()
    NSP8_pos_ct_all = Dict{String, Int}()
    NSP9_pos_ct_all = Dict{String, Int}()
    NSP10_pos_ct_all = Dict{String, Int}()
    NSP11_pos_ct_all = Dict{String, Int}()
    NSP12_pos_ct_all = Dict{String, Int}()
    NSP13_pos_ct_all = Dict{String, Int}()
    NSP14_pos_ct_all = Dict{String, Int}()
    NSP15_pos_ct_all = Dict{String, Int}()
    NSP16_pos_ct_all = Dict{String, Int}()
    NSP_pos_ct_all_array = [NSP1_pos_ct_all, NSP2_pos_ct_all, NSP3_pos_ct_all, NSP4_pos_ct_all, NSP5_pos_ct_all, NSP6_pos_ct_all, NSP7_pos_ct_all, NSP8_pos_ct_all, NSP9_pos_ct_all, NSP10_pos_ct_all, NSP11_pos_ct_all, NSP12_pos_ct_all, NSP13_pos_ct_all, NSP14_pos_ct_all, NSP15_pos_ct_all, NSP16_pos_ct_all]
#######################################################
    NSP1_pos_ct_all_v1 = Dict{Int, Int}()
    NSP2_pos_ct_all_v1 = Dict{Int, Int}()
    NSP3_pos_ct_all_v1 = Dict{Int, Int}()
    NSP4_pos_ct_all_v1 = Dict{Int, Int}()
    NSP5_pos_ct_all_v1 = Dict{Int, Int}()
    NSP6_pos_ct_all_v1 = Dict{Int, Int}()
    NSP7_pos_ct_all_v1 = Dict{Int, Int}()
    NSP8_pos_ct_all_v1 = Dict{Int, Int}()
    NSP9_pos_ct_all_v1 = Dict{Int, Int}()
    NSP10_pos_ct_all_v1 = Dict{Int, Int}()
    NSP11_pos_ct_all_v1 = Dict{Int, Int}()
    NSP12_pos_ct_all_v1 = Dict{Int, Int}()
    NSP13_pos_ct_all_v1 = Dict{Int, Int}()
    NSP14_pos_ct_all_v1 = Dict{Int, Int}()
    NSP15_pos_ct_all_v1 = Dict{Int, Int}()
    NSP16_pos_ct_all_v1 = Dict{Int, Int}()
    NSP_pos_ct_all_v1_array = [NSP1_pos_ct_all_v1, NSP2_pos_ct_all_v1, NSP3_pos_ct_all_v1, NSP4_pos_ct_all_v1, NSP5_pos_ct_all_v1, NSP6_pos_ct_all_v1, NSP7_pos_ct_all_v1, NSP8_pos_ct_all_v1, NSP9_pos_ct_all_v1, NSP10_pos_ct_all_v1, NSP11_pos_ct_all_v1, NSP12_pos_ct_all_v1, NSP13_pos_ct_all_v1, NSP14_pos_ct_all_v1, NSP15_pos_ct_all_v1, NSP16_pos_ct_all_v1]
#######################################################
    NSP_pos_ct_all = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_all_array)
        NSP_all_dict = NSP_pos_ct_all_array[i]
        NSP_all_dict_v1 = NSP_pos_ct_all_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_all_dict[NSP_pos] = 0
            NSP_pos_ct_all[NSP_pos] = 0
            NSP_all_dict_v1[j] = 0
        end
    end
############################################################
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
##################################################################################################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = parse(Int, split(sub, ":")[2][2:end-1])
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
###################################################################################################################################
    for i in 1:length(NSP_pos_ct_all_array)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_all_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_all_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_all_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = parse(Int, split(sub, ":")[2][2:end-1])
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct_all[sub_pos] += ct
        end
    end
##############################################################
    global NSP1_pos_ct_all_sort_by_pos = sort(collect(NSP1_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_all_sort_by_pos = sort(collect(NSP2_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_all_sort_by_pos = sort(collect(NSP3_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_all_sort_by_pos = sort(collect(NSP4_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_all_sort_by_pos = sort(collect(NSP5_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_all_sort_by_pos = sort(collect(NSP6_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_all_sort_by_pos = sort(collect(NSP7_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_all_sort_by_pos = sort(collect(NSP8_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_all_sort_by_pos = sort(collect(NSP9_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_all_sort_by_pos = sort(collect(NSP10_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_all_sort_by_pos = sort(collect(NSP11_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_all_sort_by_pos = sort(collect(NSP12_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_all_sort_by_pos = sort(collect(NSP13_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_all_sort_by_pos = sort(collect(NSP14_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_all_sort_by_pos = sort(collect(NSP15_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_all_sort_by_pos = sort(collect(NSP16_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
#############################
    global NSP1_pos_ct_all_sort_by_ct = sort(collect(NSP1_pos_ct_all), by = x -> x[2], rev=true)
    global NSP2_pos_ct_all_sort_by_ct = sort(collect(NSP2_pos_ct_all), by = x -> x[2], rev=true)
    global NSP3_pos_ct_all_sort_by_ct = sort(collect(NSP3_pos_ct_all), by = x -> x[2], rev=true)
    global NSP4_pos_ct_all_sort_by_ct = sort(collect(NSP4_pos_ct_all), by = x -> x[2], rev=true)
    global NSP5_pos_ct_all_sort_by_ct = sort(collect(NSP5_pos_ct_all), by = x -> x[2], rev=true)
    global NSP6_pos_ct_all_sort_by_ct = sort(collect(NSP6_pos_ct_all), by = x -> x[2], rev=true)
    global NSP7_pos_ct_all_sort_by_ct = sort(collect(NSP7_pos_ct_all), by = x -> x[2], rev=true)
    global NSP8_pos_ct_all_sort_by_ct = sort(collect(NSP8_pos_ct_all), by = x -> x[2], rev=true)
    global NSP9_pos_ct_all_sort_by_ct = sort(collect(NSP9_pos_ct_all), by = x -> x[2], rev=true)
    global NSP10_pos_ct_all_sort_by_ct = sort(collect(NSP10_pos_ct_all), by = x -> x[2], rev=true)
    global NSP11_pos_ct_all_sort_by_ct = sort(collect(NSP11_pos_ct_all), by = x -> x[2], rev=true)
    global NSP12_pos_ct_all_sort_by_ct = sort(collect(NSP12_pos_ct_all), by = x -> x[2], rev=true)
    global NSP13_pos_ct_all_sort_by_ct = sort(collect(NSP13_pos_ct_all), by = x -> x[2], rev=true)
    global NSP14_pos_ct_all_sort_by_ct = sort(collect(NSP14_pos_ct_all), by = x -> x[2], rev=true)
    global NSP15_pos_ct_all_sort_by_ct = sort(collect(NSP15_pos_ct_all), by = x -> x[2], rev=true)
    global NSP16_pos_ct_all_sort_by_ct = sort(collect(NSP16_pos_ct_all), by = x -> x[2], rev=true)
###############################
    global NSP1_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_all_v1), by = x -> x[1])
    global NSP2_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_all_v1), by = x -> x[1])
    global NSP3_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_all_v1), by = x -> x[1])
    global NSP4_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_all_v1), by = x -> x[1])
    global NSP5_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_all_v1), by = x -> x[1])
    global NSP6_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_all_v1), by = x -> x[1])
    global NSP7_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_all_v1), by = x -> x[1])
    global NSP8_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_all_v1), by = x -> x[1])
    global NSP9_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_all_v1), by = x -> x[1])
    global NSP10_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_all_v1), by = x -> x[1])
    global NSP11_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_all_v1), by = x -> x[1])
    global NSP12_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_all_v1), by = x -> x[1])
    global NSP13_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_all_v1), by = x -> x[1])
    global NSP14_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_all_v1), by = x -> x[1])
    global NSP15_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_all_v1), by = x -> x[1])
    global NSP16_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_all_v1), by = x -> x[1])
end
######################################################################################################################################
runtime = time() - start_chr_load_fx
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime); print("\n"^1)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_04__613PM

Runtime v0 = 0.9016261100769043 seconds
Runtime v2 = 0 hr, 0 min, 00.90 sec



In [366]:
### Execute Load Chronic Dicts *** SPECIAL BAL VERSION **** DQ Fx | 2026_02_22 version | chronics_2026_02_01__6153seq | 95—8—6 | Runtime: 1 min, 00.26 sec | 2026_2_13
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
chr_dict_load_start = time()

abs_min_mut_thresh = 8
DQ_mut_thresh = 10
date_pct_DQ_thresh = 95

rep_thresh = 5
revs_thresh = 6
EPCI_qc_str = "$(abs_min_mut_thresh)_$(DQ_mut_thresh)_$(date_pct_DQ_thresh)"

print_ct_thresh = 1
date = "2026_02_20"
ndjson_name = "chronics_2026_02_01__6153seq_v2"
folder_name = "$(date)__$(ndjson_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_str = $(HQCS_qc_str)")

chronic_load_dicts2_DQ(ndjson_name, folder_name, date, rep_thresh, revs_thresh, print_ct_thresh, DQ_mut_thresh, date_pct_DQ_thresh, abs_min_mut_thresh)
# ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Float64, abs_min_mut_thresh::Int
######################################################################################################################################
seq_AA_muts["EPI_ISL_8725398"] = Set{String}();             seq_AA_muts["EPI_ISL_949208"] = Set{String}();   seq_pango["EPI_ISL_6281381"] = "AY.4"
######################################################################################################################################
### Create  all_unique_chr_seqs  &  all_unique_chr_seqs_set 
rep_seq_grps_maxmut_seqs_arr = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs = union(rep_seq_grps_maxmut_seqs_arr, non_rep_seqs)
all_unique_chr_seqs_ct = length(all_unique_chr_seqs)
all_unique_chr_seqs_set = Set{String}()
for seq in all_unique_chr_seqs
    push!(all_unique_chr_seqs_set, seq)
end
all_unique_chr_seqs_set_copy = all_unique_chr_seqs_set
println("length(all_unique_chr_seqs) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set)  $(length(all_unique_chr_seqs_set))"); print("\n"^2)
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#### These sequences are related to confirmed BAL seqs (usually same patient) but are not labeled BAL. Because only one
#  sequence from each group of related chronics is used for analysis, they have to be excluded.
non_BAL_related_seqs_to_exclude = Set(["EPI_ISL_2598472", "EPI_ISL_4936533", "EPI_ISL_14518101", "EPI_ISL_14596883", "EPI_ISL_14935908", "EPI_ISL_15251240", "EPI_ISL_15251242", "EPI_ISL_15251243", "EPI_ISL_15511841", "EPI_ISL_15719142", "EPI_ISL_17960747", "EPI_ISL_18059074", "EPI_ISL_18059075", "EPI_ISL_18059076", "EPI_ISL_18136392", "EPI_ISL_18274346", "EPI_ISL_18515316"])
all_unique_chr_seqs_set = Set{String}()
for seq in all_unique_chr_seqs_set_copy
    if !(seq in non_BAL_related_seqs_to_exclude)
        push!(all_unique_chr_seqs_set, seq)
    end
end
println("length(all_unique_chr_seqs)) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set_copy)) = $(length(all_unique_chr_seqs_set_copy))")
println("length(all_unique_chr_seqs_set)) = $(length(all_unique_chr_seqs_set))")
#####################################################################################################################################
###### These are confirmed BAL seqs that did not make the unique chronic seqs list b/c of the related seqs (now excluded, see above)
######  or, in in two cases (EPI_ISL_19878655, EPI_ISL_17964486), for not meeting the minimum AA-mutation requirement
############# NEW (added 2025-11-23: EPI_ISL_17719124, EPI_ISL_17964486) ##########
#push!(all_unique_chr_seqs_set, "EPI_ISL_7473134")
push!(all_unique_chr_seqs_set, "EPI_ISL_4935949")
push!(all_unique_chr_seqs_set, "EPI_ISL_1908476")
push!(all_unique_chr_seqs_set, "EPI_ISL_14479735")
push!(all_unique_chr_seqs_set, "EPI_ISL_14935931")
push!(all_unique_chr_seqs_set, "EPI_ISL_17960749")
push!(all_unique_chr_seqs_set, "EPI_ISL_19878655")
push!(all_unique_chr_seqs_set, "EPI_ISL_17719124")
push!(all_unique_chr_seqs_set, "EPI_ISL_17964486")
println("After adding BAL seqs...")
println("length(all_unique_chr_seqs)) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set_copy)) = $(length(all_unique_chr_seqs_set_copy))")
println("length(all_unique_chr_seqs_set)) = $(length(all_unique_chr_seqs_set))")
######################################################################################
#seq_AA_muts["EPI_ISL_17719124"] = Set(["E:L27S", "ORF1a:Q1759R", "ORF1a:A3615T", "ORF1a:F3769L", "ORF1a:T4161I", "ORF1b:V728I", "ORF1b:I2114T", "S:S247N", "S:F342L", "S:R765C", "S:K1045R", "N:K299I"])
#seq_AA_muts_no_dels["EPI_ISL_17719124"] = Set(["E:L27S", "ORF1a:Q1759R", "ORF1a:A3615T", "ORF1a:F3769L", "ORF1a:T4161I", "ORF1b:V728I", "ORF1b:I2114T", "S:S247N", "S:F342L", "S:R765C", "S:K1045R", "N:K299I"])
#seq_AA_muts_pos_only_no_dels["EPI_ISL_17719124"] = Set(["E:27", "ORF1a:1759", "ORF1a:3615", "ORF1a:3769", "ORF1a:4161", "ORF1b:728", "ORF1b:2114", "S:247", "S:342", "S:765", "S:1045", "N:299"])
#seq_unknown_AA["EPI_ISL_17719124"] = Set(["10626"])        
#push!(all_unique_chr_seqs_set, "EPI_ISL_17719124")
#seq_unknown_AA["EPI_ISL_17719124"] = Set(["10626"])    
#seq_pango["EPI_ISL_17719124"] = "XBB.6.1"
#seq_pango_unaliased["EPI_ISL_17719124"] = "XBB.6.1"
#seq_clade["EPI_ISL_17719124"] = "22F"
######################################################################################
#seq_AA_muts["EPI_ISL_17964486"] = Set(["E:L27S", "ORF1a:K3365R", "ORF1a:I4205V", "ORF1b:A440V", "S:F374L", "S:A653V"])
#seq_AA_muts_no_dels["EPI_ISL_17964486"] = Set(["E:L27S", "ORF1a:K3365R", "ORF1a:I4205V", "ORF1b:A440V", "S:F374L", "S:A653V"])
#seq_AA_muts_pos_only_no_dels["EPI_ISL_17964486"] = Set(["E:27", "ORF1a:3365", "ORF1a:4205", "ORF1b:440", "S:374", "S:653"])
#seq_unknown_AA["EPI_ISL_17964486"] = Set([""])
#push!(all_unique_chr_seqs_set, "EPI_ISL_17964486")
#seq_unknown_AA["EPI_ISL_17964486"] = Set([""])
#seq_pango["EPI_ISL_17964486"] = "XBB.1.5.24"
#seq_pango_unaliased["EPI_ISL_17964486"] = "XBB.1.5.24"
#seq_clade["EPI_ISL_17964486"] = "23A"
######################################################################################
total_chronic_count_BAL = length(all_unique_chr_seqs_set)
println("total_chronic_count_BAL = $(total_chronic_count_BAL)")
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#### Because a few sequences were removed from the EPCI list and a few known BAL were added, the AA_muts_ct_no_dels dictionary
##   needs to be updated. The changes are very minor conidering there are >3000 seqs overall, & that the sequences being replaced
##   are very similar to the ones replacing them, but might as well be thorough.
AA_muts_seq = Dict{String, Set{String}}()
AA_muts_seq_pos_only = Dict{String, Set{String}}()
AA_muts_ct = Dict{String, Int}()
AA_muts_ct_no_dels = Dict{String, Int}()
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts_no_dels[seq]
        AA_muts_ct_no_dels[mut] = get(AA_muts_ct_no_dels, mut, 0) + 1
    end
    for mut in seq_AA_muts[seq]
        AA_muts_ct[mut] = get(AA_muts_ct, mut, 0) + 1
        AA_muts_seq[mut] = get(AA_muts_seq, mut, Set{String}())
        push!(AA_muts_seq[mut], seq)
        mut_po = aa_gene_and_pos_comprehensive_dict[mut]
        AA_muts_seq_pos_only[mut_po] = get(AA_muts_seq_pos_only, mut_po, Set{String}())
        push!(AA_muts_seq_pos_only[mut_po], seq)
    end
end
################################################
AA_muts_ct_pos_only_no_dels = Dict{String, Int}()
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts_pos_only_no_dels[seq]
        AA_muts_ct_pos_only_no_dels[mut] = get(AA_muts_ct_pos_only_no_dels, mut, 0) + 1
    end
end
################################################
total_chronics_AA_mut_ct = 0
for (mut, ct) in AA_muts_ct_no_dels
    total_chronics_AA_mut_ct += ct
end
total_chronics_AA_mut_ct_v2 = 0
for seq in all_unique_chr_seqs_set
    for seq_AA_mut_ct = length(seq_AA_muts_no_dels[seq])
        total_chronics_AA_mut_ct_v2 += seq_AA_mut_ct
    end
end
println("total_chronics_AA_mut_ct version 1 = $(total_chronics_AA_mut_ct)")
println("total_chronics_AA_mut_ct version 2 = $(total_chronics_AA_mut_ct_v2)")
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
if haskey(AA_muts_ct_pos_only_no_dels, "")
    delete!(AA_muts_ct_pos_only_no_dels, "")
end
if haskey(AA_muts_ct_no_dels, "")
    delete!(AA_muts_ct_no_dels, "")
end
######################################################################################################################################
chronic_pango_set = Set{String}()
for seq in all_unique_chr_seqs_set
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
for seq in all_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
end
############################################################################################################################################################################
#### Removing clearly artifactual reversions that occur directly next to dropout (and are almost always from labs that frequently have this problem)
FCS_fake_revs = Set(["S:K679N", "S:H681P", "S:R681P"])
FCS_fake_revs_pos = Set(["S:679", "S:681"])
for seq in all_qualifying_seqs_set
    if "S:683" in seq_unknown_AA[seq] || "S:676" in seq_unknown_AA[seq]
        if seq in all_unique_chr_seqs_set
            for mut in FCS_fake_revs
                if mut in seq_AA_muts[seq]
                    mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    AA_muts_ct[mut] -= 1
                    AA_muts_ct_no_dels[mut] -= 1
                    AA_muts_ct_pos_only[mut_pos] -= 1
                    AA_muts_ct_pos_only_no_dels[mut_pos] -= 1
                end
            end
        end
        setdiff!(seq_AA_muts[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_no_dels[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_pos_only[seq], FCS_fake_revs_pos)
        setdiff!(seq_AA_muts_pos_only_no_dels[seq], FCS_fake_revs_pos)
    end
end
######################################################################################################################################
## Deletions royally screw up any attempt to find correlated mutations since they frequently occur in bunches, which are, of course, highly correlated, but only in the most trivial way. 
## The code below is a preliminary attempt to include common deletions by only allowing the inclusion of a single AA deletion, though most contain multiple consecutive 
## AA deletions. Also included are mutations that only occur as 1-AA deletions, such as E:I13- and E:V14-. It can be removed or inserted without substantially changing the results 
deletion_exceptions = list_to_strings_set("E:I13-, E:V14-, M:G6-, S:C15-, S:C136-, S:D138-, S:A243-, S:F371-, S:F375-, S:V483-, S:A484-, S:E484-, S:D1257-, ORF3a:T14-, ORF3a:V255-, ORF3a:V256-, ORF3a:N257-, ORF7a:I103-, ORF7a:*122-, ORF1a:N2081-, ORF1a:D2136-, ORF1a:S4398-")
deletion_exceptions_positions = list_to_strings_set("E:13, E:14, M:6, S:15, S:136, S:138, S:243, S:371, S:375, S:483, S:484, S:1257, ORF3a:14, ORF3a:255, ORF3a:256, ORF3a:257, ORF7a:103, ORF7a:122, ORF1a:2081, ORF1a:2136, ORF1a:4398")
pos_only_deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
del_to_del_pos = Dict{String, String}("E:I13-"=>"E:13", "E:V14-"=>"E:14", "M:G6-"=>"M:6", "S:C15-"=>"S:15", "S:C136-"=>"S:136", "S:D138-"=>"S:138", "S:A243-"=>"S:243", "S:F371-"=>"S:371", "S:F375-"=>"S:375", "S:V483-"=>"S:483", "S:A484-"=>"S:484", "S:E484-"=>"S:484", "S:D1257-"=>"S:1257", "ORF3a:T14-"=>"ORF3a:14", "ORF3a:V255-"=>"ORF3a:255", "ORF3a:V256-"=>"ORF3a:256", "ORF3a:N257-"=>"ORF3a:257", "ORF7a:I103-"=>"ORF7a:103", "ORF7a:*122-"=>"ORF7a:122", "ORF1a:N2081-"=>"ORF1a:2081", "ORF1a:D2136-"=>"ORF1a:2136", "ORF1a:S4398-"=>"ORF1a:4398")
pos_only_exception_count::Int = 0
for seq in all_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            del_pos = del_to_del_pos[del]
            push!(seq_AA_muts_pos_only_no_dels[seq], del_pos)
            pos_only_exception_count += 1
            pos_only_deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    del_pos = del_to_del_pos[del]
    if !haskey(AA_muts_ct_pos_only_no_dels, del_pos)
        AA_muts_ct_pos_only_no_dels[del_pos] = 0
    end
    AA_muts_ct_pos_only_no_dels[del_pos] += pos_only_deletion_exception_ct_dict[del]
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[del_pos] = 999
end
######################################################################################################################################
deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
exception_count::Int = 0
for seq in all_unique_chr_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            push!(seq_AA_muts_no_dels[seq], del)
            exception_count += 1
            deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    AA_muts_ct_no_dels[del] = get(AA_muts_ct_no_dels, del, 0)
    AA_muts_ct_no_dels[del] += deletion_exception_ct_dict[del]
    AA_muts_ct_chr_all_ratio[del] = 999
end
######################################################################################################################################
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
for blnk in blank_mutstrings
    AA_muts_ct_chr_all_ratio[blnk] = 0
    AA_muts_ct_no_dels_chr_all_ratio[blnk] = 0
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[blnk] = 0
end
#######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
######################################################################################################################################
mp_AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_sortKey2(n) = (n[2], mp_AA_ct_sortKey1(n))
#########
mp_AA_gene_pos_only_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_pos_only_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_pos_only_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_pos_only_sortKey2(n) = (n[2], mp_AA_ct_pos_only_sortKey1(n))
#################################
function mp_AApos_sort_key(n)
    if haskey(aa_pos_comprehensive_dict, n)
        return (aa_pos_comprehensive_dict[n], n)
    else
        return (9999, n)
    end
end
function sel_muts_pt1_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_sortKey_2(string(split(n, ", ")[1]))
    end
end
##################################################################
println("Number of Deletion Exceptions Made = $(exception_count)")
deletion_exception_ct_dict_sort = sort(collect(deletion_exception_ct_dict), by = x -> x[2])
for del__ct in deletion_exception_ct_dict_sort
    del = del__ct[1]
    ct = del__ct[2]
    println("$(del) = $(ct)")
end
###########################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    seq_privAA_len[seq] = seqAAlen
end
############################################################################################################################################################################
############################################################################################################################################################################
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts[seq]
        if mut in deletion_exceptions
            mut_po = aa_gene_and_pos_comprehensive_dict[mut]
            if mut_po in deletion_exceptions_positions
                AA_muts_seq_pos_only[mut_po] = get(AA_muts_seq_pos_only, mut_po, Set{String}())
                push!(AA_muts_seq_pos_only[mut_po], seq)
            end
        end
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
############################################################################################################################################################################
############################################################################################################################################################################
corrected_count = 0
noncorrected_ct = 0
for seq in all_seqs_set
    if seq_date_index[seq] > 4000
#        println("seq_date_tuple = $(seq_date_tuple[seq]); seq_date_tuple[seq][1] = $(seq_date_tuple[seq][1]); seq_date_tuple[seq][2] = $(seq_date_tuple[seq][2]); seq_date_tuple[seq][3] = $(seq_date_tuple[seq][3])")
        if seq_date_tuple[seq][1] ≠ 0 && seq_date_tuple[seq][2] ≠ 0
            new_date_tuple = (seq_date_tuple[seq][1], seq_date_tuple[seq][2], 15)
            seq_date_tuple[seq] = new_date_tuple
            seq_date_index[seq] = tuple_to_index[new_date_tuple]
            corrected_count += 1
        elseif seq_date_tuple[seq][2] == 0
            noncorrected_ct += 1
        end
    end
end
total_baddie_ct = corrected_count + noncorrected_ct
println("seq_date_index corrected = $(corrected_count)")
println("seq_date_index not corrected = $(noncorrected_ct)")
println("total_baddie_ct = $(total_baddie_ct)")
############################################################################################################################################################################
############################################################################################################################################################################  
chr_load_runtime = time() - chr_dict_load_start
chr_load_runtime_rd = round(digits=1, chr_load_runtime)
chr_load_hms1, chr_load_hms2 = seconds_to_hrs_min_sec(chr_load_runtime)
println("Total Time to Load Chronic Dictionaries = $(chr_load_runtime_rd)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms1)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms2)"); print("\n"^1)
println("Finished!")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_04__613PM
EPCI_qc_str = 8_10_95 | HQCS_qc_str = 5_1_5
2026_03_04__613PM
length(all_unique_chr_seqs) = 3297
length(all_unique_chr_seqs_set)  3297


length(all_unique_chr_seqs)) = 3297
length(all_unique_chr_seqs_set_copy)) = 3297
length(all_unique_chr_seqs_set)) = 3293
After adding BAL seqs...
length(all_unique_chr_seqs)) = 3297
length(all_unique_chr_seqs_set_copy)) = 3297
length(all_unique_chr_seqs_set)) = 3297
total_chronic_count_BAL = 3297
total_chronics_AA_mut_ct version 1 = 58702
total_chronics_AA_mut_ct version 2 = 58702
Number of Deletion Exceptions Made = 838
S:E484- = 1
ORF1a:D2136- = 2
S:F371- = 4
ORF1a:N2081- = 6
ORF1a:S4398- = 7
E:I13- = 8
S:D1257- = 8
S:V483- = 9
ORF7a:I103- = 10
M:G6- = 12
ORF7a:*122- = 16
S:A484- = 23
S:F375- = 24
S:C136- = 32
ORF3a:V256- = 38
ORF3a:T14- = 40
ORF3a:N257- = 43
ORF3a:V255- = 51
S:D138- = 89
E:V14- = 90
S:C15- = 105
S:A243- = 220
seq_date_index corrected = 70
seq_date_index not corrected = 52
total_baddie_ct = 122
Total Time to Load C

In [367]:
### NEW | 2026_02_22 | Many, many functions (mixed_nuc, nuc_to_AA, date_pct_fx, etc) | Runtime: 1 min 33 sec
### Timing Template
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
start = time()
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
function list_to_strings_no_spaces(list::String)
    string_vec = string.(split(list, ","))
    return string_vec
end
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function get_ref_pango_nucseq_and_geneseqs(ref_pango::String)
    ref_seq = "..."
    refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
    refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
    refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
    refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
    refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
    refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
    refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
    refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
    refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
    refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
    refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
    refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
    if ref_pango ≠ "Wuhan"
        if haskey(nuc_genome_pango_dict, ref_pango)
            ref_seq = nuc_genome_pango_dict[ref_pango]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        end
##########################################################################################
        if haskey(gene_AA_pango_dict, ref_pango)
            refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
            refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
            refAA_S = gene_AA_pango_dict[ref_pango]["S"]
            refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
            refAA_E = gene_AA_pango_dict[ref_pango]["E"]
            refAA_M = gene_AA_pango_dict[ref_pango]["M"]
            refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
            refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
            refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
            refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
            refAA_N = gene_AA_pango_dict[ref_pango]["N"]
            refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        end
    end
    return ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(["Y", "R", "K", "M", "W", "S"])
    for mut in mut_arr
        if string(mut[end]) in mixed_nuc_res_list
            push!(mixed_nuc_arr, mut)
        end
    end
    return mixed_nuc_arr
end
#####################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    if length(mix_mut) ≥ 4
        if qrynuc == "T"
            nuc_mut = mix_mut
        elseif qrynuc == "C"
            nuc_mut = mix_mut
        elseif qrynuc == "A"
            nuc_mut = mix_mut
        elseif qrynuc == "G"
            nuc_mut = mix_mut
        elseif qrynuc == "N"
            nuc_mut = mix_mut
        end   
        if refnuc == "T"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if refnuc == "A"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if refnuc == "G"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
function muts_to_strings(mut_list_in_string_form::String)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for mut in mut_arr
        if string(mut[end]) ≠ "-"
            mutstr = string(mut)
            push!(mut_array, mutstr)
        end
    end
    sortKey(n) = (length(n), nuc_mut_int_comprehensive_dict[n])  ## Fx ##
    mut_array_sort = sort(mut_array, by = x -> sortKey(x))
#    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ### New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        mut = mixed_nuc_muts[i]
        qrynuc = qry_nuc_comprehensive_dict[mut]
        refnuc = ref_nuc_comprehensive_dict[mut]
        pos_str = nuc_mut_int_string_comprehensive_dict[mut]
        ref_n_pos = refnuc*pos_str
        if refnuc == "T"
            if qrynuc == "Y"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if refnuc == "A"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "G"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts_filtered = filter(!isempty, muts)
#    all_muts_sort = sort(muts_filtered, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
################################################        
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end

    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
    
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                                          ### FUNCTION #####################
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)                  ### FUNCTION #####################
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                                            ### FUNCTION #####################
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA   ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
    
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#######################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end
            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    noncoding_to_noncoding_region_dict[nuc_mut] = place
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#################################################################################
    mut_vec_dict = Dict{String, Vector{String}}()
    for gene in gene_print_array
        mut_vec_dict[gene] = Vector{String}()
    end
    for mut in AA_mut_set
        jeen = aa_gene_comprehensive_dict[mut]
        mut_only = string(split(mut, ":")[2])
        push!(mut_vec_dict[jeen], mut_only) 
    end
    for gene in keys(mut_vec_dict)
        if !isempty(mut_vec_dict[gene])
            sort!(mut_vec_dict[gene], by = x -> x[2:end-1])
        end
    end
#####################################################################################################################################
    aasortkeyhere(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
    AA_sort = sort(collect(AA_mut_set), by = x -> aasortkeyhere(x))
    AA_sort2 = Vector{String}()
    for mut in AA_sort
        refAA = refAA_comprehensive_dict[mut]
        qryAA = qryAA_comprehensive_dict[mut]
        if !(refAA == qryAA)
            push!(AA_sort2, mut)
        end
    end
#    print("\n"^2)
#    println("##################### Amino Acid Mutations ######################")
#    for i in 1:length(AA_sort2)
#        mut = AA_sort2[i]
#        gene = aa_gene_comprehensive_dict[mut]
#        non_gene = string(split(mut, ":")[2])
#        if i == 1
#            print("               ", AA_sort2[i])
#        elseif i > 1
#            lastmut = AA_sort2[i-1]
#            last_gene = aa_gene_comprehensive_dict[lastmut]
#            last_non_gene = string(split(lastmut, ":")[2])
#            if gene == last_gene
#                print(", $(non_gene)")
#            else
#                println()
#                print("               $(mut)")
#            end
#        end
#    end
#    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
####################################################################################################################################
    nc_len = length(noncoding_to_noncoding_region_dict_sort)
#    println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
    if !isempty(noncoding_to_noncoding_region_dict_sort)
        for i in 1:length(noncoding_to_noncoding_region_dict_sort)
            nc_nuc = noncoding_to_noncoding_region_dict_sort[i][1]
            nc_nuc_pad = rpad(nc_nuc, 7)
            nc_region = noncoding_to_noncoding_region_dict_sort[i][2]
            nc_region_len = length(nc_region)
            ncpad1 = (13 - nc_region_len)÷2
            ncpad12 = " "^ncpad1*nc_region
            nc_region_pad2 = rpad(ncpad12, 13)
            nc_context = noncoding_nuc_to_context_dict_sort[i][2]
            premut_context = ""
            postmut_context = ""
            context = split(nc_context, "-->")
            if !isempty(context)
                if length(context) >1
                    premut_context = split(nc_context, "-->")[1]
                    postmut_context = split(nc_context, "-->")[2]
                end
            end    
            postpad = lpad(postmut_context, 39)
#            println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#            println(postpad)
#                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#                println(g, postpad)
        end
#        println("#"^94); println() 
#            println(g, "#"^94); println(g)
    end
######################################################################################################
    total_syn = length(synonymous_nuc_to_AA_dict_sort)
#    println("SYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(synonymous_nuc_to_AA_dict_sort)
        synnuc = synonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = synonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = synonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
#################################################################
    total_syn = length(nonsynonymous_nuc_to_AA_dict_sort)
#    println("NONSYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(nonsynonymous_nuc_to_AA_dict_sort)
        synnuc = nonsynonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = nonsynonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = nonsynonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
###################################################################
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
#    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
#    println("################ Nonsynonymous Nuc Mutations ################")
#    println(nonsynonymous_nuc_sort_join)
#    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
#        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        println("                 $(premut_nucseq)")
#        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ### New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ### New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
#    println("Total Mixed Nucs = $(total_mixed_muts)")
#    print("\n"^2)
    for i in 1:length(mixed_muts)
        if ct == 0
#            print(mixed_muts[i])
            ct = 1
        else
#            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
#    print("\n"^2)
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
#            print(mixed_muts_regular[i])
            ct2 = 1
#        else
#            print(", ", mixed_muts_regular[i])
        end
    end
#    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
#    println("\n"^1)
#    println(aapad_join)
#    println(nucpad_join)
#    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = ref_nuc_comprehensive_dict[nuc_mut]
        qry = ref_nuc_comprehensive_dict[nuc_mut]
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function AA_triple(pos::Int, rem0::BitSet, rem1::BitSet, rem2::BitSet, mut::String, ref_pango::String)
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ref_triple = ""
    qry_triple = ""
    if pos in rem0 && pos%3 == 0
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem1 && pos%3 == 1
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem2 && pos%3 == 2
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem0 && pos%3 == 1
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem1 && pos%3 == 2
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem2 && pos%3 == 0
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem0 && pos%3 == 2
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem1 && pos%3 == 0
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem2 && pos%3 == 1
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    end
    return ref_triple, qry_triple
end
######################################################################################################################################
### Fx: SIMPLER_nuc_to_AA (no context)
function SIMPLER_nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts = filter(!isempty, muts)
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ###############################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
        all_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
        N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut_set = Set{String}()
        AA_mut = ""
        for nuc_mut in muts
            pos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos in coding_ranges
                mut = mixed2nuc(nuc_mut)  
                gene_number = gene_num(mut)
                ref_triple = ""
                qry_triple = ""
                if nuc_codon_pos_dict[pos] == 1
                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                elseif nuc_codon_pos_dict[pos] == 2
                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                elseif nuc_codon_pos_dict[pos] == 3
                    ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                    qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                end
                refAA = AA_triplets[ref_triple]
                qryAA = AA_triplets[qry_triple]
                AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                push!(AA_mut_set, AA_mut)
                all_nuc_to_AA_dict[mut] = AA_mut
                if refAA == qryAA && !(pos in rem9b)
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                else
                    nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
#                    push!(nonsynonymous_nuc_muts, mut)
                end
###################################
                for nuc_mut2 in muts
                    mut2 = mixed2nuc(nuc_mut2)
                    pos2 = nuc_mut_int_comprehensive_dict[mut2]
                    if pos2 in coding_ranges
                        gene_number2 = gene_num(mut2)
                        if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                            if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                            end
                            refAA2 = AA_triplets[ref_triple]
                            qryAA2 = AA_triplets[qry_triple]
                            AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                            push!(AA_mut_set, AA_mut2)
                            delete!(AA_mut_set, AA_mut)
                            all_nuc_to_AA_dict[mut2] = AA_mut2
                            all_nuc_to_AA_dict[mut] = AA_mut2
                            if refAA2 == qryAA2 && !(pos2 in rem9b)
                                synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            end
                        end
                    end
                end
            elseif !isempty(nuc_mut)
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if pos ≥ frst && pos ≤ last
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                push!(AA_mut_set, AA_mut_9b)
                all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
                if refAA_9b ≠ qryAA_9b
                    nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
#                    push!(nonsynonymous_nuc_muts, mut_9b)
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            push!(AA_mut_set, AA_mut2_9b)
                            delete!(AA_mut_set, AA_mut_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
        AA_sort2 = Vector{String}()
        for aa in AA_sort
            if !(refAA_comprehensive_dict[aa] == qryAA_comprehensive_dict)
                push!(AA_sort2, aa)
            end
        end
#        print("\n"^1)
#        ct = 0
#        println("############# AA Mutations #############")
#        for i in 1:length(AA_sort2)
#            mut = AA_sort2[i]
#            gene = string(split(mut, ":")[1])
#            non_gene = string(split(mut, ":")[2])
#            if i == 1
#                print("        ", AA_sort2[i])
#            elseif i > 1
#                lastmut = AA_sort2[i-1]
#                last_gene = string(split(lastmut, ":")[1])
#                last_non_gene = string(split(lastmut, ":")[2])
#                if gene == last_gene
#                    print(", $(non_gene)")
#                else
#                    print("        $(mut)")
#                end
#            end
#        end
#####################################################################################################################################
        all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
        for i in 1:length(synonymous_nuc_sort)
            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
        end
#        return synonymous_nuc_to_AA_dict, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
        return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
    end
    if isempty(muts)
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
function pango_minus_1_fx(pango::String)
    if '.' in pango
        dot_ct = count(".", pango)
        dotsplits = split(pango, ".")
        minus_1 = join(dotsplits(1:dot_ct-1), ".")
        return minus_1
    else
        return pango
    end
end
############################################################################################################################################################################
### Fx: SIMPLER_syn_noncoding_nonsyn_nuc (no context)
function SIMPLER_syn_noncoding_nuc(ref_pango::String, muts::Set{String})
    B_1_1_list = ["B.1.1.53", "B.1.1.273"]
    if ref_pango in B_1_1_list
        ref_pango = "B.1.1"
    end
    if ref_pango == "XBB.1.5.82"
         ref_pango = "XBB.1.5"
    end
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
###################################################################################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut = ""
        for nuc_mut in muts
            if !isempty(muts) && !isempty(ref_seq) && nuc_mut ≠ ""
                pos = nuc_mut_int_comprehensive_dict[nuc_mut]
                if pos in coding_ranges
                    mut = mixed2nuc(nuc_mut)
                    if isempty(mut)
                        println(nuc_mut)
                        println(mut)
                        println("$(pango)")
                    end
                    try
                        nuc_mut_int_comprehensive_dict[mut]
                    catch e
                        println("Problematic mutation string: ", repr(mut))
                        println("Length: ", length(mut))
                        println("Extracted substring: ", repr(mut[2:end-1]))
                        rethrow(e)
                    end
                    gene_number = nuc_gene_num[pos]
                    ref_triple = ""
                    qry_triple = ""
                    if nuc_codon_pos_dict[pos] == 1
                        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    elseif nuc_codon_pos_dict[pos] == 2
                        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    elseif nuc_codon_pos_dict[pos] == 3
                        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                    end
                    refAA = AA_triplets[ref_triple]
                    qryAA = AA_triplets[qry_triple]
                    AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                    if refAA == qryAA && !(pos in rem9b)
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    end
###################################
                    for nuc_mut2 in muts
                        mut2 = mixed2nuc(nuc_mut2)
                        pos2 = nuc_mut_int_comprehensive_dict[mut2]
                        if pos2 in coding_ranges
                            gene_number2 = gene_num(mut2)
                            if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                                if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                                elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                                end
                                refAA2 = AA_triplets[ref_triple]
                                qryAA2 = AA_triplets[qry_triple]
                                AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                                if refAA2 == qryAA2 && !(pos2 in rem9b)
                                    synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                end
                            end
                        end
                    end
                else
                    qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                    for (start_end, place) in noncoding_range_dict
                        frst = start_end[1]
                        last = start_end[2]
                        if pos ≥ frst && pos ≤ last
                            mut_vec = [nuc_mut, place]
                            push!(noncoding_nuc_vector, nuc_mut)
                        end
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b && !isempty(ref_seq) && nuc_mut ≠ ""
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
#        for i in 1:length(synonymous_nuc_sort)
#            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
#        end
        return synonymous_nuc_sort, noncoding_nuc_vector_sort
    else
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
##############################################################################################################################
###################################################################################################################
index_to_date_str = Dict{Int, String}()
date_str_to_index = Dict{String, Int}()
function convert_date_to_date_index(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    date_index = tuple_to_index[date_tuple]
    return date_index
end
print("\n"^1)
println("Done Loading Functions (line #1618 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##############################################################################################################################################
##############################################################################################################################################
##############################################################################################################################################
for tuple in keys(tuple_to_index)
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    date_str_to_index[date_string] = convert_date_to_date_index(date_string)
end
for (index, tuple) in index_to_tuple
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    index_to_date_str[index] = date_string
end
println("Done Making index_to_tuple & tuple_to_index (line #1641 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################
date_to_tuple = Dict{String, Tuple{Int,Int, Int}}()
tuple_to_date = Dict{Tuple{Int,Int, Int}, String}()
function convert_date_to_date_tuple(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    return date_tuple
end
for date in keys(date_str_to_index)
    date_to_tuple[date] = convert_date_to_date_tuple(date)
end
for (date, date_tuple) in date_to_tuple
    tuple_to_date[date_tuple] = date
end
println("Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################################################################
function find_X_pct_date(clade_pango::String, pct::Float64, cp_date_cumul_dict::Dict{String, Dict{Int, Int}}, cp_total_dict::Dict{String, Int})
    cp_total = cp_total_dict[clade_pango]
    pct_date_index = 0
    pct_date_tuple = nothing
    for date_index in 1:2500
        cumulative_ct = cp_date_cumul_dict[clade_pango][date_index]
        if 100*cumulative_ct/cp_total ≥ pct
            pct_date_index = date_index
            pct_date_tuple = index_to_tuple[date_index]
            break
        end
    end
    return pct_date_index, pct_date_tuple
end
##################################################################################################################
### Truly necessary stuff from old megacell | 2026_02_08
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_AAsub_WT, pango)
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_predecessor_meta_dict, pango)
        if haskey(pango_predecessor_meta_dict[pango], 1)
            pango_pre1 = pango_predecessor_meta_dict[pango][1]
            for mut in pango_AAsub_WT[pango_pre1]
                all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
            end
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("Finished!"); print("\n"^1)

6:13.50_PM

Done Loading Functions (line #1618 as of 2022_02_22)!
6:13.50_PM

Done Making index_to_tuple & tuple_to_index (line #1641 as of 2022_02_22)!
6:13.51_PM

Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!
6:13.51_PM

Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!
6:13.51_PM

Done Filling seq_syn_nucs, seq_noncoding_nucs!
6:15.07_PM

Runtime v0 = 76.40723705291748 seconds
Runtime v2 = 0 hr, 1 min, 16.41 sec
Finished!



In [11]:
### Calculates and makes dictionaries for the density of mutations in EPCI and HQCS datasets and the ratios between the two for each gene/protein
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_time_post_load_cell = time()
######################################################################################################################################
gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
######################################################################################################################################
##### Filling all types of NSP muts in all seqs--both circulating & chronic--for DataFrame purposes (to make # of rows the same) #####
######################################################################################################################################
sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                           Gene      AAsite sub_type_ct
sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
for gene in gene_array
    sub_types_at_every_site_combined[gene] = Dict{Int, Set{String}}()
    sub_types_at_every_site_combined_ct[gene] = Dict{Int, Int}()
end
gene_AA_lengths = Dict{String, Int}()
for (gene, AAseq) in gene_AA_dict
    gene_AA_lengths[gene] = length(AAseq)
end
for gene in gene_array
    gene_len = gene_AA_lengths[gene]
    for i in 1:gene_len
        sub_types_at_every_site_combined[gene][i] = Set{String}()
        sub_types_at_every_site_combined_ct[gene][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = aa_gene_comprehensive_dict[sub]
    pos = aa_pos_comprehensive_dict[sub]
    mut_type = refAA_comprehensive_dict[sub]*qryAA_comprehensive_dict[sub]
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for sub in keys(AA_muts_ct_no_dels)
    gene = aa_gene_comprehensive_dict[sub]
    pos = aa_pos_comprehensive_dict[sub]
    mut_type = refAA_comprehensive_dict[sub]*qryAA_comprehensive_dict[sub]
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for gene in gene_array
    for (AApos, mut_type_set) in sub_types_at_every_site_combined[gene]
        sub_type_ct = length(mut_type_set)
        sub_types_at_every_site_combined_ct[gene][AApos] = sub_type_ct
    end
end
###########################################################################################################################################################################
##################### Filling all types of NSP muts in all seqs--both circulating & chronic--for DataFrame purposes (to make # of rows the same) ##########################
###########################################################################################################################################################################
#                                             NSP      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                               NSP      AAsite sub_type_ct
NSP_sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
for NSP in NSP_array
    NSP_sub_types_at_every_site_combined[NSP] = Dict{Int, Set{String}}()
    NSP_sub_types_at_every_site_combined_ct[NSP] = Dict{Int, Int}()
end
for NSP in NSP_array
    NSP_len = NSP_AA_size[NSP]
    for i in 1:NSP_len
        NSP_sub_types_at_every_site_combined[NSP][i] = Set{String}()
        NSP_sub_types_at_every_site_combined_ct[NSP][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            sub_type = NSP_ref_AA_dict[sub]*NSP_qry_AA_dict[sub]
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for sub in keys(AA_muts_ct_no_dels)
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            sub_type = NSP_ref_AA_dict[sub]*NSP_qry_AA_dict[sub]
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for NSP in NSP_array
    for (NSP_pos, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
        sub_type_ct = length(mut_type_set)
        NSP_sub_types_at_every_site_combined_ct[NSP][NSP_pos] = sub_type_ct
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
counted_seq_total_private_AA_subs = Dict{String, Int}()
total_private_AA_subs_all_counted_chronics = 0
total_counted_sequences = 0
for name in all_unique_chr_seqs_set
    total_counted_sequences += 1
    total_AA = length(seq_AA_muts_no_dels[name])
    total_private_AA_subs_all_counted_chronics += total_AA
    counted_seq_total_private_AA_subs[name] = total_AA
end 
global chronic_avg_AA_subs_per_seq = total_private_AA_subs_all_counted_chronics/total_counted_sequences
chronic_avg_AA_subs_per_seq_rd = round(digits=2, chronic_avg_AA_subs_per_seq)
avg_private_AA_per_circ_seq_rd = round(digits=3, avg_private_AA_per_circ_seq)
println()
println("Avg AA Subs per Chronic Sequence = ", chronic_avg_AA_subs_per_seq_rd)
println("Avg AA Subs per Circulating Sequence = ", avg_private_AA_per_circ_seq_rd)
date = Dates.format(today(), "yyyy_mm_dd")
fx_name = "chr_to_circ_mut_density_ratios"
chr_to_circ_gene_density_ratio = Dict{String, Float64}()
chr_to_circ_domain_density_ratio = Dict{String, Float64}()
# The adjusted dict below adjusts for the greater number of AA muts per chronic seq compared to circ seq
adj_factor = avg_private_AA_per_circ_seq/chronic_avg_AA_subs_per_seq
adj_factor_rd = round(digits=4, adj_factor)

simple_chr_to_circ_adj_factor = total_AA_subs_all/total_chr_AA_subs
simple_chr_to_circ_adj_factor_rd = round(digits=2, simple_chr_to_circ_adj_factor)

println("simple_chr_to_circ_adj_factor = $(simple_chr_to_circ_adj_factor_rd)")
println("Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = $(adj_factor_rd)")
############################################################################################################################################################################ 
############################################################################################################################################################################
tot_singlets = length(non_rep_seqs)
tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
tot_chronics = tot_singlets + tot_groups
tot_all = qualifying_seq_ct_all
total_independent_DQ_qualifying_chronic_seqs = length(all_unique_chr_seqs)
circ_to_chronic_seq_total_ratio = qualifying_seq_ct_all/tot_chronics
combined_adjustment_factor = circ_to_chronic_seq_total_ratio*adj_factor
combined_adjustment_factor_check = (avg_private_AA_per_circ_seq*qualifying_seq_ct_all)/(chronic_avg_AA_subs_per_seq*tot_chronics)
combined_adjustment_factor_rd = round(digits=3, combined_adjustment_factor)
combined_adjustment_factor_check_rd = round(digits=3, combined_adjustment_factor_check)
println()
print("Total Singlets = $(tot_singlets) | ")
print("Total Groups = $(tot_groups) | ")
println("Total Independent Chronics = $(tot_chronics)")
println("Total Qualifying Circulating Sequences = $(qualifying_seq_ct_all)")
circ_to_chronic_seq_total_ratio_rd = round(digits=2, circ_to_chronic_seq_total_ratio)
println("(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = $(circ_to_chronic_seq_total_ratio_rd)")
print("Combined Adjustment Factor = $(combined_adjustment_factor_rd)  |  ")
println("Combined Adjustment Factor Check = $(combined_adjustment_factor_check_rd)")
#####################################################################################################################################
########### Note: the 1/1000 factor below is to make up for the 1000 factor that's used in the current ALL_SEQS function ############
#####################################################################################################################################
for (gene, chr_density) in gene_mut_density
    all_density = gene_mut_density_all[gene]
    chr_circ_ratio = (chr_density/all_density)*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_gene_density_ratio[gene] = ratio_rd
end
for (domain, chr_density) in domain_mut_density
    all_density = domain_mut_density_all[domain]
    chr_circ_ratio = chr_density/all_density*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_domain_density_ratio[domain] = ratio_rd
end
######################################################################################################################################
chr_to_circ_gene_density_ratio_sort_by_gene = sort(collect(chr_to_circ_gene_density_ratio), by = x -> gene_protein_order[x[1]])
chr_to_circ_gene_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_gene_density_ratio), by = x -> x[2], rev=true)
chr_to_circ_domain_density_ratio_sort_by_domain = sort(collect(chr_to_circ_domain_density_ratio), by = x -> domain_order[x[1]])
chr_to_circ_domain_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_domain_density_ratio), by = x -> x[2], rev=true)
######################################################################################################################################
save("$(date)_chr_to_circ_gene_density_ratio.jld2", "chr_to_circ_gene_density_ratio", chr_to_circ_gene_density_ratio)
save("$(date)_chr_to_circ_domain_density_ratio.jld2", "chr_to_circ_domain_density_ratio", chr_to_circ_domain_density_ratio)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_gene.jld2", chr_to_circ_gene_density_ratio_sort_by_gene)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_ratio.jld2", chr_to_circ_gene_density_ratio_sort_by_ratio)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_domain.jld2", chr_to_circ_domain_density_ratio_sort_by_domain)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_ratio.jld2", chr_to_circ_domain_density_ratio_sort_by_ratio)
######################################################################################################################################
domain_residues_NSP = Dict{String, String}("NSP3_Ubl1"=>"NSP3:1-111", "NSP3_HVR"=>"NSP3:112-206", "NSP3_Mac1"=>"NSP3:207-374", "NSP3_Mac2"=>"NSP3:375-551", "NSP3_Mac3"=>"NSP3:552-673", "NSP3_DPUP"=>"NSP3:674-743", "NSP3_Ubl2"=>"NSP3:744-803", "NSP3_PLpro"=>"NSP3:804-1052", "NSP3_NAB"=>"NSP3:1053-1203", "NSP3_BSM"=>"NSP3:1204-1334", "NSP3_TM1"=>"NSP3:1335-1440", "NSP3_Ecto3"=>"NSP3:1441-1499", "NSP3_TM234HLX"=>"NSP3:1500-1586", "NSP3_Y1"=>"NSP3:1587-1764", "NSP3_CoVY"=>"NSP3:1765-1945", "NSP4_TM1"=>"NSP4:1-31", "NSP4_Ecto4"=>"NSP4:32-271", "NSP4_TM2_TM6"=>"NSP4:272-410", "NSP4_CTD"=>"NSP4:411-500", "NSP6_AmphHlx"=>"NSP6:219-236", "NSP6_MAE"=>"NSP6:237-276", "NSP6_cyto_CTD"=>"NSP6:277-290", "NSP12_NiRAN"=>"NSP12:1-250", "NSP12_intrfce"=>"NSP12:251-398", "NSP12_fingers"=>"NSP12:399-581", "NSP12_palm"=>"NSP12:582-627", "NSP12_palmLnk"=>"NSP12:628-687", "NSP12_palm2"=>"NSP12:688-812", "NSP12_thumb"=>"NSP12:813-932", "NSP13_ZBD"=>"NSP13:1-101", "NSP13_stalk"=>"NSP13:102-150", "NSP13_1B"=>"NSP13:151-226", "NSP13_RecA1"=>"NSP13:261-442", "NSP13_RecA2"=>"NSP13:443-601", "NSP14_nsp10"=>"NSP14:1-58", "NSP14_EXON"=>"NSP14:67-282 ", "NSP14_hinge1"=>"NSP14:286-300", "NSP14_hinge2"=>"NSP14:411-434", "NSP14_N7MTase"=>"NSP14:302-527", "NSP15_NTD"=>"NSP15:1-64", "NSP15_MD"=>"NSP15:65-182", "NSP15_endoU"=>"NSP15:207-347", "S_S1"=>"S:2-686", "S_S2"=>"S:687-1273", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_N3"=>"N:176-246", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419", "N_SR"=>"N:176-206", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_9b_overlap"=>"N:4-101")
domain_residues_ORF = Dict{String, String}("ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "NSP3_Ubl1"=>"ORF1a:819-929", "NSP3_HVR"=>"ORF1a:930-1024", "NSP3_Mac1"=>"ORF1a:1025-1192", "NSP3_Mac2"=>"ORF1a:1193-1369", "NSP3_Mac3"=>"ORF1a:1370-1491", "NSP3_DPUP"=>"ORF1a:1492-1561", "NSP3_Ubl2"=>"ORF1a:1562-1621", "NSP3_PLpro"=>"ORF1a:1622-1870", "NSP3_NAB"=>"ORF1a:1871-2021", "NSP3_BSM"=>"ORF1a:2022-2152", "NSP3_TM1"=>"ORF1a:2153-2258", "NSP3_Ecto3"=>"ORF1a:2259-2317", "NSP3_TM234HLX"=>"ORF1a:2318-2404", "NSP3_Y1"=>"ORF1a:2405-2582", "NSP3_CoVY"=>"ORF1a:2583-2763", "NSP4_TM1"=>"ORF1a:2764-2794", "NSP4_Ecto4"=>"ORF1a:2795-3034", "NSP4_TM2_TM6"=>"ORF1a:3035-3173", "NSP4_CTD"=>"ORF1a:3174-3263", "NSP6_AmphHlx"=>"ORF1a:3788-3805", "NSP6_MAE"=>"ORF1a:3806-3845", "NSP6_cyto_CTD"=>"ORF1a:3846-3859", "NSP12_NiRAN"=>"1a:4393-1b:241", "NSP12_intrfce"=>"ORF1b:242-389", "NSP12_fingers"=>"ORF1b:390-572", "NSP12_palm"=>"ORF1b:573-618", "NSP12_palmLnk"=>"ORF1b:619-678", "NSP12_thumb"=>"ORF1b:804-923", "NSP13_ZBD"=>"ORF1b:924-1024", "NSP13_stalk"=>"ORF1b:1025-1073", "NSP13_1B"=>"ORF1b:1074-1149", "NSP13_RecA1"=>"ORF1b:1184-1365", "NSP13_RecA2"=>"ORF1b:1366-1524", "NSP14_nsp10"=>"ORF1b:1525-1582", "NSP14_EXON"=>"ORF1b:1591-1806", "NSP14_hinge1"=>"ORF1b:1810-1824", "NSP14_hinge2"=>"ORF1b:1935-1958", "NSP14_N7MTase"=>"ORF1b:1826-2051", "NSP15_NTD"=>"ORF1b:2052-2115", "NSP15_MD"=>"ORF1b:2116-2233", "NSP15_endoU"=>"ORF1b:2258-2398", "S_S1"=>"S:2-686", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_S2"=>"S:687-1273", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_9b_overlap"=>"N:4-101", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_SR"=>"N:176-206", "N_N3"=>"N:176-246", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419" )
######################################################################################################################################
function NSP_to_ORF(domain::String, NSP::Int)
    ORFadd = NSP1ab_add[NSP]
    first1 = minimum(domain_residues_NSP_BitSet[domain])
    last1 = maximum(domain_residues_NSP_BitSet[domain])
    first = first1 + ORFadd
    last = last1 + ORFadd
    ORF_range_BitSet = BitSet(first:last)
    firstORF = string(first)
    lastORF = string(last)
    ORF_range = ""
    if NSP < 12
        ORF_range = "ORF1a:$(firstORF)-$(lastORF)"
    else
        ORF_range = "ORF1b:$(firstORF)-$(lastORF)"
    end
    return ORF_range, ORF_range_BitSet
end
#####################################################################################################################################
sorted_ratio_gene_dicts = [chr_to_circ_gene_density_ratio_sort_by_gene, chr_to_circ_gene_density_ratio_sort_by_ratio] 
sorted_ratio_domain_dicts = [chr_to_circ_domain_density_ratio_sort_by_domain, chr_to_circ_domain_density_ratio_sort_by_ratio]
headers_gene = ["###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Gene ######", "###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Ratio ######"]
headers_domain = ["######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Domain #########", "######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Ratio #########"]
open("$(date)_$(fx_name).txt", "w") do g
    ct = 0
    for sorted_dict in sorted_ratio_gene_dicts
        ct += 1
        println()
        println(headers_gene[ct])
        println(g)
        println(g, headers_gene[ct])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
            println(g, "                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
        end
    end
    ct2 = 0
    for sorted_dict in sorted_ratio_domain_dicts
        ct2 += 1
        println()
        println(headers_domain[ct2])
        println(g)
        println(g, headers_domain[ct2])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
            println(g, "               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
        end
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
gene_sub_ranks_chr(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels)
gene_sub_ranks_all(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
gene_sub_pos_ranks_all(gene_array, gene_AA_lengths,  AA_muts_ct_pos_only_no_dels_all)
gene_sub_pos_ranks_chr(gene_array, gene_AA_lengths, AA_muts_ct_pos_only_no_dels )
NSP_sub_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels)
NSP_sub_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels); print("\n"^1)
######################################################################################################################################
print(sub_types_at_every_site_combined["ORF9b"][2])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][2]);    print("\n"^1)
print(sub_types_at_every_site_combined["ORF9b"][92])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][92]);   print("\n"^1)
############################################################################################################################################################################ 
############################################################################################################################################################################
gene_AA_sites = Dict{String, BitSet}()
for (gene, length) in gene_AA_lengths
    gene_AA_sites[gene] = BitSet([1:length...])
end
#for (gene, bitset) in gene_AA_sites; println(gene); println(bitset); end
############################################################
gene_AA_sites_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_used[gene] = BitSet()
end
for (sub, ct) in AA_muts_ct_no_dels_all
    gene = aa_gene_comprehensive_dict[sub]
    AA_site = aa_pos_comprehensive_dict[sub]
    push!(gene_AA_sites_used[gene], AA_site)
end
############################################################
gene_AA_sites_not_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_not_used[gene] = BitSet()
end
for (gene, bitset) in gene_AA_sites
    bitset_used = gene_AA_sites_used[gene]
    unused_AA_sites = setdiff(bitset, bitset_used)
    gene_AA_sites_not_used[gene] = unused_AA_sites
end
###########################################################
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    print("$(gene) = "); print(" | ")
    for unused in unused_AA_sites
        print("$(unused) |")
    end
end
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    for AA_site in unused_AA_sites
        placeholder_sub = "$(gene):W$(AA_site)W"
        AA_muts_ct_no_dels_all[placeholder_sub] = 0
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
#NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
NSP1_muts_ct_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_no_dels_all = Dict{String, Int}()
NSP_muts_ct_no_dels_all_array = [NSP1_muts_ct_no_dels_all, NSP2_muts_ct_no_dels_all, NSP3_muts_ct_no_dels_all, NSP4_muts_ct_no_dels_all, NSP5_muts_ct_no_dels_all, NSP6_muts_ct_no_dels_all, NSP7_muts_ct_no_dels_all, NSP8_muts_ct_no_dels_all, NSP9_muts_ct_no_dels_all, NSP10_muts_ct_no_dels_all, NSP11_muts_ct_no_dels_all, NSP12_muts_ct_no_dels_all, NSP13_muts_ct_no_dels_all, NSP14_muts_ct_no_dels_all, NSP15_muts_ct_no_dels_all, NSP16_muts_ct_no_dels_all]       
NSP_muts_ct_no_dels_all = Dict{String, Int}()
for (sub, count) in AA_muts_ct_no_dels_all
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP_muts_ct_no_dels_all[NSP_sub] = count
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_num = parse(Int, NSP[4:end])
            NSP_dict = NSP_muts_ct_no_dels_all_array[NSP_num]
            NSP_dict[NSP_sub] = count
        end
    end
end
###################################################################################################################################
NSP1_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP_muts_ct_pos_only_no_dels_all_array = [NSP1_muts_ct_pos_only_no_dels_all, NSP2_muts_ct_pos_only_no_dels_all, NSP3_muts_ct_pos_only_no_dels_all, NSP4_muts_ct_pos_only_no_dels_all, NSP5_muts_ct_pos_only_no_dels_all, NSP6_muts_ct_pos_only_no_dels_all, NSP7_muts_ct_pos_only_no_dels_all, NSP8_muts_ct_pos_only_no_dels_all, NSP9_muts_ct_pos_only_no_dels_all, NSP10_muts_ct_pos_only_no_dels_all, NSP11_muts_ct_pos_only_no_dels_all, NSP12_muts_ct_pos_only_no_dels_all, NSP13_muts_ct_pos_only_no_dels_all, NSP14_muts_ct_pos_only_no_dels_all, NSP15_muts_ct_pos_only_no_dels_all, NSP16_muts_ct_pos_only_no_dels_all]
NSP_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP_all_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    NSP = NSP_array[i]
    NSP_len = NSP_AA_size[NSP]
    for j in 1:NSP_len
        NSP_pos = NSP*":"*"$(j)"
        NSP_all_dict[NSP_pos] = 0
        NSP_muts_ct_pos_only_no_dels_all[NSP_pos] = 0
    end
end
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP = NSP_array[i]
    NSP_dict = NSP_muts_ct_no_dels_all_array[i]
    NSP_pos_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    for (sub, ct) in NSP_dict
        pos = NSP_muts_pos_dict[sub]
        sub_pos = NSP*":"*"$(pos)"
        NSP_pos_dict[sub_pos] += ct
        NSP_muts_ct_pos_only_no_dels_all[sub_pos] += ct
    end
end
total_time_post_load_cell = time() - start_time_post_load_cell
total_time_post_load_cell_rd = round(digits=1, total_time_post_load_cell)
println()
println("Total Time for Post-Load Cell = $(total_time_post_load_cell_rd) Seconds")
# Total Time for Post-Load Cell = 139.3 Seconds (2025-08-18)

2025-08-31__723PM

Avg AA Subs per Chronic Sequence = 17.67
Avg AA Subs per Circulating Sequence = 2.198
Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = 0.1244

Total Singlets = 2219 | Total Groups = 915 | Total Independent Chronics = 3134
Total Qualifying Circulating Sequences = 11857354
(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = 3783.46
Combined Adjustment Factor = 470.753  |  Combined Adjustment Factor Check = 470.753


LoadError: KeyError: key "ORF3a_TM12Link" not found

In [368]:
### Getting all RBM & RBD muts + ORF9b/N Doubles + artifactual_private_muts_subs + subs vs pos_only determination | Runtime = 
start = time(); print("\n"^1)
date = Dates.format(today(), "yyyy_mm_dd")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); date_hour = Dates.format(now(), "yyyy_mm_dd_Ip"); print("\n"^1)
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
                                                        sub_0__posonly_1 = 1
                                                        normal_0__spikeonly_1 = 0
                                                        noBAL_0__withBAL_1 = 1
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
########################################################################################################################################################################### 
############################################################# Begin EXCLUDED MUTS SECTION #################################################################################
########################################################################################################################################################################### 
##### The mutations added below are ones that have been found to produce artifactual correlations. For example, 18/19 EPCI sequences 
####      with ORF1a:M2606I are B.1.2 (the other is likely a B.1.2 miscategorized as a B.1), and seven of these have N:D377Y, indicating
####      that these are both inherited (not private) mutations, which can be confirmed by viewing the Usher Tree for these sequences. 
####      Nextclade miscategorizes both of these as private mutations. Similarly, 5/6 sequences with ORF1a:E3764D are B.1.258, and all five also 
####      have ORF3a:A72S. Both are inherited and mischaracterized as private by Nextclade. 
global artifactual_private_muts_subs = Set{String}()
#push!(artifactual_private_muts_subs, "ORF1a:T2501I") ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_subs, "ORF1a:M2606I") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "N:D377Y")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "ORF3a:A72S")   ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:E3764D") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:I2283T") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF1a:A599T")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "S:F59S")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF8:L60F")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "ORF1a:H2125Y") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "S:Q146K")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
push!(artifactual_private_muts_subs, "N:M203K")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:G204R")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:R204G")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_subs, "ORF1b:M1156I") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_subs, "ORF6:L61D")    ## Artifactual 3-nuc reversion, often in BA.1/2 or BA.2/5 recombinants
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## Omicron mut misattributed as private in 3 recombinants
push!(artifactual_private_muts_subs, "M:A30T")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_subs, "ORF1a:S1612L") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:E39*")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:F18L")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_subs, "ORF1a:A2554V") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1b:H1087Y") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:M2796T") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:P1803S") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:P104S")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "N:A208S")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF8:K68*")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1975S") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:V2178F") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:I97V")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A2589T") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF9b:T83I")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T842I")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF3a:V48F")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:G49C")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A1204T") ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:A376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:T376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:F375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:S375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:D339G")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:N417K")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:K440N")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "ORF1b:T1555I") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A4285V") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:N2361K") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:K1191N")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:R158G")      ## Artifactual Delta reversion
push!(artifactual_private_muts_subs, "ORF1a:I1023-") ## Dumb mistake I made that I have to undo with this.
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF7b:I2V")    ## Corresponds to ORF7a:*122W
push!(artifactual_private_muts_subs, "ORF1b:P218L")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G2118D") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF1a:A1298V") ## Inherited BA.1 mut, miscategorized by Nextclade as private


push!(artifactual_private_muts_subs, "ORF3a:V112F")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2695I") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:I35K")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E1871G") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1903Y") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:K2219R") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:S771I")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "N:S26C")       ## Inherited BA.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V108I")   ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:G128S")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2452S") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:S1182T") ## Inherited FL.4 mutation in Australian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1570T") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E2355Q") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:T19R")       ## Delta mutation present in recombinants, falsely labeled as private mutation
push!(artifactual_private_muts_subs, "ORF7a:A66V")   ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T3459M") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:M1T")    ## Corresponds to ORF7a:*122R
push!(artifactual_private_muts_subs, "ORF1b:A869V")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E1015G") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:L4F")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V365I")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1869Y") ## Inherited C.37 and AY.85 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:L65H")   ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
   
push!(artifactual_private_muts_subs, "ORF3a:S220I")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A176T")  ## Inherited EG.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E352D")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A434S")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G3676C") ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:Q991L")  ## Inherited BA.5.2 (North America) mutation miscategorized by Nextclade as private
######################################################################################################################################
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
#### This list was created in one of the above cells (the one with the ORF9b_N_nucmuts_to_AAres function) & then copied & pasted here.
global Double_N_ORF9b_muts = Set(["N:G63D", "N:F33S, N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])            
# global  Double_N_ORF9b_muts = Set(["ORF9b:M1L", "ORF9b:M1L", "ORF9b:M1V", "ORF9b:M1K", "ORF9b:M1R", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:D2Y", "ORF9b:D2H", "ORF9b:D2N", "ORF9b:D2E", "ORF9b:D2E", "ORF9b:P3S", "ORF9b:P3T", "ORF9b:P3A", "ORF9b:K4*", "ORF9b:K4Q", "ORF9b:K4E", "ORF9b:K4I", "ORF9b:K4T", "ORF9b:K4N", "ORF9b:K4N", "ORF9b:I5F", "ORF9b:I5L", "ORF9b:I5V", "ORF9b:I5N", "ORF9b:I5S", "ORF9b:I5M", "ORF9b:S6C", "ORF9b:S6R", "ORF9b:S6G", "ORF9b:S6I", "ORF9b:S6T", "ORF9b:S6R", "ORF9b:E7*", "ORF9b:E7Q", "ORF9b:E7K", "ORF9b:E7D", "ORF9b:E7D", "ORF9b:M8L", "ORF9b:M8L", "ORF9b:M8V", "ORF9b:M8K", "ORF9b:M8R", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:H9Y", "ORF9b:H9N", "ORF9b:H9D", "ORF9b:H9Q", "ORF9b:P10S", "ORF9b:P10T", "ORF9b:P10A", "ORF9b:A11S", "ORF9b:A11P", "ORF9b:A11T", "ORF9b:L12I", "ORF9b:L12V", "ORF9b:L12*", "ORF9b:L12F", "ORF9b:L12F", "ORF9b:R13C", "ORF9b:R13S", "ORF9b:R13G", "ORF9b:L14M", "ORF9b:L14V", "ORF9b:L14*", "ORF9b:L14W", "ORF9b:L14F", "ORF9b:L14F", "ORF9b:V15L", "ORF9b:V15L", "ORF9b:V15M", "ORF9b:D16Y", "ORF9b:D16H", "ORF9b:D16N", "ORF9b:D16E", "ORF9b:D16E", "ORF9b:P17S", "ORF9b:P17T", "ORF9b:P17A", "ORF9b:Q18*", "ORF9b:Q18K", "ORF9b:Q18E", "ORF9b:Q18H", "ORF9b:Q18H", "ORF9b:I19F", "ORF9b:I19L", "ORF9b:I19V", "ORF9b:I19N", "ORF9b:I19S", "ORF9b:I19M", "ORF9b:Q20*", "ORF9b:Q20K", "ORF9b:Q20E", "ORF9b:Q20H", "ORF9b:Q20H", "ORF9b:L21M", "ORF9b:L21V", "ORF9b:A22S", "ORF9b:A22P", "ORF9b:A22T", "ORF9b:V23L", "ORF9b:V23L", "ORF9b:V23I", "ORF9b:V23E", "ORF9b:V23G", "ORF9b:T24S", "ORF9b:T24P", "ORF9b:T24A", "ORF9b:T24N", "ORF9b:T24S", "ORF9b:R25*", "ORF9b:R25G", "ORF9b:R25I", "ORF9b:R25T", "ORF9b:R25S", "ORF9b:R25S", "ORF9b:M26L", "ORF9b:M26L", "ORF9b:M26V", "ORF9b:M26K", "ORF9b:M26R", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:E27*", "ORF9b:E27Q", "ORF9b:E27K", "ORF9b:E27D", "ORF9b:E27D", "ORF9b:N28Y", "ORF9b:N28H", "ORF9b:N28D", "ORF9b:N28I", "ORF9b:N28T", "ORF9b:N28K", "ORF9b:N28K", "ORF9b:A29S", "ORF9b:A29P", "ORF9b:A29T", "ORF9b:V30L", "ORF9b:V30L", "ORF9b:V30M", "ORF9b:V30E", "ORF9b:V30G", "ORF9b:G31W", "ORF9b:G31R", "ORF9b:G31R", "ORF9b:R32C", "ORF9b:R32S", "ORF9b:R32G", "ORF9b:D33Y", "ORF9b:D33H", "ORF9b:D33N", "ORF9b:D33E", "ORF9b:D33E", "ORF9b:Q34*", "ORF9b:Q34K", "ORF9b:Q34E", "ORF9b:Q34H", "ORF9b:Q34H", "ORF9b:N35Y", "ORF9b:N35H", "ORF9b:N35D", "ORF9b:N35I", "ORF9b:N35T", "ORF9b:N35K", "ORF9b:N35K", "ORF9b:N36Y", "ORF9b:N36H", "ORF9b:N36D", "ORF9b:N36I", "ORF9b:N36T", "ORF9b:N36K", "ORF9b:N36K", "ORF9b:V37F", "ORF9b:V37L", "ORF9b:V37I", "ORF9b:G38C", "ORF9b:G38R", "ORF9b:G38S", "ORF9b:P39S", "ORF9b:P39T", "ORF9b:P39A", "ORF9b:K40*", "ORF9b:K40Q", "ORF9b:K40E", "ORF9b:K40M", "ORF9b:K40T", "ORF9b:K40N", "ORF9b:K40N", "ORF9b:V41F", "ORF9b:V41L", "ORF9b:V41I", "ORF9b:Y42H", "ORF9b:Y42N", "ORF9b:Y42D", "ORF9b:Y42F", "ORF9b:Y42S", "ORF9b:Y42*", "ORF9b:Y42*", "ORF9b:P43S", "ORF9b:P43T", "ORF9b:P43A", "ORF9b:I44L", "ORF9b:I44L", "ORF9b:I44V", "ORF9b:I44K", "ORF9b:I44R", "ORF9b:I44M", "ORF9b:I45L", "ORF9b:I45L", "ORF9b:I45V", "ORF9b:I45K", "ORF9b:I45R", "ORF9b:I45M", "ORF9b:L46M", "ORF9b:L46V", "ORF9b:R47C", "ORF9b:R47S", "ORF9b:R47G", "ORF9b:L48F", "ORF9b:L48I", "ORF9b:L48V", "ORF9b:G49C", "ORF9b:G49R", "ORF9b:G49S", "ORF9b:G49V", "ORF9b:G49A", "ORF9b:G49D", "ORF9b:S50P", "ORF9b:S50T", "ORF9b:S50A", "ORF9b:S50*", "ORF9b:S50*", "ORF9b:P51S", "ORF9b:P51T", "ORF9b:P51A", "ORF9b:L52F", "ORF9b:L52I", "ORF9b:L52V", "ORF9b:S53P", "ORF9b:S53T", "ORF9b:S53A", "ORF9b:L54F", "ORF9b:L54I", "ORF9b:L54V", "ORF9b:N55Y", "ORF9b:N55H", "ORF9b:N55D", "ORF9b:N55I", "ORF9b:N55T", "ORF9b:N55K", "ORF9b:N55K", "ORF9b:M56L", "ORF9b:M56L", "ORF9b:M56V", "ORF9b:M56K", "ORF9b:M56R", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:A57S", "ORF9b:A57P", "ORF9b:A57T", "ORF9b:R58W", "ORF9b:R58G", "ORF9b:R58M", "ORF9b:R58T", "ORF9b:R58S", "ORF9b:R58S", "ORF9b:K59*", "ORF9b:K59Q", "ORF9b:K59E", "ORF9b:K59M", "ORF9b:K59T", "ORF9b:K59N", "ORF9b:K59N", "ORF9b:T60S", "ORF9b:T60P", "ORF9b:T60A", "ORF9b:T60N", "ORF9b:T60S", "ORF9b:L61I", "ORF9b:L61V", "ORF9b:L61F", "ORF9b:L61F", "ORF9b:N62Y", "ORF9b:N62H", "ORF9b:N62D", "ORF9b:N62I", "ORF9b:N62T", "ORF9b:N62K", "ORF9b:N62K", "ORF9b:S63P", "ORF9b:S63T", "ORF9b:S63A", "ORF9b:S63Y", "ORF9b:S63C", "ORF9b:L64F", "ORF9b:L64I", "ORF9b:L64V", "ORF9b:E65*", "ORF9b:E65Q", "ORF9b:E65K", "ORF9b:E65D", "ORF9b:E65D", "ORF9b:D66Y", "ORF9b:D66H", "ORF9b:D66N", "ORF9b:D66E", "ORF9b:D66E", "ORF9b:K67*", "ORF9b:K67Q", "ORF9b:K67E", "ORF9b:K67M", "ORF9b:K67T", "ORF9b:K67N", "ORF9b:K67N", "ORF9b:A68S", "ORF9b:A68P", "ORF9b:A68T", "ORF9b:F69L", "ORF9b:F69I", "ORF9b:F69V", "ORF9b:F69L", "ORF9b:F69L", "ORF9b:Q70*", "ORF9b:Q70K", "ORF9b:Q70E", "ORF9b:Q70H", "ORF9b:Q70H", "ORF9b:L71I", "ORF9b:L71V", "ORF9b:L71*", "ORF9b:L71F", "ORF9b:L71F", "ORF9b:T72S", "ORF9b:T72P", "ORF9b:T72A", "ORF9b:T72K", "ORF9b:T72R", "ORF9b:P73S", "ORF9b:P73T", "ORF9b:P73A", "ORF9b:I74L", "ORF9b:I74L", "ORF9b:I74V", "ORF9b:I74K", "ORF9b:I74R", "ORF9b:I74M", "ORF9b:A75S", "ORF9b:A75P", "ORF9b:A75T", "ORF9b:A75E", "ORF9b:A75G", "ORF9b:V76F", "ORF9b:V76L", "ORF9b:V76I", "ORF9b:V76D", "ORF9b:V76G", "ORF9b:Q77*", "ORF9b:Q77K", "ORF9b:Q77E", "ORF9b:Q77H", "ORF9b:Q77H", "ORF9b:M78L", "ORF9b:M78L", "ORF9b:M78V", "ORF9b:M78K", "ORF9b:M78R", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:T79S", "ORF9b:T79P", "ORF9b:T79A", "ORF9b:T79N", "ORF9b:T79S", "ORF9b:K80*", "ORF9b:K80Q", "ORF9b:K80E", "ORF9b:K80I", "ORF9b:K80T", "ORF9b:K80N", "ORF9b:K80N", "ORF9b:L81M", "ORF9b:L81V", "ORF9b:L81W", "ORF9b:L81F", "ORF9b:L81F", "ORF9b:A82S", "ORF9b:A82P", "ORF9b:A82T", "ORF9b:T83S", "ORF9b:T83P", "ORF9b:T83A", "ORF9b:T83N", "ORF9b:T83S", "ORF9b:T84S", "ORF9b:T84P", "ORF9b:T84A", "ORF9b:T84N", "ORF9b:T84S", "ORF9b:E85*", "ORF9b:E85Q", "ORF9b:E85K", "ORF9b:E85D", "ORF9b:E86*", "ORF9b:E86Q", "ORF9b:E86K", "ORF9b:E86V", "ORF9b:E86A", "ORF9b:E86D", "ORF9b:E86D", "ORF9b:L87I", "ORF9b:L87V", "ORF9b:P88S", "ORF9b:P88T", "ORF9b:P88A", "ORF9b:D89Y", "ORF9b:D89H", "ORF9b:D89N", "ORF9b:D89V", "ORF9b:D89A", "ORF9b:D89E", "ORF9b:E90*", "ORF9b:E90Q", "ORF9b:E90K", "ORF9b:E90D", "ORF9b:E90D", "ORF9b:F91L", "ORF9b:F91I", "ORF9b:F91V", "ORF9b:F91C", "ORF9b:F91L", "ORF9b:F91L", "ORF9b:V92L", "ORF9b:V92L", "ORF9b:V92M", "ORF9b:V93L", "ORF9b:V93L", "ORF9b:V93M", "ORF9b:V94L", "ORF9b:V94L", "ORF9b:V94M", "ORF9b:T95S", "ORF9b:T95P", "ORF9b:T95A", "ORF9b:T95K", "ORF9b:T95R", "ORF9b:V96L", "ORF9b:V96L", "ORF9b:V96I", "ORF9b:K97*", "ORF9b:K97Q", "ORF9b:K97E", "ORF9b:K97I", "ORF9b:K97T", "ORF9b:K97N", "ORF9b:K97N", "ORF9b:*98R", "ORF9b:*98R", "ORF9b:*98G", "ORF9b:*98L", "ORF9b:*98S", "ORF9b:*98C", "ORF9b:*98C", "ORF9b:*98W"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_OG = list_to_strings_set("ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*") 
artifactual_private_muts_pos_only = Set{String}()
#push!(artifactual_private_muts_pos_only, "ORF1a:2501) ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_pos_only, "ORF1a:2606") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "N:377")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "ORF3a:72")   ## Inherited B.1.25 8 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:3764") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:2283") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF1a:599")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "S:59")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF8:L60")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "ORF1a:2125") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "S:146")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
#push!(artifactual_private_muts_pos_only, "N:203")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_pos_only, "ORF1b:1156") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_pos_only, "ORF:61")     ## Artifactual 3-nuc reversion
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## Omicron mut misattributed as private in 3 recombinants
#push!(artifactual_private_muts_pos_only, "M:30")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_pos_only, "ORF1a:1612") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:39")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:18")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_pos_only, "ORF1a:2554") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1b:1087") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:2796") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:1803") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:104")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "N:208")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF8:68")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1975") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2178") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:97")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2589") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF9b:83")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:842")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF3a:48")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:49")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1204") ## Inherited BE.1 mutation miscategorized by Nextclade as private
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:339")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:417")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:440")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_pos_only, "ORF1b:1555") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:4285") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2361") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:1191")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:44")   ## Deletions here that maintain the stop codon get misinterpreted as substitutions
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF7b:1")    ## Necessarily overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF7b:2")    ## Overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF1b:218")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2118") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_pos_only, "ORF1a:1298") ## Inherited BA.1 mut, miscategorized by Nextclade as private

push!(artifactual_private_muts_pos_only, "ORF3a:112")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2695") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:35")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1903") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:771")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private 

push!(artifactual_private_muts_pos_only, "ORF1a:108")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:365")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:397")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:925")  ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1015") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3459") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:128")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:869")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1871") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2452") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:4")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
global Double_N_ORF9b_muts_pos_only = Set(["N:4", "N:5", "N:6", "N:7", "N:8", "N:9", "N:10", "N:11", "N:12", "N:13", "N:14", "N:15", "N:16", "N:17", "N:18", "N:19", "N:20", "N:21", "N:22", "N:23", "N:24", "N:25", "N:26", "N:27", "N:28", "N:29", "N:30", "N:31", "N:32", "N:33", "N:34", "N:35", "N:36", "N:37", "N:38", "N:39", "N:40", "N:41", "N:42", "N:43", "N:44", "N:45", "N:46", "N:47", "N:48", "N:49", "N:50", "N:51", "N:52", "N:53", "N:54", "N:55", "N:56", "N:57", "N:58", "N:59", "N:60", "N:61", "N:62", "N:63", "N:64", "N:65", "N:66", "N:67", "N:68", "N:69", "N:70", "N:71", "N:72", "N:73", "N:74", "N:75", "N:76", "N:77", "N:78", "N:79", "N:80", "N:81", "N:82", "N:83", "N:84", "N:85", "N:86", "N:87", "N:88", "N:89", "N:90", "N:91", "N:92", "N:93", "N:94", "N:95", "N:96", "N:97", "N:98", "N:99", "N:100", "N:101"])
# global  Double_N_ORF9b_muts = Set(["ORF9b:1", "ORF9b:2", "ORF9b:3", "ORF9b:4", "ORF9b:5", "ORF9b:6", "ORF9b:7", "ORF9b:8", "ORF9b:9", "ORF9b:10", "ORF9b:11", "ORF9b:12", "ORF9b:13", "ORF9b:14", "ORF9b:15", "ORF9b:16", "ORF9b:17", "ORF9b:18", "ORF9b:19", "ORF9b:20", "ORF9b:21", "ORF9b:22", "ORF9b:23", "ORF9b:24", "ORF9b:25", "ORF9b:26", "ORF9b:27", "ORF9b:28", "ORF9b:29", "ORF9b:30", "ORF9b:31", "ORF9b:32", "ORF9b:33", "ORF9b:34", "ORF9b:35", "ORF9b:36", "ORF9b:37", "ORF9b:38", "ORF9b:39", "ORF9b:40", "ORF9b:41", "ORF9b:42", "ORF9b:43", "ORF9b:44", "ORF9b:45", "ORF9b:46", "ORF9b:47", "ORF9b:48", "ORF9b:49", "ORF9b:50", "ORF9b:51", "ORF9b:52", "ORF9b:53", "ORF9b:54", "ORF9b:55", "ORF9b:56", "ORF9b:57", "ORF9b:58", "ORF9b:59", "ORF9b:60", "ORF9b:61", "ORF9b:62", "ORF9b:63", "ORF9b:64", "ORF9b:65", "ORF9b:66", "ORF9b:67", "ORF9b:68", "ORF9b:69", "ORF9b:70", "ORF9b:71", "ORF9b:72", "ORF9b:73", "ORF9b:74", "ORF9b:75", "ORF9b:76", "ORF9b:77", "ORF9b:78", "ORF9b:79", "ORF9b:80", "ORF9b:81", "ORF9b:82", "ORF9b:83", "ORF9b:84", "ORF9b:85", "ORF9b:86", "ORF9b:87", "ORF9b:88", "ORF9b:89", "ORF9b:90", "ORF9b:91", "ORF9b:92", "ORF9b:93", "ORF9b:94", "ORF9b:95", "ORF9b:96", "ORF9b:97", "ORF9b:98"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_pos_only = list_to_strings_set("ORF1a:2183, ORF1a:2972, ORF1a:2972, ORF1a:3049, ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, ORF1a:3123, ORF1a:3195, ORF1a:3359, ORF1a:3454, ORF1a:3456, ORF1a:4110, ORF1a:4175, ORF1a:4197, ORF1a:4205, ORF1a:4207, ORF1a:4387, ORF1a:314, ORF1b:1181, ORF1b:1247, ORF1b:1424, ORF1b:2339, ORF1b:2340, ORF1b:2537, S:50, S:330, S:354, S:367, S:374, S:375-, S:376, S:405, S:508, S:551, S:573, S:647, S:653, S:657, S:659, S:668, S:859, S:944, S:958, S:978, S:1169, S:1179, S:1186, E:5, E:5, E:9, E:10, E:10, E:13-, E:14-, E:16, E:23, E:30, E:36, E:42, M:2, M:4, M:10, M:28, M:77, M:94, M:94, M:125, M:148, M:188, N:80, N:416, N:417, ORF7a:115, ORF9b:1, ORF9b:77")
###########################################################################################################################################################################
###########################################################################################################################################################################
### Justifications for excluded_muts:
# • "ORF1a:D1323G", "ORF1a:T1638N", "ORF1a:T4088I", "ORF1a:T4164I" —— All common Eightfold Muts that tangentially pop up.
#          Eightfold muts are an extensive set of CD8 T-cell escape mutations that occur together. If they sneak in at any point, they tend to 
#          completely dominate and snuff out all other muts, so it's easiest just to exclude them from the start. 
# • "ORF1b:A440V" —— Remdesivir-resistance mutation. Likely overrepresented in BAL sequences simply because the peopl most likely to have BAL samples
#                    are also the most likely to be treated with an antiviral like remdesivir.
# • "N:A208S" —— Inherited AY.103 mutation miscategorized by Nextclade as private 
# • "ORF1a:T2186A" —— Made one late version of BAL list, but in 2/3 seqs it is in, it is not actually a private mutation but was inherited.
###########################################################################################################################################################################
# This is to exclude all mutations that are part of the very extensive "8fold" CD8 T-cell escape mutation pattern. 
#   The difficulty with these is that if even one of these very tightly associated mutations sneaks in, it tends
#   to pull in dozens of other 8fold mutations, completely swamping any other mutation patterns present.
EightFold = Set(["ORF1a:R1318G", "ORF1a:T1322A", "ORF1a:T1322I", "ORF1a:D1323N", "ORF1a:D1323A", "ORF1a:D1323G", "ORF1a:N1324T", "ORF1a:N1324S", "ORF1a:N1324D", "ORF1a:I1326V", "ORF1a:T1638A", "ORF1a:T1638I", "ORF1a:T1638N", "ORF1a:D1639N", "ORF1a:D1639E", "ORF1a:D1639A", "ORF1a:P1640L", "ORF1a:L1643F", "ORF1a:T2274I", "ORF1a:A2279V", "ORF1a:T2280I", "ORF1a:T4087A", "ORF1a:T4087M", "ORF1a:T4087I", "ORF1a:T4088I", "ORF1a:T4090I", "ORF1a:T4090A", "ORF1a:T4164N", "ORF1a:T4164I", "ORF1a:D4165N", "ORF1a:D4165V", "ORF1a:D4165G", "ORF1a:D4165A", "ORF1a:D4166N", "ORF1a:D4166Y", "ORF1a:D4166G", "ORF1a:D4166A", "ORF1a:N4167S", "ORF1a:N4167K", "ORF1b:T730I", "ORF3a:Y211C", "ORF3a:Q213R", "ORF3a:Q213K", "ORF3a:Y215C", "ORF3a:Y215H"])
EightFold_pos_only = Set(["ORF1a:1318", "ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1326", "ORF1a:1638", "ORF1a:1639", "ORF1a:1640", "ORF1a:2274", "ORF1a:2280", "ORF1a:4087", "ORF1a:4088", "ORF1a:4090", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1b:730", "ORF3a:211", "ORF3a:213", "ORF3a:215"])
###########################################################################################################################################################################
###########################################################################################################################################################################
exclude_extras = Set(["ORF1b:A440V", "ORF1b:440", "ORF1a:T2186A", "ORF1a:2186", "N:A208S", "N:208"])
artifacts_ORF9bdoubles_8fold = union(artifactual_private_muts_subs, artifactual_private_muts_pos_only, Double_N_ORF9b_muts, Double_N_ORF9b_muts_pos_only, EightFold, EightFold_pos_only, exclude_extras)
artifacts_ORF9bdoubles = union(artifactual_private_muts_subs, artifactual_private_muts_pos_only, Double_N_ORF9b_muts, Double_N_ORF9b_muts_pos_only, exclude_extras)
###########################################################################################################################################################################
### keyless muts needed just to debug an error that kept cropping up
#keyless_muts = list_to_strings("E:A32V, S:N405D, ORF1a:H3076Y, ORF1b:T2537I, ORF1a:T4175I, E:V5A, E:T30I, ORF1a:T3058I, M:H125Y, E:V14-, S:A653V, ORF1a:S2972F, ORF1b:G662S, E:32, S:405, ORF1a:3076, ORF1b:2537, ORF1a:4175, E:5, E:30, ORF1a:3058, M:125, E:14, S:653, ORF1a:2972, ORF1b:662")
#keyless_muts = list_to_strings("E:32, S:405, ORF1a:3076, ORF1b:2537, ORF1a:4175, E:5, E:30, ORF1a:3058, M:125, E:14, S:653, ORF1a:2972, ORF1b:662")
###########################################################################################################################################################################
#####################################################   BEGIN Sub/pos_only Section   ######################################################################################
###########################################################################################################################################################################
artifactual_private_muts = Set{String}()
pango_AAsub_WT_universal = Dict{String, String}()
AA_muts_seq_universal = Dict{String, Set{String}}()
global mp_folder_universal = ""
mp_chr_all_ratio = Dict()
if sub_0__posonly_1 == 0
    pango_AAsub_WT_universal = pango_AAsub_WT
    AA_muts_seq_universal = AA_muts_seq
    artifactual_private_muts = artifactual_private_muts_subs
    Double_N_ORF9b_muts = Double_N_ORF9b_muts
    BAL_muts = BAL_muts_OG
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_subs_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_subs_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
    end
    mp_chr_all_ratio = AA_muts_ct_chr_all_ratio
elseif sub_0__posonly_1 == 1
    pango_AAsub_WT_universal = pango_AAsub_WT_pos_only
    AA_muts_seq_universal = AA_muts_seq_pos_only
    artifactual_private_muts = artifactual_private_muts_pos_only
    Double_N_ORF9b_muts = Double_N_ORF9b_muts_pos_only
    BAL_muts = BAL_muts_pos_only
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_pos_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_pos_only_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
    end    
    mp_chr_all_ratio = AA_muts_ct_pos_only_no_dels_chr_all_ratio
end
#################################################################################################
function sel_muts_pt1_pos_only_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_pos_only_sortKey_2(string(split(n, ", ")[1]))
    end
end
############################################################################################
function sel_muts_pt1_sort_key_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return sel_muts_pt1_sort_key(n)
    elseif sub_0__posonly_1 == 1
        return sel_muts_pt1_pos_only_sort_key(n)
    end
end
############################################################
function mp_AA_gene_sortKey_2_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return mp_AA_gene_sortKey_2(n)
    elseif sub_0__posonly_1 == 1
        return mp_AA_gene_pos_only_sortKey_2(n)
    end
end    
############################################################
function AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return AA_muts_ct_no_dels
    end
    if sub_0__posonly_1 == 1
        return AA_muts_ct_pos_only_no_dels
    end
end
AA_muts_ct_no_dels_universal = AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1)
if AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_no_dels")
end
if AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only_no_dels")
end
############################################################
function seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return seq_AA_muts_no_dels
    elseif sub_0__posonly_1 == 1
        return seq_AA_muts_pos_only_no_dels
    end
end
seq_AA_muts_no_dels_universal = seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1)
####################################################################################
artifacts_ORF9bdoubles = union(artifactual_private_muts, Double_N_ORF9b_muts)
###########################################################################################################################################################################
###########################################################################################################################################################################
global RBM_muts = Set{String}()
global RBD_muts = Set{String}()
global RBD_not_RBM_muts = Set{String}()
global spike_not_RBD_muts = Set{String}()
global spike_muts = Set{String}()
global nonspike_muts = Set{String}()
RBD_sites = BitSet(335:528)
RBD_no_RBM_1 = BitSet(335:437)
RBD_no_RBM_2 = BitSet(507:528)
RBD_not_RBM_sites = union(RBD_no_RBM_1, RBD_no_RBM_2)
RBM_sites = BitSet(438:506)
spike_not_RBD1 = BitSet(1:334)
spike_not_RBD2 = BitSet(529:1273)
spike_not_RBD_sites = union(spike_not_RBD1, spike_not_RBD2)
for mut in keys(AA_muts_ct_no_dels_universal)
    if aa_gene_comprehensive_dict[mut] == "S"
        push!(spike_muts, mut)
    else
        push!(nonspike_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_not_RBM_sites
        push!(RBD_not_RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in spike_not_RBD_sites
        push!(spike_not_RBD_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBM_sites
        push!(RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_sites
        push!(RBD_muts, mut)
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
global purespikebanned_0__purespikeallowed_1 = 0
global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
global all_excluded_muts = Set{String}()
if normal_0__spikeonly_1 == 0
    purespikebanned_0__purespikeallowed_1 = 0
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifacts_ORF9bdoubles_8fold, BAL_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifacts_ORF9bdoubles_8fold)
    end
elseif normal_0__spikeonly_1 == 1
    purespikebanned_0__purespikeallowed_1 = 1
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 3
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, BAL_muts, nonspike_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, nonspike_muts)
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
RBD_not_RBM_muts_sort = sort(collect(RBD_not_RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_not_RBM_tot = length(RBD_not_RBM_muts_sort)
println("Total Different RBD-not-RBM Muts = $(RBD_not_RBM_tot)")
RBD_not_RBM_muts_sort_join = join(RBD_not_RBM_muts_sort, ", ")
########################
RBM_muts_sort = sort(collect(RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBM_tot = length(RBM_muts_sort)
println("Total Different RBM Muts = $(RBM_tot)")
RBM_muts_sort_join = join(RBM_muts_sort, ", ")
########################
RBD_muts_sort = sort(collect(RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_tot = length(RBD_muts_sort)
println("Total Different RBD Muts = $(RBD_tot)")
RBD_muts_sort_join = join(RBD_muts_sort, ", ")
########################
spike_not_RBD_muts_sort = sort(collect(spike_not_RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
spike_not_RBD_tot = length(spike_not_RBD_muts_sort)
println("Total Different spike_not_RBD Muts = $(spike_not_RBD_tot)")
spike_not_RBD_muts_sort_join = join(spike_not_RBD_muts_sort, ", ")
#####################################################################################################################################
ORF9bNdoubles_artifacts = union(Double_N_ORF9b_muts, artifactual_private_muts)
tot_grp_ct = length(rep_seq_grps_maxmut_seqs)
tot_single_ct = length(non_rep_seqs)
tot_chr_seq_ct = tot_grp_ct + tot_single_ct
total_chronic_AA_ct = 0
total_chronic_AA_ct_nonRBM = 0
total_chronic_AA_ct_nonRBD = 0
total_chronic_AA_ct_spike_nonRBD = 0
total_chronic_AA_ct_nonspike = 0
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in artifacts_ORF9bdoubles)
        total_chronic_AA_ct += ct
        if aa_gene_comprehensive_dict[mut] ≠ "S"
            total_chronic_AA_ct_nonspike += ct
            total_chronic_AA_ct_nonRBM += ct
            total_chronic_AA_ct_nonRBD += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
            total_chronic_AA_ct_nonRBM += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
            total_chronic_AA_ct_nonRBD += ct
            total_chronic_AA_ct_spike_nonRBD += ct
        end
    end
end
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq = round(digits=2, total_chronic_AA_ct/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq = $(avg_AA_sub_ct_per_chronic_seq)")
avg_AA_sub_ct_per_chronic_seq_nonspike = round(digits=2, total_chronic_AA_ct_nonspike/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonspike = $(avg_AA_sub_ct_per_chronic_seq_nonspike)")
avg_AA_sub_ct_per_chronic_seq_nonRBM = round(digits=2, total_chronic_AA_ct_nonRBM/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_nonRBD = round(digits=2, total_chronic_AA_ct_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = round(digits=2, total_chronic_AA_ct_spike_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_spike_nonRBD)")
######################################################################################################################################
#seq_privAA_nonRBD_len = Dict{String, Int}()
#for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
#    seq_nonRBD_ct = 0
#    for sub in AAsubvec
#        if !(sub in RBD_muts)
#            seq_nonRBD_ct += 1
#        end
#    end
#    seq_privAA_nonRBD_len[seq] = seq_nonRBD_ct
#end
######################################################################################################################################
#seq_privAA_nonRBD_len_relative = Dict{String, Float64}()
#for (seq, ct) in seq_privAA_nonRBD_len
#    relative_nonRBD = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_nonRBD)
#    seq_privAA_nonRBD_len_relative[seq] = relative_nonRBD
#end
#####################################################################################################################
global avg_AA_sub_ct_per_chronic_seq_for_main_fx = 0.0
#####################################################################################################################
if normal_0__spikeonly_1 == 0
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_nonRBD
elseif normal_0__spikeonly_1 == 1
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_spike_nonRBD
end
#####################################################################################################################
# all_excluded_muts
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
    seq_mut_ct = 0
    for sub in AAsubvec
        if !(sub in all_excluded_muts)
            seq_mut_ct += 1
        end
    end
    seq_privAA_len[seq] = seq_mut_ct
end
#####################################################################################################################
seq_privAA_len_relative = Dict{String, Float64}()
for (seq, ct) in seq_privAA_len
    relative_muts = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
    seq_privAA_len_relative[seq] = relative_muts
end
#####################################################################################################################
total_chronic_AA_ct_v2 = 0
total_chronic_AA_ct_v2_nonRBM = 0
total_chronic_AA_ct_v2_nonRBD = 0
total_chronic_AA_ct_v2_spike_nonRBD = 0
total_chronic_AA_ct_v2_nonspike = 0
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts_no_dels_universal[seq]
        if !(mut in artifacts_ORF9bdoubles)
            total_chronic_AA_ct_v2 += 1
            if aa_gene_comprehensive_dict[mut] ≠ "S"
                total_chronic_AA_ct_v2_nonspike += 1
                total_chronic_AA_ct_v2_nonRBM += 1
                total_chronic_AA_ct_v2_nonRBD += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
                total_chronic_AA_ct_v2_nonRBM += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
                total_chronic_AA_ct_v2_nonRBD += 1
                total_chronic_AA_ct_v2_spike_nonRBD += 1
            end
        end
    end
end
####################################################################################################################################
avg_AA_sub_ct_per_chronic_seq_v2 = round(digits=2, total_chronic_AA_ct_v2/all_unique_chr_seqs_ct)  #### Double checking the first count
println("avg_AA_sub_ct_per_chronic_seq_v2 = $(avg_AA_sub_ct_per_chronic_seq_v2)")
println("total_chronic_AA_ct = $(total_chronic_AA_ct)")
println("total_chronic_AA_ct_v2 = $(total_chronic_AA_ct_v2)")
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = round(digits=2, total_chronic_AA_ct_v2_nonspike/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonspike = $(avg_AA_sub_ct_per_chronic_seq_v2_nonspike)")
println("total_chronic_AA_ct_nonspike = $(total_chronic_AA_ct_nonspike)")
println("total_chronic_AA_ct_v2_nonspike = $(total_chronic_AA_ct_v2_nonspike)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = round(digits=2, total_chronic_AA_ct_v2_nonRBM/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBM)")
println("total_chronic_AA_ct_nonRBM = $(total_chronic_AA_ct_nonRBM)")
println("total_chronic_AA_ct_v2_nonRBM = $(total_chronic_AA_ct_v2_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = round(digits=2, total_chronic_AA_ct_v2_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBD)")
println("total_chronic_AA_ct_nonRBD = $(total_chronic_AA_ct_nonRBD)")
println("total_chronic_AA_ct_v2_nonRBD = $(total_chronic_AA_ct_v2_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = round(digits=2, total_chronic_AA_ct_v2_spike_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD)")
println("total_chronic_AA_ct_spike_nonRBD = $(total_chronic_AA_ct_spike_nonRBD)")
println("total_chronic_AA_ct_v2_spike_nonRBD = $(total_chronic_AA_ct_v2_spike_nonRBD)"); print("\n"^1)
println("Finished!"); print("\n"^1)
##############################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip"); print("\n"^1)
####################################################################################################################################
####################################################################################################################################


2026_03_04__615PM

AA_muts_ct_no_dels_universal ≠ AA_muts_ct_no_dels
AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels
Total Different RBD-not-RBM Muts = 81
Total Different RBM Muts = 54
Total Different RBD Muts = 135
Total Different spike_not_RBD Muts = 793
avg_AA_sub_ct_per_chronic_seq = 17.6
avg_AA_sub_ct_per_chronic_seq_nonspike = 11.7
avg_AA_sub_ct_per_chronic_seq_nonRBM = 16.22
avg_AA_sub_ct_per_chronic_seq_nonRBD = 15.32
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = 3.62
avg_AA_sub_ct_per_chronic_seq_v2 = 17.44
total_chronic_AA_ct = 58036
total_chronic_AA_ct_v2 = 57513
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = 11.64
total_chronic_AA_ct_nonspike = 38562
total_chronic_AA_ct_v2_nonspike = 38373
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = 16.06
total_chronic_AA_ct_nonRBM = 53463
total_chronic_AA_ct_v2_nonRBM = 52961
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = 15.17
total_chronic_AA_ct_nonRBD = 50512
total_chronic_AA_ct_v2_nonRBD = 50012
avg_AA_sub_ct_per_chronic_seq_v2_spike

In [369]:
### Making megaclade_EPCI_ct_dict and megaclade_EPCI_proportion_dict
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
x_minor_ct = 0
x_minor_dict_ct = Dict{String, Int}()
unknown_EPCI_ct = 0
pre_VOC_ct = 0
pre_VOC_dict_ct = Dict{String, Int}()
pre_VOC_vec = String[]
total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
megaclade_EPCI_ct_dict = Dict{String, Int}()
megaclade_EPCI_proportion_dict = Dict{String, Float64}()
for seq in all_unique_chr_seqs_set
    complete = 0
    pango = seq_pango[seq]
    unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
    if length(unaliased) ≥ 3
        if unaliased[1:3] == "XBB"
            megaclade_EPCI_ct_dict["XBB"] = get(megaclade_EPCI_ct_dict, "XBB", 0) + 1
            complete = 1
        elseif unaliased[1:3] == "XEC" || unaliased[1:3] == "XEK" || unaliased[1:3] == "XFG"
            megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
            complete = 1
        elseif unaliased[1:3] == "XCH"
            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
            complete = 1
        end
        if length(unaliased) ≥ 7
            if unaliased[1:7] == "B.1.1.7"
                megaclade_EPCI_ct_dict["B.1.1.7"] = get(megaclade_EPCI_ct_dict, "B.1.1.7", 0) + 1
                complete = 1
            elseif unaliased[1:7] == "B.1.351"
                megaclade_EPCI_ct_dict["B.1.351"] = get(megaclade_EPCI_ct_dict, "B.1.351", 0) + 1
                complete = 1
            elseif unaliased[1:7] == "B.1.617"
                megaclade_EPCI_ct_dict["B.1.617"] = get(megaclade_EPCI_ct_dict, "B.1.617", 0) + 1
                complete = 1
            end
            if length(unaliased) ≥ 9
                if unaliased == "B.1.1.529"
                    megaclade_EPCI_ct_dict["Omicron_Unknown"] = get(megaclade_EPCI_ct_dict, "Omicron_Unknown", 0) + 1
                    complete = 1
                end
                if length(unaliased) ≥ 10
                    if unaliased[1:10] == "B.1.1.28.1"
                        megaclade_EPCI_ct_dict["P.1"] = get(megaclade_EPCI_ct_dict, "P.1", 0) + 1
                        complete = 1
                    end
                    if length(unaliased) ≥ 11
                        if unaliased[1:11] == "B.1.1.529.1"
                            megaclade_EPCI_ct_dict["BA.1"] = get(megaclade_EPCI_ct_dict, "BA.1", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.2"
                            megaclade_EPCI_ct_dict["BA.2"] = get(megaclade_EPCI_ct_dict, "BA.2", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.4"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.5"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            complete = 1
                        end
                        if length(unaliased) ≥ 14
                            if unaliased[1:14] == "B.1.1.529.2.86"
                                megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
                                megaclade_EPCI_ct_dict["BA.2"] = megaclade_EPCI_ct_dict["BA.2"] - 1
                                complete = 1
                            end
                        end
                    end
                end
            end
        end
    end
    if complete == 0
        if unaliased[1] == 'X'
            x_minor_ct += 1
            x_minor_dict_ct[unaliased] = get(x_minor_dict_ct, unaliased, 0) + 1
            megaclade_EPCI_ct_dict["Minor_Recombinant"] = get(megaclade_EPCI_ct_dict, "Minor_Recombinant", 0) + 1
            complete = 1
        end
        if unaliased[1] == 'A' || unaliased[1] == 'B'
            pre_VOC_ct += 1
            pre_VOC_dict_ct[unaliased] = get(pre_VOC_dict_ct, unaliased, 0) + 1
#            println(pango)
            megaclade_EPCI_ct_dict["pre_VOC"] = get(megaclade_EPCI_ct_dict, "pre_VOC", 0) + 1
            complete = 1
        end
    end
    if complete == 0
        println("Messed Up Pango = $(pango)")
        unknown_EPCI_ct += 1
    end
end
######################################################################################################################
for (megaclade, count) in megaclade_EPCI_ct_dict
    megaclade_EPCI_proportion_dict[megaclade] = count/total_EPCI_seq_ct
end
######################################################################################################################
print("\n"^2)
println("unknown_EPCI_ct = $(unknown_EPCI_ct)")
println("x_minor_ct = $(x_minor_ct)")
println("pre_VOC_ct =$(pre_VOC_ct)")
print("\n"^1)
x_minor_dict_ct_sort = sort(collect(x_minor_dict_ct), by = x -> x[1])
for minorX___ct in x_minor_dict_ct_sort
    minorX_pad = rpad(minorX___ct[1], 12)
    count_pad = lpad(minorX___ct[2], 2)
#    println("$(minorX_pad) = $(count_pad)")
end
#print("\n"^1)   
pre_VOC_dict_ct_sort = sort(collect(pre_VOC_dict_ct), by = x -> x[1])
for preVOC___ct in pre_VOC_dict_ct_sort
    preVOC_pad = rpad(preVOC___ct[1], 12)
    count_pad = lpad(preVOC___ct[2], 2)
#    println("$(preVOC_pad) = $(count_pad)")
end
print("\n"^1) 
######################################################################################################################
megaclade_EPCI_ct_dict_sort = sort(collect(megaclade_EPCI_ct_dict), by = x -> x[2], rev=true)
for megaclade____count in megaclade_EPCI_ct_dict_sort
    megaclade = megaclade____count[1]
    count = megaclade____count[2]
    println("$(megaclade) = $(count)")
end
print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
print("\n"^1)
######################################################################################################################
######################################################################################################################

2026_03_04__615PM
6:15.07_PM


unknown_EPCI_ct = 0
x_minor_ct = 28
pre_VOC_ct =292


BA.5 = 718
BA.2 = 656
BA.1 = 640
XBB = 498
pre_VOC = 292
BA.2.86 = 228
B.1.617 = 183
B.1.1.7 = 30
Minor_Recombinant = 28
B.1.351 = 10
Omicron_Unknown = 9
P.1 = 5

2026_03_04__615PM
6:15.08_PM



In [370]:
######################### AA_mut_pattern_iterative function BAL version COMBINED ################################ 
date_now_fx = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now_fx)
date = Dates.format(today(), "yyyy_mm_dd")
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
#####################################################################################################################################
pos_only_option = ""
if sub_0__posonly_1 == 0
    pos_only_option = ""
elseif sub_0__posonly_1 == 1
    pos_only_option = "POS_ONLY"
end
#####################################################################################################################################
rep_seq_grps_maxmut_seqs_set = collect(values(rep_seq_grps_maxmut_seqs))
total_chronic_ct = length(all_unique_chr_seqs)
#####################################################################################################################################
### The unknown/missing AA are used to adjust the denominator for a given mutation. This part isn't that relevant.
# E.g. if we have 10 seqs & 4 have M:S4P but two lack coverage or have a mixed nuc at M:4...
#      then the percentage of seqs with M:S4P is 50% (4/8) rather than 40% (4/10)
#total_AApos_ct_unknown = Dict{String, Int}()  ### Total unknown count for each AA position in independent chronic lineages (not currently used, may dump later)
#for seq in all_unique_chr_seqs_set
#    for pos in seq_unknown_AA[seq]
#        total_AApos_ct_unknown[pos] = get(total_AApos_ct_unknown, pos, 0) + 1
#    end
#end 
######################################################################################################################################
###################################### Preliminary BAL Mutation Set Determination ####################################################
###################################################################################################################################### 
### These are seqs explicitly described as being from "bronchoalveolar lavage" (30), "Broncho alveolar liquid" (1), "Bronchial Aspiration" (1), "Bronchoaspiration" (1),
#   "Lung, Right" (1), "Lung, Right, Middle Lobe" (1), "lavage" (1), "alveolar lavage fluid" (1), "lower respiratory sample" (1), or "organ" (1).
BAL_uniq = list_to_strings("EPI_ISL_1517099, EPI_ISL_1908476, EPI_ISL_2484152, EPI_ISL_3982251, EPI_ISL_4072038, EPI_ISL_4935949, EPI_ISL_9873278, EPI_ISL_10590270, EPI_ISL_11349763, EPI_ISL_13503362, EPI_ISL_14439530, EPI_ISL_14479735, EPI_ISL_14553998, EPI_ISL_15511842, EPI_ISL_15687965, EPI_ISL_16157875, EPI_ISL_16356910, EPI_ISL_16691487, EPI_ISL_16868655, EPI_ISL_16957015, EPI_ISL_17374807, EPI_ISL_17408352, EPI_ISL_17634290, EPI_ISL_17709926, EPI_ISL_17719124, EPI_ISL_17777729, EPI_ISL_17826285, EPI_ISL_17960749, EPI_ISL_17964486, EPI_ISL_18111040, EPI_ISL_18111041, EPI_ISL_18116015, EPI_ISL_19147578, EPI_ISL_19302363, EPI_ISL_19364322, EPI_ISL_19742521, EPI_ISL_19878655, EPI_ISL_19977768")
# Consider adding confirmed BAL sequence: EPI_ISL_2036230 (only 5 private AA muts, but collected 2-4 months after related seqs; private AA muts: E—T30I, ORF1a—N2361K, T3058I, Y3074H, F3366L
tot_BAL_seq = length(BAL_uniq)
### Tracheal Aspirate Seqs (not included): EPI_ISL_678289, EPI_ISL_5944948, EPI_ISL_12157187
### Plasma seq (not included): EPI_ISL_14596883
#### Eightfold muts are an extensive set of CD8 T-cell escape mutations that occur together. If they sneak in at any point, they tend to 
#        completely dominate and snuff out all other muts, so it's easiest just to exclude them from the start. 
BAL_uniq_seq_set = Set(BAL_uniq)
BAL_muts = Dict{String, Int}()
BAL_muts_start = Set{String}()
for seq in BAL_uniq_seq_set
    for mut in seq_AA_muts_no_dels_universal[seq]
        if !(mut in all_excluded_muts)
            BAL_muts[mut] = get(BAL_muts, mut, 0) + 1
        end
    end
end
## Creating an initial set of BAL-associated mutations to start with, i.e. the muts in ≥ 3 of the 38 confirmed chronic BAL sequences
#    and which also is ≥ 3 times as common in confirmed BAL seqs than it is in all EPCI seqs
## S:S621P very likely artifactual reversion (or misinterpretation of ∆620-621     # "ORF1b:662", "ORF1a:1500", 
## M:I8F is part of a M:∆4-7/∆5-8 deletion, and M:G6- is aready on the list. 
banned_initial_BAL_muts = ["S:S621P", "M:I8F"] 
for (mut, ct) in BAL_muts
    BAL_pct = 100*ct/tot_BAL_seq
    tot_pct = 100*AA_muts_ct_no_dels_universal[mut]/total_chronic_count_BAL
    ratio_rd = round(digits = 2, BAL_pct/tot_pct)
    if (ct ≥ 4 && BAL_pct/tot_pct ≥ 3) || (ct ≥ 3 && BAL_pct/tot_pct ≥ 5) || (ct ≥ 2 && BAL_pct/tot_pct ≥ 10) && !(mut in banned_initial_BAL_muts) # && !(mut in banned_initial_BAL_muts) #  && !(mut == "ORF1a:T814I") && !(mut == "ORF1a:814")
        push!(BAL_muts_start, mut)
    end
end
##########################################################################################
println("Length of confirmed BAL seqs list = $(tot_BAL_seq)") 
for m in BAL_muts_start
    print(m, ", ")
end
println()
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### mp_seqs stands for "mutational pattern sequences"; These are the ones that qualify from previous rund by having a minimum number
#     of mutations in the pattern. 
#                                   Round  list_of_muts_in_round
pattern_mut_set_by_round_dict = Dict{Int, Set{String}}()
pattern_mut_set_by_round_dict[0] = Set{String}()
#                                 Round  ct_of_muts_in_round
pattern_mut_ct_by_round_dict = Dict{Int, Int}()
pattern_mut_ct_by_round_dict[0] = 0
new_mut_by_round_dict = Dict{Int, Set{String}}()
new_mut_by_round_dict[0] = Set{String}()
lost_mut_by_round_dict = Dict{Int, Vector{String}}()
lost_mut_by_round_dict[0] = Vector{String}()
reason_lost_by_round = Dict{Int, Vector{String}}()
reason_lost_by_round[0] = Vector{String}()
### Stores the min Chi^2 scores a mutation needs to qualify for each round
min_chi2_by_round_dict = Dict{Int,Float64}()
min_chi2_by_round_dict[0] = 50
mut_pattern_round_ct = 0
removed_for_low_ratios = Set{String}()
################################################## Parameter explanations ###########################################################
# • mut_pattern_name: self-explanatory
# • tot_rounds: number of rounds to run
# • plus_or_minus: used to detect muts near (within plus_or_minus) already-qualified muts & reduce the strictness of the 
#     requirements they have to meet to qualify. 
# • min_AA_ct: The minimum times an AA sub must appear in all chronics in order to qualify. 
# • mp_seqs: stands for "mutational pattern sequences", i.e. sequences that qualified in previous round (by meeting all criteria). 
#     At the very beginning, a "seed" set of sequences is usually specified. E.g., all confirmed chronic bronchoalveoloar lavage-sample sequences.
# • mp_ct_min: minimum number of mutations a from the mutation pattern a sequence must have to qualify
# • min_log_pv_fish: the *starting value* for the minimum -log10 p-value for Fisher's Exact Test a sequence needs to qualify
#      After each round, this minimum score is decreased until it reaches abs_min_fish
# • abs_min_fish: the absolute minimum -log10 p-value for Fisher's Exact Test, which is gradually approached as the function proceeds through rounds
# • min_chi_sq: the *starting value* for the minimum Chi^2 a mut must reach to be added to the mut-pattern list
#      After each round, this minimum score is decreased until it reaches abs_chi2_min
# • abs_chi2_min: the absolute minimum Chi^2 value, which is gradually reached as the function proceeds through rounds
# • min_chr_all_ratio: (NO LONGER USED) minimum chronic-to-circulating mutation ratio. Designed to exclude mutations that are very common in general 
#      and which therefore can't really be described as characteristic of a certain mutation pattern. 
#      It's tough to know exactly where to draw the line on this one.
# • seq_factor_OFF_HALF_ON__0_1_2: Optional & somewhat experimental parameter that adjusts the number of mut-pattern AA muts a
#      sequence must have based on the total number of AA muts the seq has. It's based on the average number of AA subs per sequence
#      on the entire chronic list (~3200 seqs). If a sequence has twice as many AA subs as the average chronic seq, it has to have
#      twice as many muts from the mut-pattern list to qualify. The half_seq_factor option splits the difference, i.e. if a sequence
#      has twice as many AA subs as the avg chronic, it must have 50% more muts from the mut pattern.
#####################################################################################################################################
#                                          
#                   Round     Mutation     EPCI_pct,MP_pct,non_MP_pct,MP_ct,EPCIct,MPseqct,chi2,log_pv_fish,mut_BAL_prop_avg,mut_BAL_seq_tot_avg,fold_incr
#df_props_dict = Dict{Int, Dict{String, Tuple{Float64,Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64,Float64} }}()
df_props_dict = Dict{Int, Dict{String, Tuple{Float64,Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Union{Float64,String}} } }()
#### df_props_dict is needed to record all the specific stats for each mutation in each round (for printing results & csv/tsv purposes)
## mp_seqs = all seqs that meet criteria            1                      2                3                      4                  5                6               7                      8                      9                      10                    11                    12                        13                               14
function AA_mut_pattern_iterative_2025_11_29(mut_pattern_name::String, minQryPct::Int, min_BAL_prop::Float64, tot_rounds::Int, plus_or_minus::Int, min_AA_ct::Int, mp_seqs::Set{String}, mp_ct_min::Float64, min_log_pv_fish::Float64, abs_min_fish::Float64, min_chi_sq::Float64, abs_chi2_min::Float64, min_chr_all_ratio::Float64, seq_factor_OFF_HALF_ON__0_1_2::Int)
    global all_unique_chr_seqs_set
    global BAL_seqs
    global BAL_muts_start
    global mut_pattern_round_ct
    global pattern_mut_set_by_round_dict
    global pattern_mut_ct_by_round_dict
    global new_mut_by_round_dict
    global lost_mut_by_round_dict
    global reason_lost_by_round
    global df_props_dict
    total_mut_ct = 0
    total_AA_mut_ct_nd = 0
    total_mut_po_ct = 0
###############################################
########### df_mp columns key #################
#        Mutation = mutation being analyzed
#        BAL_Mut_ct = Total BAL-MP sequences w/mutation
#        EPCI_Mut_ct = Total EPCI seqs w/mutation
#        Adjusted_BAL_Seq_ct = Total BAL-MP seqs, adjusted for dropout & inherited mutations
#        EPCI_pct = Percentage of EPCI muts w/mutation that appear in BAL-MP seqs  (e.g. if 10 EPCI seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
#        BAL_MP_pct = Percentage of all BAL-MP seqs w/the mutation  (e.g. if 100 seqs meet the criteria & 4 of those have E:T30I, MP_prop = 0.04)
#        Fishers_Exact_Pval = p-value for Fisher's Exact test
#        Fold_Incr = Fold-increase in frequency of seqs w/mutation in BAL-MP seqs relative to frequency in all EPCI seqs
#        MP_AvgBALProp = Average proportion of each qualifying sequence's private mutations that are BAL-MP mutations
#        MP_AvgBALMut = Average number of private BAL-MP mutations for each qualifying seq w/mutation
#        EPCI_AvgBALProp = Average proportion of private mutations that are BAL-MP mutations for all EPCI sequences w/mutation
#        EPCI_AvgBALMut = Average number of private BAL-MP mutations for all EPCI sequences w/mutation
#        log10pvFish = -log10 p-value for Fisher's Exact test
#        Chi2 = chi-squared value
    df_mp = DataFrame(
        Mutation = String[], 
        BAL_Mut_ct = Float64[], 
        EPCI_Mut_ct = Int[], 
        Adjusted_BAL_Seq_ct = Int[],
        EPCI_pct = Float64[],  
        BAL_MP_pct = Float64[],
        nonBAL_MP_pct = Float64[],
        Fishers_Exact_Pval = Float64[], 
        Fold_Incr = Union{Float64,String}[],
        MP_AvgBALProp = Float64[],
        MP_AvgBALMut = Float64[],
        EPCI_AvgBALProp_v1 = Float64[],
        EPCI_AvgBALMut_v1 = Float64[],
        EPCI_AvgBALProp_v2 = Float64[],
        EPCI_AvgBALMut_v2 = Float64[],
        log10pvFish = Float64[],
        Chi2 = Float64[])
###############################################
    minBALpct = 100*min_BAL_prop
    folder_name = "$(mut_pattern_name)_$(pos_only_option)_$(date_now_fx)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(round(digits=2, abs_min_fish))__minEPCIpct$(minQryPct)__minBALpct$(minBALpct)__EPCI_$(EPCI_qc_str)"
    mkpath(folder_name)
    mut_file = "$(folder_name)/$(folder_name)_muts_by_round.txt"
######################################################################################################################################
################################ Below: New stuff for dealing with unknown AA (2025-3-24) ############################################
######################################################################################################################################
#                      unknown_AA_pos  count
    mp_mut_ct_unknown = Dict{String, Int}()      ## mp_mut_ct_unknown = number of unknown/mixed AA at eery site in current seq set
    mp_mut_ct = Dict{String, Float64}()          ## mp_mut_ct = number of sequences meeting specified criteria that have a given AA mut
######################################################################################################################################
    mp_seqs_priv_AA_adjustment_factors = Set{Float64}()
#    excluded_muts = Set(["S:F375A", "S:A376P", "N:P80L", "ORF1a:D1323G", "ORF1a:T1638N", "ORF1a:T4088I", "ORF1a:T4164I", "ORF1b:A440V"])
######################################################################################################################################
######################################################################################################################################
### mp_seqs_1_mut = Set of sequences that would have qualified for list even is the given mutation is subtracted from each sequence's 
#     count. (Only for sequences that were in the previous round & therefore were used to select the current sequences.)
#     E.g. if each seq must have 2 muts from the BAL list to qualify, then to qualify for the mp_seqs_1_mut["M:S4P"] list,
#     a sequence that has M:S4P must have ≥ 3 mutations from the list (i.e., M:S4P plus two others). 
    mp_seqs_1_mut = Dict{String, Set{String}}()
###  mp_mut_ct_1_mut = Separate mp_mut_ct counts for each mut that was in the previous round (& therefore used to select the current sequences)
#    Reminder: mp_mut_ct = number of sequences meeting specified criteria that have a given AA mut
    mp_mut_ct_1_mut = Dict{String, Float64}()     
    mp_mut_ct_unknown_1_mut = Dict{String, Int}()
    total_AA_mut_ct_nd_1_mut = Dict{String, Int}()
######################################################################################################################################
######################################################################################################################################
##### The parameter seq_factor_OFF_HALF_ON__0_1_2 is designed to adjust for the number of private AA subs in a sequence. If a sequence 
#     has 70 private AA muts, for example, it's not as surprising if it has a couple AA subs on the mut-pattern list, and its AA subs
#     are therefore discounted by a factor of (private_AA_subs_inseq/avg_AA_subs_per_chronic). Sequences w/fewer AA subs than
#     the typical chronic seq are similarly amplified by the same factor. The half_seq_factor option cuts seq_factor in half, and
#     the clamp option limits this adjustment factor to a minimum of 0.6 and a maximum of 1.5.  
######################################################################################################################################
#    total_starts = 0
#    total_stops = 0
    for seq in mp_seqs
        seq_priv_AA_mut_total = length(seq_AA_muts_no_dels_universal[seq])
        for mut in seq_AA_muts_no_dels_universal[seq]
            if !(mut in all_excluded_muts)
                mp_mut_ct[mut] = get(mp_mut_ct, mut, 0) + 1
                total_mut_ct += 1 
##### Because the effect of stop codons are very similar, regardless of the exact mutation, I initially created two new categories 
#     of mutations for each gene, stop codon-creating muts & start codon-destroying muts, tested just like other mutations. 
#     Start-codon muts had no effect (possibly due to "backup" start codons), so I've commented them out for the sake of simplicity.
#                if string(mut[end]) == "*"
#                    gene = aa_gene_comprehensive_dict[mut]
#                    stops = "$(gene):X999*"
#                    mp_mut_ct[stops] = get(mp_mut_ct, stops, 0) + 1
#                    total_stops += 1
#                end
#                if aa_pos_comprehensive_dict[mut] == 1
#                    gene = aa_gene_comprehensive_dict[mut]
#                    starts = "$(gene):M1X" 
#                    mp_mut_ct[starts] = get(mp_mut_ct, starts, 0) + 1
#                    total_starts += 1
#                end
            end
        end
        total_AA_mut_ct_nd += seq_priv_AA_mut_total
    end
#    keyless_muts = list_to_strings("E:A32V, S:N405D, ORF1a:H3076Y, ORF1b:T2537I, ORF1a:T4175I, E:V5A, E:T30I, ORF1a:T3058I, M:H125Y, E:V14-, S:A653V, ORF1a:S2972F, ORF1b:G662S")
#    for mut in keyless_muts
#        mp_mut_ct[mut] = get(mp_mut_ct, mut, 0)
#    end
######################################################################################################################################
    for seq in mp_seqs
        for mutpo in seq_unknown_AA[seq]
            mp_mut_ct_unknown[mutpo] = get(mp_mut_ct_unknown, mutpo, 0) + 1
        end
    end
######################################################################################################################################
    last_rd_BAL_ct = length(pattern_mut_set_by_round_dict[mut_pattern_round_ct])
    MP_seq_ct = length(mp_seqs)
    avg_mutations_per_seq = total_AA_mut_ct_nd/MP_seq_ct
    avg_mutations_per_seq_rd = round(digits=2, avg_mutations_per_seq)
    ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics = avg_mutations_per_seq/avg_AA_sub_ct_per_chronic_seq
    ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd = round(digits=2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics)
######################################################################################################################################
# EPCI_prop, MP_prop, EPCI_mut_ct, ct, tot_qual_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg, mut_BAL_seq_tot_avg, mut_EPCI_BAL_prop_avg, mut_EPCI_BAL_seq_tot_avg, fold_incr
    prop_dict = Dict{String, Tuple{Float64,Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Union{Float64,String},Float64,Float64,Float64,Float64} }()
    prop_dict_unknown = Dict{String, Float64}()
    for (mut, ct) in mp_mut_ct_unknown
        MP_pct_rd = round(digits=1, 100*ct/MP_seq_ct)
        prop_dict_unknown[mut] = MP_pct_rd
    end
######################################################################################################################################
######################################################################################################################################
#### coAAmut_ct_pt1_unknown counts the # of times a given AA position is "unknown," either due to dropout or a mixed nucleotide 
#       for the sequences that qualify
#                                 unknown_AA_pos | count
            coAAmut_ct_pt1_unknown = Dict{String, Int}()
            total_AApos_ct_unknown = Dict{String, Int}()  ### Total unknown count for each AA position in each independent EPCI lineage
            for seq in all_unique_chr_seqs_set
                for pos in seq_unknown_AA[seq]
                    if !haskey(total_AApos_ct_unknown, pos)
                        total_AApos_ct_unknown[pos] = 1
                    else
                        total_AApos_ct_unknown[pos] += 1
                    end
                end
            end
######################################################################################################################################
######################################################################################################################################
        mp_seqs_pangos = Dict{String, Int}()  ## mp_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
        mp_seqs_inherited = Dict{String, Int}()  ## mp_seqs_inherited = Counts for inherited mutations among sequences on list. 
######################################################################################################################################
#### This counts the pango lineages for each sequence on the list, then counts all the mutations inherited by that pango lineage.
#### Similar to the "unknown" dicts below, the purpose here is to remove from the denominator sequences that could not possibly have had a given 
####    private mutation (because you can't acquire a mutation you already have). For example, let's say we're testing whether the private mutation 
####    S:H655Y correlates with other mutations in a list of 100 sequences. If 70 of thoser sequences are Omicron, all of which inherited S:H655Y,
####    then only the 30 non-Omicron sequences will be used in calculating the Fisher's exact test (as well as all other calculations).
            for seq in mp_seqs
                pango = seq_pango[seq]
                mp_seqs_pangos[pango] = get(mp_seqs_pangos, pango, 0) + 1
                if !haskey(pango_AAsub_WT_universal, pango)
                    pango = pango_predecessor_meta_dict[pango][1]
                    if !haskey(pango_AAsub_WT_universal, pango)
                        pango = pango_predecessor_meta_dict[pango][1]
                    end
                end
                for mut in pango_AAsub_WT_universal[pango]
                    mp_seqs_inherited[mut] = get(mp_seqs_inherited, mut, 0) + 1
                end
            end
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
    for (mut, co_mut_count) in mp_mut_ct
        mut_BAL_BALmut_tot = 0
        mut_BAL_ALLmut_tot = 0
        mut_BAL_prop_sum = 0
        mut_BAL_seq_tot = 0
        mut_BAL_prop_avg1 = 0  #### mut_BAL_prop_avg1——written to the CSV file——averages the proportion of BAL muts for each sequence
        mut_BAL_prop_avg2 = 0  #### mut_BAL_prop_avg2——NOT written to the CSV file——calculates BAL proportion by summing all the BAL muts & total muts from all BAL seqs
        mut_BAL_prop_avg_v3 = 0
        mut_BAL_seq_tot_avg = 0
################
        mut_EPCI_BAL_seq_tot = 0
        mut_EPCI_BAL_BALmut_tot_v1 = 0    ### All the v1 versions EXCLUDE the mutation under consideration from being a BAL mut
        mut_EPCI_BAL_ALLmut_tot_v1 = 0
        mut_EPCI_BAL_prop_sum_v1 = 0
        mut_EPCI_BAL_prop_avg_v1 = 0
        mut_EPCI_BAL_seq_tot_avg_v1 = 0
#############################             ### All the v2 versions INCLUDE the mutation under consideration as being a BAL mut
        mut_EPCI_BAL_BALmut_tot_v2 = 0
        mut_EPCI_BAL_ALLmut_tot_v2 = 0
        mut_EPCI_BAL_prop_sum_v2 = 0
        mut_EPCI_BAL_prop_avg_v2 = 0
        mut_EPCI_BAL_seq_tot_avg_v2 = 0
####### If a mutation was not used to select the current sequences, do the normal statistical calculations for it
        if !(mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct])
##########################################################################################
##########################################################################################
            for seq in AA_muts_seq_universal[mut]                                      #### AA_muts_seq_universal[mut] = all EPCI sequences with "mut" as a private AA mut
                prev_round_BAL_muts = intersect(seq_AA_muts_no_dels_universal[seq], pattern_mut_set_by_round_dict[mut_pattern_round_ct])
                tot_seq_muts_v1 = length(seq_AA_muts_no_dels_universal[seq])
                tot_seq_muts_v2 = length(seq_AA_muts_no_dels_universal[seq])
                tot_seq_BAL_muts_v1 = length(prev_round_BAL_muts)
                tot_seq_BAL_muts_v2 = length(prev_round_BAL_muts) + 1
                seq_BAL_prop_v1 = tot_seq_BAL_muts_v1/tot_seq_muts_v1
                seq_BAL_prop_v2 = tot_seq_BAL_muts_v2/tot_seq_muts_v2
##########################
                mut_EPCI_BAL_BALmut_tot_v1 += tot_seq_BAL_muts_v1
                mut_EPCI_BAL_BALmut_tot_v2 += tot_seq_BAL_muts_v2
                mut_EPCI_BAL_ALLmut_tot_v1 += tot_seq_muts_v1
                mut_EPCI_BAL_ALLmut_tot_v2 += tot_seq_muts_v2
                mut_EPCI_BAL_prop_sum_v1 += seq_BAL_prop_v1
                mut_EPCI_BAL_prop_sum_v2 += seq_BAL_prop_v2
##########################             
                mut_EPCI_BAL_seq_tot += 1
##########################
                if seq in mp_seqs
                    mut_BAL_BALmut_tot += tot_seq_BAL_muts_v1
                    mut_BAL_ALLmut_tot += tot_seq_muts_v1
                    mut_BAL_seq_tot += 1
                    mut_BAL_prop_sum += seq_BAL_prop_v1
#                    push!(mut_BAL_prop_vec, seq_BAL_prop)
#                    push!(mut_BAL_seq_tot_vec, tot_seq_BAL_muts)
                end
            end
##########################################################################################
##########################################################################################
##########################################################################################
            mut_BAL_prop_avg1       = mut_BAL_prop_sum/mut_BAL_seq_tot
            mut_BAL_prop_avg2       = mut_BAL_BALmut_tot/mut_BAL_ALLmut_tot
            mut_BAL_seq_tot_avg     = mut_BAL_BALmut_tot/mut_BAL_seq_tot
            mut_BAL_prop_avg1_rd    = round(digits=3, mut_BAL_prop_avg1)
            mut_BAL_prop_avg2_rd    = round(digits=3, mut_BAL_prop_avg2)
            mut_BAL_seq_tot_avg_rd  = round(digits=2, mut_BAL_seq_tot_avg)            
######################################
            mut_EPCI_BAL_prop_avg_v1       = mut_EPCI_BAL_prop_sum_v1/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_prop_avg_v2       = mut_EPCI_BAL_prop_sum_v2/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_prop_avg_v1_rd    = round(digits=3, mut_EPCI_BAL_prop_avg_v1)
            mut_EPCI_BAL_prop_avg_v2_rd    = round(digits=3, mut_EPCI_BAL_prop_avg_v2)
            mut_EPCI_BAL_seq_tot_avg_v1    = mut_EPCI_BAL_BALmut_tot_v1/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_seq_tot_avg_v2    = mut_EPCI_BAL_BALmut_tot_v2/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_seq_tot_avg_v1_rd = round(digits=2, mut_EPCI_BAL_seq_tot_avg_v1)
            mut_EPCI_BAL_seq_tot_avg_v2_rd = round(digits=2, mut_EPCI_BAL_seq_tot_avg_v2)
##########################################################################################
##########################################################################################
##########################################################################################
##########################################################################################
##########################################################################################
            mut_po = aa_gene_and_pos_comprehensive_dict[mut]
            unknown = get(mp_mut_ct_unknown, mut_po, 0)
            total_unknown = get(total_AApos_ct_unknown, mut_po, 0)
            inherited = 0
            all_chr_inherited = 0
            if sub_0__posonly_1 == 0
                inherited = get(mp_seqs_inherited, mut, 0)
                all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
            end
            adjusted_MP_seq_ct = MP_seq_ct - unknown - inherited                                        ### MP_seq_ct = Total # of seqs meeting criteria
            adjusted_EPCI_seq_ct = total_chronic_count_BAL - total_unknown - all_chr_inherited
            tot_EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]                                         ### tot_EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut] = total AA_mut_ct in all independent EPCI seqs/lineages
            EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                                     ### EPCI_prop = proportion of all mut in entire EPCI list that appear in current BAL seq list
            MP_prop = co_mut_count/adjusted_MP_seq_ct                                                      ### MP_prop = proportion of seqs in current seq list that contain the mut
            non_MP_prop = (tot_EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)  ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation 
            EPCI_pct = 100*EPCI_prop                                                                    ### AA_muts_ct_no_dels_universal[mut] = total AA_mut_ct in all independent EPCI seqs/lineages
            MP_pct = 100*MP_prop  
            non_MP_pct = 100*non_MP_prop
            EPCI_all_prop = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                      ### adjusted_EPCI_seq_ct = total # of independent EPCI seqs in (adjusted) EPCI list  
            MP_all_prop = MP_seq_ct/adjusted_EPCI_seq_ct                                                ### EPCI_all_prop = proportion of all seqs on EPCI list that have this mutation     
            fold_incr = MP_prop/non_MP_prop                                                             ### MP_all_prop = proportion of all seqs on EPCI list on the current seq list 
            fold_incr_rd = round(digits=2, fold_incr)
            observed = co_mut_count
            expected = EPCI_all_prop*adjusted_MP_seq_ct
            if observed ≥ expected
#                diff = observed-expected;    chi_squared = diff^2/expected;    chi_squared_rd = round(digits=2, chi_squared)
                fishexact_up_left = Int(round(co_mut_count))                                                                                                 ### seq with mut + meet search criteria (usually 2+ muts in list)
                fishexact_up_rt = max(Int(round(adjusted_MP_seq_ct - co_mut_count)), 0)                                                                      ### seq without mut + meet search criteria
                fishexact_down_left = max(Int(round(AA_muts_ct_no_dels_universal[mut] - co_mut_count)), 0)                                                   ### seq with mut + does not meet search criteria
                fishexact_down_rt = max(Int(round(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - MP_seq_ct + co_mut_count - total_unknown)), 0)  ### seq w/o mut + does not meet search criteria
                fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                pv_fish = pvalue(fish)
                log_pv_fish = -log10(pv_fish)
                log_pv_fish_rd = round(digits=3, log_pv_fish)
################# Chi-squared formula —— Straight from Claude.ai which says I was doing it all wrong before ########
                a = fishexact_up_left; b = fishexact_up_rt; c = fishexact_down_left; d = fishexact_down_rt
                n = a + b + c + d  # total sample size
                chi_2_top = n*(a*d - b*c)^2
                chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                chi_squared = 0.0
                chi_squared_rd = 0.0
                if chi_2_bottom > 0
                    chi_squared = chi_2_top/chi_2_bottom
                    expected_a = (a + b) * (a + c) / n
                    expected_b = (a + b) * (b + d) / n
                    expected_c = (c + d) * (a + c) / n
                    expected_d = (c + d) * (b + d) / n
                    if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                        corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                        chi_squared_yates = corrected_numerator/chi_2_bottom
                        chi_squared = chi_squared_yates
                        chi_squared_rd = round(digits=1, chi_squared)
                    end
                end
################# End Claude's Chi-squared formula calculations ################
                co_mut_count = Int(round(co_mut_count))
                props = (EPCI_pct, MP_pct, non_MP_pct, tot_EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg1, mut_BAL_seq_tot_avg, fold_incr, mut_EPCI_BAL_prop_avg_v1, mut_EPCI_BAL_seq_tot_avg_v1, mut_EPCI_BAL_prop_avg_v2, mut_EPCI_BAL_seq_tot_avg_v2)
                prop_dict[mut] = props
#     prop_dict =         {Float64,Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Union(String, Float64),Float64,Float64,Float64,Float64} }()
            end
#####################################################################################################################################
    ####### If a mutation WAS used to select the current seqs, we have to remove all sequences from the list that only made the
    ##      list due to the presence of this mutation. So a sequence needs to have 3 mutations instead of 2 (or 4 instead of 3, etc)
    ##      from the list in order to be used to calculate the stats for these mutations. Requires making a new sequence list, then
    ##      completely refilling the dictionaries from scratch *for each mutation*. :-(
#####################################################################################################################################           
#### Reminder: mp_seqs_1_mut = Set of sequences that would have qualified for list even if the given mutation is subtracted from each 
#           sequence's count. (Only for mutations that were in the previous round & therefore were used to select the current sequences.)
        elseif mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct] ## && haskey(mp_mut_ct, mut)
            mp_seqs_1_mut[mut] = Set{String}()    
            total_AA_mut_ct_nd_1_mut[mut] = 0
            for seq in mp_seqs
                seq_priv_AA_mut_total = length(seq_AA_muts_no_dels_universal[seq])
                seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq/seq_priv_AA_mut_total), 1.5)
                seq_factor_half = (seq_factor + 1)/2
                if seq_factor_OFF_HALF_ON__0_1_2 == 0
                    seq_factor = 1
                elseif seq_factor_OFF_HALF_ON__0_1_2 == 1
                    seq_factor = seq_factor_half
                end
                mp_seqs_mut_ct = 0
                raw_mp_seqs_mut_ct = 0
                for seq_mut in seq_AA_muts_no_dels_universal[seq]
                    if seq_mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct] && !(seq_mut == mut)
                        mp_seqs_mut_ct += 1*seq_factor
                        raw_mp_seqs_mut_ct += 1
                    end
                end
                if mp_seqs_mut_ct ≥ mp_ct_min # && raw_mp_seqs_mut_ct ≥ 2
                    push!(mp_seqs_1_mut[mut], seq)
                end 
            end
########################################################################
            for seq in AA_muts_seq_universal[mut]                                        #### AA_muts_seq_universal[mut] = all EPCI sequences with "mut" as a private AA mut
                prev_round_BAL_muts = intersect(seq_AA_muts_no_dels_universal[seq], pattern_mut_set_by_round_dict[mut_pattern_round_ct])
                tot_seq_BAL_muts_v1 = length(prev_round_BAL_muts) - 1             ## Subtracting one to exclude the mutation under consideration from being counted as a BAL mut      
                tot_seq_BAL_muts_v2 = length(prev_round_BAL_muts)
                tot_seq_muts_v1 = length(seq_AA_muts_no_dels_universal[seq]) - 1  ## Subtracting one from the denominator as well  
                tot_seq_muts_v2 = length(seq_AA_muts_no_dels_universal[seq])
                seq_BAL_prop_v1 = tot_seq_BAL_muts_v1/tot_seq_muts_v1
                seq_BAL_prop_v2 = tot_seq_BAL_muts_v2/tot_seq_muts_v2
##########################
                mut_EPCI_BAL_BALmut_tot_v1 += tot_seq_BAL_muts_v1
                mut_EPCI_BAL_BALmut_tot_v2 += tot_seq_BAL_muts_v2
                mut_EPCI_BAL_ALLmut_tot_v1 += tot_seq_muts_v1
                mut_EPCI_BAL_ALLmut_tot_v2 += tot_seq_muts_v2
                mut_EPCI_BAL_prop_sum_v1 += seq_BAL_prop_v1
                mut_EPCI_BAL_prop_sum_v2 += seq_BAL_prop_v2
##########################             
                mut_EPCI_BAL_seq_tot += 1
##########################
                if seq in mp_seqs_1_mut[mut]
                    mut_BAL_BALmut_tot += tot_seq_BAL_muts_v1
                    mut_BAL_ALLmut_tot += tot_seq_muts_v1
                    mut_BAL_seq_tot += 1
                    mut_BAL_prop_sum += seq_BAL_prop_v1
                end
            end
##########################################################################################
##########################################################################################
##########################################################################################
            mut_BAL_prop_avg1       = mut_BAL_prop_sum/mut_BAL_seq_tot
            mut_BAL_prop_avg2       = mut_BAL_BALmut_tot/mut_BAL_ALLmut_tot
            mut_BAL_seq_tot_avg     = mut_BAL_BALmut_tot/mut_BAL_seq_tot
            mut_BAL_prop_avg1_rd    = round(digits=3, mut_BAL_prop_avg1)
            mut_BAL_prop_avg2_rd    = round(digits=3, mut_BAL_prop_avg2)
            mut_BAL_seq_tot_avg_rd  = round(digits=2, mut_BAL_seq_tot_avg)            
######################################
            mut_EPCI_BAL_prop_avg_v1       = mut_EPCI_BAL_prop_sum_v1/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_prop_avg_v2       = mut_EPCI_BAL_prop_sum_v2/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_prop_avg_v1_rd    = round(digits=3, mut_EPCI_BAL_prop_avg_v1)
            mut_EPCI_BAL_prop_avg_v2_rd    = round(digits=3, mut_EPCI_BAL_prop_avg_v2)
            mut_EPCI_BAL_seq_tot_avg_v1    = mut_EPCI_BAL_BALmut_tot_v1/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_seq_tot_avg_v2    = mut_EPCI_BAL_BALmut_tot_v2/mut_EPCI_BAL_seq_tot
            mut_EPCI_BAL_seq_tot_avg_v1_rd = round(digits=2, mut_EPCI_BAL_seq_tot_avg_v1)
            mut_EPCI_BAL_seq_tot_avg_v2_rd = round(digits=2, mut_EPCI_BAL_seq_tot_avg_v2)
##########################################################################################
##########################################################################################
##########################################################################################
            for seq in mp_seqs_1_mut[mut]
                seq_priv_AA_mut_total = length(seq_AA_muts_no_dels_universal[seq])
                if mut in seq_AA_muts_no_dels_universal[seq]
                    mp_mut_ct_1_mut[mut] = get(mp_mut_ct_1_mut, mut, 0) + 1
                end
                total_AA_mut_ct_nd_1_mut[mut] += seq_priv_AA_mut_total
            end
##################################
            total_seq_count_1_mut = length(mp_seqs_1_mut[mut])                                         ### total_seq_count_1_mut = Total # of seqs meeting inclusion criteria when NOT counting current mut
            avg_mutations_per_seq_1_mut = total_AA_mut_ct_nd_1_mut[mut]/total_seq_count_1_mut
            mut_po = aa_gene_and_pos_comprehensive_dict[mut]
            for seq in mp_seqs_1_mut[mut]
                if mut_po in seq_unknown_AA[seq]
                    mp_mut_ct_unknown_1_mut[mut_po] = get(mp_mut_ct_unknown_1_mut, mut_po, 0) + 1
                end
            end
            co_mut_count = get(mp_mut_ct_1_mut, mut, 0)
            unknown = get(mp_mut_ct_unknown_1_mut, mut_po, 0)
            total_unknown = get(total_AApos_ct_unknown, mut_po, 0)
            inherited = 0
            all_chr_inherited = 0
            if sub_0__posonly_1 == 0
                inherited = get(mp_seqs_inherited, mut, 0)
                all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
            end
            adjusted_MP_seq_ct = MP_seq_ct - unknown - inherited                                       ### tot_EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut] = total AA_mut_ct in all independent EPCI seqs/lineages
            adjusted_EPCI_seq_ct = total_chronic_count_BAL - total_unknown - all_chr_inherited         ### EPCI_prop = proportion of all mut in EPCI list that appear in current BAL seq list
            tot_EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]                                        ### MP_prop = proportion of seqs in current seq list that contain the mut
            EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                                 ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation
            MP_prop = co_mut_count/adjusted_MP_seq_ct                                                  ### AA_muts_ct_no_dels_universal[mut] = total AA_mut_ct in all independent EPCI seqs/lineages
            non_MP_prop = (tot_EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct) ### adjusted_EPCI_seq_ct = total # of independent (slightly adjusted) EPCI seqs in EPCIs list
            EPCI_pct = 100*EPCI_prop                                                                   ### EPCI_all_prop = proportion of all seqs on EPCIs list that have this mutation
            MP_pct = 100*MP_prop                                                                       ### MP_all_prop = proportion of all seqs on EPCIs list on the current seq list
            non_MP_pct = 100*non_MP_prop                                                 
            EPCI_all_prop = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                     
            MP_all_prop = total_seq_count_1_mut/adjusted_EPCI_seq_ct                                                                                         
            fold_incr::Union{Float64,String} = 0.0
            if non_MP_prop ≠ 0
                fold_incr = MP_prop/non_MP_prop
            elseif non_MP_prop == 0
                fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
                fake_fold_incr = MP_prop/fake_non_MP_prop
                fake_fold_incr_rd = round(digits=2, fake_fold_incr)
                fold_incr = ">$(fake_fold_incr_rd)"
            end
            observed = co_mut_count
            expected = EPCI_all_prop*total_seq_count_1_mut
            if observed ≥ expected
#               diff = observed-expected;     chi_squared = diff^2/expected;      chi_squared_rd = round(digits=2, chi_squared)
                fishexact_up_left = Int(round(co_mut_count))                                                                                                                ### seq with mut + meet search criteria (usually 2+ muts in list)
                fishexact_up_rt = max(Int(round(adjusted_MP_seq_ct - co_mut_count)), 0)                                                                                        ### seq without mut + meet search criteria
                fishexact_down_left = max(Int(round(AA_muts_ct_no_dels_universal[mut] - co_mut_count)), 0)                                                                  ### seq with mut + does not meet search criteria 
                fishexact_down_rt = max(Int(round(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - total_seq_count_1_mut + co_mut_count - total_unknown)), 0)  ### seq w/o mut + does not meet search criteria
                fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                pv_fish = pvalue(fish)
                log_pv_fish = -log10(pv_fish)
                log_pv_fish_rd = round(digits=3, log_pv_fish)
################# Chi-squared formula —— Straight from Claude.ai which says I was doing it all wrong before ########
                a = fishexact_up_left; b = fishexact_up_rt; c = fishexact_down_left; d = fishexact_down_rt
                n = a + b + c + d  # total sample size
                chi_2_top = n*(a*d - b*c)^2
                chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                chi_squared = 0.0
                chi_squared_rd = 0.0
                chi_squared_pvalue = 1.0
                if chi_2_bottom > 0
                    chi_squared = chi_2_top/chi_2_bottom
                    expected_a = (a + b) * (a + c) / n
                    expected_b = (a + b) * (b + d) / n
                    expected_c = (c + d) * (a + c) / n
                    expected_d = (c + d) * (b + d) / n
                    if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                        corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                        chi_squared_yates = corrected_numerator/chi_2_bottom
                        chi_squared = chi_squared_yates
                        chi_squared_rd = round(digits=1, chi_squared)
                    end
                end
################# End Claude's Chi-squared formula calculations ################
                co_mut_count = Int(round(co_mut_count))
#                            1       2          3             4                5               6               7          8          9              10                  11               12                13                        14                           15                        16 
#               props = (EPCI_pct, MP_pct, non_MP_pct, tot_EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg1, mut_BAL_seq_tot_avg, fold_incr, mut_EPCI_BAL_prop_avg_v1, mut_EPCI_BAL_seq_tot_avg_v1, mut_EPCI_BAL_prop_avg_v2, mut_EPCI_BAL_seq_tot_avg_v2)  
                props = (EPCI_pct, MP_pct, non_MP_pct, tot_EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg1, mut_BAL_seq_tot_avg, fold_incr, mut_EPCI_BAL_prop_avg_v1, mut_EPCI_BAL_seq_tot_avg_v1, mut_EPCI_BAL_prop_avg_v2, mut_EPCI_BAL_seq_tot_avg_v2)  
                prop_dict[mut] = props
            end
        end 
    end
#    for mut in keyless_muts
#        if !haskey(prop_dict, mut)
#            prop_dict[mut] = (0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
#        end
#    end
######################################################################################################################################
##### PROPS KEY: props[1]=qry_pct | [2]=MP_prop | [3]=EPCI_mut_ct | [4]=qual_mut_ct | [5]=qual_seq_ct | [6]=Chi^2 | [7]=log10_pv_fish | [8]=mut_BAL_prop_avg | [9]=mut_BAL_seq_tot_avg
    new_pattern_mut_set = Set{String}()
    new_pattern_seq_set = Set{String}()
    non_search_muts = Set([])
    ##### This collects all the muts lost in the past 5 rounds. Used to avoid gaining & losing the same mut every other round.
    muts_lost_in_last_4_rounds = Set{String}()
    for i in mut_pattern_round_ct-3:mut_pattern_round_ct
        if i ≥ 1
            lost_mut_set = lost_mut_by_round_dict[i]
            for mut in lost_mut_set
                push!(muts_lost_in_last_4_rounds, mut)
            end
        end
    end
##########################################################################################################################################################################
# mut_to_mutpos_surround_int(mut::String, plus_or_minus::Int) returns set of mut_pos within plus_or_minus of mut. 
#### Purpose of the plus_or_minus parameter is to include nearby muts that are likely to have a similar effect & which cluster
#    with a qualifying mutation. For example, if S:N657K qualifies and 70% of all other muts at S:657 & S:659 are in sequences that
#    possess the mutation pattern, this seems very unlikely to be a coincidence. So even if those other mutations at S:657/659 do not
#    qualify individually, they would qualify due to being part of a cluster of clearly related mutations. last_rd_mut_pos_ct_plus_minus
#### If a mutation is within `plus_or_minus` AA positions of a current BAL mut, the requirements for qualifying as a BAL mut are slightly loosened.
#    The fact that a mutation was in the previous round's BAL muts should not affect its requirements, so most of the time only the surrounding AA positions
#    are included in this range. However, if a mutation is at the same position as a current BAL mut but is a different residue, it should get the "discounted"
#    requirements, which is the reason for `last_rd_mut_pos_ct_plus_minus` dictionary, which counts the number of BAL muts from the previous round 
#    that occur within `plus_or_minus` AA positions of the mutation in question.
#    If the mutation was not in the BAL muts from the previous round and it is located in `close_mut_positions`, it gets the discounted requirements.
#    But if it *was* in the BAL muts from the previous round, it only gets the discounted rate if there was at least two BAL mutations (itself and one other) 
#    within `plus_or_minus` AA positions of it. 
    close_mut_positions = Set{String}()
    last_rd_mut_pos_ct_dict = Dict{String, Int}()
    last_rd_mut_pos_ct_plus_minus = Dict{String, Int}()
########################
    for mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct]
        mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
        last_rd_mut_pos_ct_dict[mut_pos] = get(last_rd_mut_pos_ct_dict, mut_pos, 0) + 1
    end
########################
    for mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct]
        mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
        mut_aa_gene = aa_gene_comprehensive_dict[mut]
        mut_aa_pos = aa_pos_comprehensive_dict[mut]
        for i in -plus_or_minus:plus_or_minus
            pos = mut_aa_pos + i
            gene_pos_near = "$(mut_aa_gene):$(pos)"
            gene_pos_near_ct = get(last_rd_mut_pos_ct_dict, gene_pos_near, 0)
            last_rd_mut_pos_ct_plus_minus[mut_pos] = get(last_rd_mut_pos_ct_plus_minus, mut_pos, 0) + gene_pos_near_ct
        end
    end
########################
    for mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct]
        mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
        close_mut_po_vec = mut_to_mutpos_surround_int(mut, plus_or_minus)
        for mut_po in close_mut_po_vec
            if sub_0__posonly_1 == 0
                push!(close_mut_positions, mut_po)
            elseif sub_0__posonly_1 == 1
                if mut_po ≠ mut_pos
                    push!(close_mut_positions, mut_po)
                end
            end 
        end
    end
################################################################################################################
    print_ct = 0
#    muts2chk = ["ORF9b:1", "S:1175", "S:1182", "S:1185", "ORF1a:2967", "S:405", "ORF1b:1957", "ORF1a:3946", "ORF1a:4391", "ORF1a:376", "ORF1a:2193", "ORF7a:122"]    #  , "S:N354D", "S:T274I"
#    muts2chk = ["S:S1175P", "S:S1175L", "S:I1179S", "S:E1182G", "S:R1185S", "ORF1a:T2967I", "S:N354D", "ORF1a:T4175I", "S:N405D", "ORF1b:K1957R", "ORF1a:S3946L", "ORF1a:L4391F", "ORF1a:S376P", "ORF1a:S2193F"]    #  , "S:N354D", "S:T274I"
#    muts2chk = ["ORF9b:1", "S:405", "ORF7a:122", "N:417", "ORF1a:4205", "ORF1a:4207"]    #  , "S:N354D", "S:T274I"
     muts2chk = ["S:N405D", "S:S1175P", "ORF7a:*122R", "ORF7a:*122W", "ORF7a:*122-", "ORF1a:T4175I", "ORF1a:R287G", "E:L27S", "E:T9I", "M:C64S", "ORF1a:A3697V"]   
    for (mut, props) in prop_dict
        if !(mut in muts_lost_in_last_4_rounds)
##########################################################################################################################################################################
#              1       2          3             4               5               6                7          8          9              10                  11               12                13                        14                           15                        16 
# props = (EPCI_pct, MP_pct, non_MP_pct, tot_EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg1, mut_BAL_seq_tot_avg, fold_incr, mut_EPCI_BAL_prop_avg_v1, mut_EPCI_BAL_seq_tot_avg_v1, mut_EPCI_BAL_prop_avg_v2, mut_EPCI_BAL_seq_tot_avg_v2)  
            MP_ct = props[5]
            min_AA_ct_temp = min_AA_ct
            minQryPct_temp = minQryPct
            min_BAL_prop_temp = min_BAL_prop
            min_log_pv_fish_temp = min_log_pv_fish
            mut_po = aa_gene_and_pos_comprehensive_dict[mut]
            if mut_po in close_mut_positions && !(mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct])
#                min_AA_ct_temp = min_AA_ct - 1
                minQryPct_temp = minQryPct - 2.5
                min_BAL_prop_temp = min_BAL_prop - 0.01
                min_log_pv_fish_temp = min_log_pv_fish_temp - 0.5
            elseif mut_po in close_mut_positions && last_rd_mut_pos_ct_plus_minus[mut_po] ≥ 2
#                min_AA_ct_temp = min_AA_ct - 1
                minQryPct_temp = minQryPct - 2.5
                min_BAL_prop_temp = min_BAL_prop - 0.01
                min_log_pv_fish_temp = min_log_pv_fish_temp - 0.5
            end
#            if mut_pattern_round_ct < 30
#                min_BAL_prop_temp = 0.0
#            end
######## Recursive functions like these have a tendency to stagnate at low numbers or explode.
###      By incorporating the current total number of mutation into the `min_BAL_prop_temp` and `minQryPct_temp` requirements, I'm trying to temper this tendency 
###      & make it easier to make small adjustments that don't lead to ruin. Also adjusts min_BAL_prop & minQryPct required for a mut to qualify based on the total
###      # of muts currently on the list—a necessity early on when there are very few muts on the BAL list & few EPCI sequences have two of the 13 starting BAL muts. 
            max_muts_point = 85
            if sub_0__posonly_1 == 1
                max_muts_point = 75
            end  
            min_BAL_prop_temp = min_BAL_prop_temp - ((max_muts_point - last_rd_BAL_ct)*0.001)
            minQryPct_temp = minQryPct_temp - ((max_muts_point - last_rd_BAL_ct)*0.15)
##############################################################################################################################
                                if mut in muts2chk
                                    print_ct += 1
                                    if print_ct ≤ 1
                                        println("Rd $(mut_pattern_round_ct): min_fish = $(round(digits=2, min_log_pv_fish))|minPct = $(round(digits=2, minQryPct_temp))%|minProp = $(round(digits=2, min_BAL_prop_temp))")
                                    end
fsh_rd=round(digits=2,props[9]); pct_rd=round(digits=1,100*props[1]); BALprop_rd=round(digits=3,props[10]); BALtot_rd=round(digits=2,props[11]); EPCIprop_rd_v1=round(digits=3,props[13]); EPCItot_rd_v1=round(digits=2,props[14]); EPCIprop_rd_v2=round(digits=3,props[15]); EPCItot_rd_v2=round(digits=2,props[16]); EPCIseq_pad=lpad(AA_muts_ct_no_dels_universal[mut], 3)
mt_pad=rpad(mut,12); fshpad=lpad(fsh_rd,5); pctPad=lpad(pct_rd,4); EPCI_Mut_ct_pad=lpad(props[4],3); MPmutpad=lpad(props[5],2); MPseqpad=lpad(props[6],3); BALpropPad=lpad(BALprop_rd,5); BALtotpad=lpad(BALtot_rd,4); EPCIpropPad_v1=lpad(EPCIprop_rd_v1,5); EPCItotpad_v1=lpad(EPCItot_rd_v1,4); EPCIpropPad_v2=lpad(EPCIprop_rd_v2,5); EPCItotpad_v2=lpad(EPCItot_rd_v2,4) 
                                    println("$(mt_pad)|logpvFish=$(fshpad)|EPCI%=$(pctPad)|totChr=$(EPCI_Mut_ct_pad)|MPmut=$(MPmutpad)|qualSeq=$(MPseqpad)|EPCIct=$(EPCIseq_pad)|BALprop=$(BALpropPad)|BALtot=$(BALtotpad)|EPCIpropv1=$(EPCIpropPad_v1)|EPCItotv1=$(EPCItotpad_v1)|EPCIpropv2=$(EPCIpropPad_v2)|EPCItotv2=$(EPCItotpad_v2)|")
                                end
##########################################################################################################################################################################
#            if mut_pattern_round_ct ≥ 30
            log10PVfish = props[9]
            EPCI_prop = props[1]
            EPCI_pct = 100*EPCI_prop
            BAL_prop = props[10]
            EPCI_BAL_prop_v1 = props[13]
            EPCI_BAL_prop_v2 = props[15]
            if log10PVfish ≥ min_log_pv_fish_temp && AA_muts_ct_no_dels_universal[mut] ≥ min_AA_ct && EPCI_pct ≥ minQryPct_temp && EPCI_BAL_prop_v2 ≥ min_BAL_prop_temp   
#                if 2*log10PVfish + EPCI_pct > minQryPct_temp + 25
                push!(new_pattern_mut_set, mut)
                if !(mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct])
                    push!(get!(new_mut_by_round_dict, mut_pattern_round_ct+1, Set{String}()), mut)
                end
#                end
#                elseif props[10] ≥ min_BAL_prop_temp + 0.05 && props[5] ≥ 5.0 && AA_muts_ct_no_dels_universal[mut] ≥ min_AA_ct_temp && props[1] ≥ minQryPct_temp && props[9] ≥ min_log_pv_fish_temp-0.5
#                    println("$(mut) qualifies by alternate BAL_prop ≥ 0.25 + BAL_mut_tot ≥ 5.0)")
#                    println("log_pv_fish = $(props[9]) |totChr=$(props[4])|qualMut=$(props[5])|qualSeq=$(props[6])|BALprop=$(props[10])|BALtot=$(props[11])|")
#                    println("#####################################################################################")
            end
#            end
#            if mut_pattern_round_ct < 30
#                log10PVfish = props[9]
#                EPCI_prop = props[1]
#                EPCI_pct = 100*EPCI_prop
#                BAL_prop = props[10]
#                if log10PVfish ≥ min_log_pv_fish_temp && AA_muts_ct_no_dels_universal[mut] ≥ min_AA_ct && EPCI_pct > minQryPct_temp && BAL_prop ≥ min_BAL_prop_temp-0.03                   
#                    push!(new_pattern_mut_set, mut)
#                    if !(mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct])
#                        push!(get!(new_mut_by_round_dict, mut_pattern_round_ct+1, Set{String}()), mut)
#                    end
#                    elseif props[10] ≥ min_BAL_prop_temp + 0.05 && props[5] ≥ 5.0 && AA_muts_ct_no_dels_universal[mut] ≥ min_AA_ct_temp && props[1] ≥ minQryPct_temp && props[9] ≥ min_log_pv_fish_temp-0.5
#                        println("$(mut) qualifies by alternate BAL_prop ≥ 0.25 + BAL_mut_tot ≥ 5.0)")
#                        println("log_pv_fish = $(props[9]) |totChr=$(props[4])|qualMut=$(props[5])|qualSeq=$(props[6])|BALprop=$(props[10])|BALtot=$(props[11])|")
#                        println("#####################################################################################")
#                end
#            end
        end
    end
#### If the mutation appeared ≥4 times in the confirmed BAL-sample sequences, it automatically qualifies during the first 10 rounds
    if mut_pattern_round_ct ≤ 30
        for mut in BAL_muts_start
            push!(new_pattern_mut_set, mut)
        end
    end
###########################################################################################################################################################################
################################ This is where the mp_seqs for the next round are determined ##############################################################################
###########################################################################################################################################################################
    muts_len = length(new_pattern_mut_set)
    for seq in all_unique_chr_seqs_set
        seq_priv_AA_mut_total = length(seq_AA_muts_no_dels_universal[seq])
        seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq/seq_priv_AA_mut_total), 1.5)
        seq_factor_half = (seq_factor + 1)/2
        if seq_factor_OFF_HALF_ON__0_1_2 == 0
            seq_factor = 1
        elseif seq_factor_OFF_HALF_ON__0_1_2 == 1
            seq_factor = seq_factor_half
        end
        mp_ct = 0
        raw_mp_ct = 0
        for mut in new_pattern_mut_set
            if mut in seq_AA_muts_no_dels_universal[seq]
                mp_ct += 1*seq_factor
                raw_mp_ct += 1
            end
        end
###################################         
        mut_total = length(seq_AA_muts_no_dels_universal[seq])
        mp_seq_mut_prop = (mp_ct/seq_factor)/mut_total
        if mp_ct ≥ mp_ct_min && raw_mp_ct ≥ 2
            push!(new_pattern_seq_set, seq)
        end
    end
    total_qualifying_BAL_seqs = length(new_pattern_seq_set)
###########################################################################################################################################################################
########################################## End of mp_seq qualifying section ###############################################################################################
###########################################################################################################################################################################
    for seq in BAL_uniq_seq_set
        bal_muts_share = intersect(new_pattern_mut_set, seq_AA_muts_no_dels_universal[seq])
        bal_muts_total = length(bal_muts_share)
        push!(new_pattern_seq_set, seq)
    end
    pattern_seq_set_len = length(new_pattern_seq_set)
    pattern_mut_set_len = length(new_pattern_mut_set)
######################
    mut_lost_gained = "mut_pattern_$(mut_pattern_name)_$(pos_only_option)__muts_lost_or_gained_by_round__$(date_hour).txt"
######################
    local_mut_pattern_round_ct = mut_pattern_round_ct
    mut_pattern_round_ct += 1
    local_mut_pattern_round_ct += 1
    muts_lost_total = 0
    muts_gained_total = 0
    if haskey(new_mut_by_round_dict, mut_pattern_round_ct)
        muts_gained_total = length(new_mut_by_round_dict[mut_pattern_round_ct])
    end
######################################
    pattern_mut_set_by_round_dict[mut_pattern_round_ct] = new_pattern_mut_set
    pattern_mut_ct_by_round_dict[mut_pattern_round_ct] = pattern_mut_set_len
    lost_mut_by_round_dict[mut_pattern_round_ct] = Vector{String}()
    reason_lost_by_round[mut_pattern_round_ct] = Vector{String}()
######################################
    open(mut_file, "a") do g
        println(g, "#########################################  Mutations Lost in Round $(mut_pattern_round_ct)  ##########################################")
        for mut in pattern_mut_set_by_round_dict[mut_pattern_round_ct-1]
            if !(mut in new_pattern_mut_set) && haskey(prop_dict, mut)
                muts_lost_total += 1
                print(g, mut, ", ")
                push!(lost_mut_by_round_dict[mut_pattern_round_ct], mut)
                print(g, " | Reason: ")
#                            1       2          3            4              5              6                7          8          9              10                  11               12                13                        14                           15                        16 
#               props = (EPCI_pct, MP_pct, non_MP_pct, tot_EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, mut_BAL_prop_avg1, mut_BAL_seq_tot_avg, fold_incr, mut_EPCI_BAL_prop_avg_v1, mut_EPCI_BAL_seq_tot_avg_v1, mut_EPCI_BAL_prop_avg_v2, mut_EPCI_BAL_seq_tot_avg_v2)
                if prop_dict[mut][9] < min_log_pv_fish
                    push!(reason_lost_by_round[mut_pattern_round_ct], "min_log_pv_fish")
                    print(g, " logPVfish $(min_log_pv_fish) ")
                end
                if AA_muts_ct_no_dels_universal[mut] < min_AA_ct
                    push!(reason_lost_by_round[mut_pattern_round_ct], "min_AA_ct")
                    print(g, " AA_muts_ct_no_dels_universal[mut]|$(min_AA_ct)|")
                end
                if prop_dict[mut][13] < min_BAL_prop
                    push!(reason_lost_by_round[mut_pattern_round_ct], "min_BAL_prop")
                    print(g, " min_BAL_prop|$(min_BAL_prop)|")
                end
                        
            end
        end
        print(g, "\n"^1)
        println(g, "########################################  Mutations Gained in Round $(mut_pattern_round_ct)  #########################################")
        if haskey(new_mut_by_round_dict, mut_pattern_round_ct)
            for mut in new_mut_by_round_dict[mut_pattern_round_ct]
                print(g, mut, ", ")
            end
        end
        print(g, "\n"^1)
####################################################################################################################################
        if pattern_mut_ct_by_round_dict[mut_pattern_round_ct] ≠ pattern_mut_ct_by_round_dict[mut_pattern_round_ct-1]
            println("               Total AA Subs in $(mut_pattern_name), Round $(mut_pattern_round_ct) = $(pattern_mut_set_len) | Total seqs = $(pattern_seq_set_len)")
        end
        println(g, "################################################################################################################")
        println(g, "          Total AA Subs in $(mut_pattern_name), Round $(mut_pattern_round_ct) = $(pattern_mut_set_len)")
        println(g, "################################################################################################################")
    end
    pattern_mut_set_sort = sort(collect(new_pattern_mut_set), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
######################################################################################################################################
    open(mut_file, "a") do g
        min_chi_sq_rd = round(digits=2, min_chi_sq)
        avg_AA_sub_ct_per_chronic_seq_rd = round(digits=2, avg_AA_sub_ct_per_chronic_seq)
        println(g, "               Number of $(mut_pattern_name) Sequences Last Round = $(length(mp_seqs))" )
        println(g, "               Number of $(mut_pattern_name) Sequences This Round = $(pattern_seq_set_len)")
        println(g, "               Average Number of AA Subs per Seq = $(avg_mutations_per_seq_rd)" )
        println(g, "               Average Number of AA Subs per Chronic Sequence in Entire List = $(avg_AA_sub_ct_per_chronic_seq_rd)" )
        println(g, "               Ratio of (Avg AA Subs per Sequence in Group)/(Avg AA Subs per Sequence in All Chronics) = $(ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd)")
        println(g, "################################################################################################################")
        println(g, "  min -log10fish = $(min_log_pv_fish) | min_chi^2 = $(min_chi_sq_rd) | min_mut_ct = $(mp_ct_min) |") 
        println(g, "################################################################################################################")
        println(g, "                            |   |   |     |      |      |        |       |       |      |")
        println(g, "               Mutation     |MCt|TCt|SeqCt| Qry% | Grp% |logPVfsh| Chi2  |avgProp|avgBAL|")
        println(g, "                            |   |   |     |      |      |        |       |       |      |")
        for mut in pattern_mut_set_sort
            if mut in keys(prop_dict)
                EPCI_pct      = prop_dict[mut][1]
                MP_pct        = prop_dict[mut][2]
                non_MP_pct    = prop_dict[mut][3]
                EPCI_mut_ct   = prop_dict[mut][4]
                BAL_Mut_ct    = prop_dict[mut][5]
         Adjusted_BAL_Seq_ct  = prop_dict[mut][6]
                Chi2          = prop_dict[mut][7]
                pv_fish       = prop_dict[mut][8]
                log10pvFISH   = prop_dict[mut][9]
             MP_BAL_avg_prop  = prop_dict[mut][10]
             MP_BAL_mut_avg   = prop_dict[mut][11]
                fold_incr     = prop_dict[mut][12]
         EPCI_BAL_avg_prop_v1 = prop_dict[mut][13]
         EPCI_BAL_mut_avg_v1  = prop_dict[mut][14]
         EPCI_BAL_avg_prop_v2 = prop_dict[mut][15]
         EPCI_BAL_mut_avg_v2  = prop_dict[mut][16]
###########################################
                Chi2_rd = round(digits=2, Chi2)
                log10pvFISH_rd = round(digits=2, log10pvFISH)
                Chi2_string = string(Chi2_rd)
                Chi2_post_decimal = split(Chi2_string, ".")[2]
                if length(Chi2_post_decimal) == 1
                    Chi2_string = Chi2_string*"0"
                end
                MP_BAL_avg_prop_rd = round(digits=2, MP_BAL_avg_prop)
                MP_BAL_avg_prop_pad = rpad(MP_BAL_avg_prop_rd, 5)
                MP_BAL_mut_avg_rd = round(digits=2, MP_BAL_mut_avg)
                MP_BAL_mut_avg_pad = rpad(MP_BAL_mut_avg_rd, 4)
                EPCI_BAL_avg_prop_v1_rd = round(digits=2, EPCI_BAL_avg_prop_v1)
                EPCI_BAL_avg_prop_v1_pad = rpad(EPCI_BAL_avg_prop_v1_rd, 5)
                EPCI_BAL_avg_prop_v2_rd = round(digits=2, EPCI_BAL_avg_prop_v2)
                EPCI_BAL_avg_prop_v2_pad = rpad(EPCI_BAL_avg_prop_v2_rd, 5)
                EPCI_BAL_mut_avg_v1_rd = round(digits=2, EPCI_BAL_mut_avg_v1)
                EPCI_BAL_mut_avg_v1_pad = rpad(EPCI_BAL_mut_avg_v1_rd, 4)
                EPCI_BAL_mut_avg_v2_rd = round(digits=2, EPCI_BAL_mut_avg_v2)
                EPCI_BAL_mut_avg_v2_pad = rpad(EPCI_BAL_mut_avg_v2_rd, 4)
                mutpad = rpad(mut, 13)
                BAL_Mut_ct_pad = lpad(BAL_Mut_ct, 3)
                EPCI_mut_ct_pad = lpad(EPCI_mut_ct, 3)
                Adjusted_BAL_Seq_ct_pad = lpad(Adjusted_BAL_Seq_ct, 5)
                EPCI_pct_rd = round(digits=2, EPCI_pct)
                EPCI_pct_pad = lpad(EPCI_pct_rd, 5)
                MP_pct_rd = round(digits=2, MP_pct)
                MP_pct_pad = lpad(MP_pct_rd, 5)
                log10pvFISH_rd2 = add_zeros_to_rounded(log10pvFISH_rd, 2)
                log10pvFISH_rd_pad = lpad(log10pvFISH_rd2, 7)
                Chi2_pad = lpad(Chi2_string, 6)
                df_props_dict[mut_pattern_round_ct] = Dict{String, Tuple{Float64,Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Union{Float64,String} } }()
                df_props_dict[mut_pattern_round_ct][mut] = (EPCI_pct, MP_pct, non_MP_pct, EPCI_mut_ct, BAL_Mut_ct, Adjusted_BAL_Seq_ct, Chi2, pv_fish, log10pvFISH, MP_BAL_avg_prop, MP_BAL_mut_avg, EPCI_BAL_avg_prop_v1, EPCI_BAL_mut_avg_v1, EPCI_BAL_avg_prop_v2, EPCI_BAL_mut_avg_v2, fold_incr)
                println(g, "               $(mutpad)|$(BAL_Mut_ct_pad)|$(EPCI_mut_ct_pad)|$(Adjusted_BAL_Seq_ct_pad)|$(EPCI_pct_pad)%|$(MP_pct_pad)%|$(log10pvFISH_rd_pad) | $(Chi2_pad)| $(EPCI_BAL_avg_prop_v1_pad) | $(EPCI_BAL_mut_avg_v1_pad)  | $(EPCI_BAL_avg_prop_v2_pad) | $(EPCI_BAL_mut_avg_v2_pad)  |")
            end
        end
    end
#    println("Total Number of Mutations in $(mut_pattern_name) = $(pattern_mut_set_len)")
######################################################################################################################################
### If more than 3 mutations were added in previous round, keep minimum Chi^2 threshhold the same.
    new_min_chi_sq = max(abs_chi2_min, min_chi_sq) 
######################################################################################################################################
### The minimum required -log10 Fisher's Exact test p-value starts at min_log_pv_fish and gradually decreases. The speed of the 
#   decrease depends on the number of new mutations added in the previous round.
######################################################################################################################################
    new_min_log_pv_fish = max(min_log_pv_fish, abs_min_fish)
    if muts_gained_total ≤ 1
        new_min_log_pv_fish = max(min_log_pv_fish-0.2, abs_min_fish)
    elseif muts_gained_total ≤ 3
        new_min_log_pv_fish = max(min_log_pv_fish-0.1, abs_min_fish)
    end
########################################################
    new_mp_ct_min = mp_ct_min
########################################################
    if local_mut_pattern_round_ct < tot_rounds
#                                                  1               2           3            4            5             6              7                  8                 9               10             11              12              13                       14
        AA_mut_pattern_iterative_2025_11_29(mut_pattern_name, minQryPct, min_BAL_prop, tot_rounds, plus_or_minus, min_AA_ct, new_pattern_seq_set, new_mp_ct_min, new_min_log_pv_fish, abs_min_fish, new_min_chi_sq, abs_chi2_min, min_chr_all_ratio, seq_factor_OFF_HALF_ON__0_1_2)
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
    end 
    if local_mut_pattern_round_ct == tot_rounds
        all_mut_file = "$(folder_name)/$(folder_name)__ALL_STATS_final_round.txt"
        open(all_mut_file, "w") do g
            min_chi_sq_rd = round(digits=2, min_chi_sq)
            avg_AA_sub_ct_per_chronic_seq_rd = round(digits=2, avg_AA_sub_ct_per_chronic_seq)
            println(g, "               Number of $(mut_pattern_name) Sequences Last Round = $(length(mp_seqs))" )
            println(g, "               Number of $(mut_pattern_name) Sequences This Round = $(pattern_seq_set_len)")
            println(g, "               Average Number of AA Subs per Seq = $(avg_mutations_per_seq_rd)" )
            println(g, "               Average Number of AA Subs per Chronic Sequence in Entire List = $(avg_AA_sub_ct_per_chronic_seq_rd)" )
            println(g, "               Ratio of (Avg AA Subs per Sequence in Group)/(Avg AA Subs per Sequence in All Chronics) = $(ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd)")
            println(g, "################################################################################################################")
            println(g, "  min -log10fish = $(min_log_pv_fish) | min_chi^2 = $(min_chi_sq_rd) | min_mut_ct = $(mp_ct_min) |") 
            println(g, "################################################################################################################")
            println(g, "                            |   |   |     |      |      |        |       |       |      |")
            println(g, "               Mutation     |MCt|TCt|SeqCt| Qry% | Grp% |logPVfsh| Chi2  |avgProp|avgBAL|")
            println(g, "                            |   |   |     |      |      |        |       |       |      |")
            prop_keys = keys(prop_dict)
            prop_keys_sort = sort(collect(prop_keys), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
            for mut in prop_keys_sort
                if mut in new_pattern_mut_set
                    EPCI_pct      = prop_dict[mut][1]
                    MP_pct        = prop_dict[mut][2]
                    non_MP_pct    = prop_dict[mut][3]
                    EPCI_mut_ct   = prop_dict[mut][4]
                    BAL_Mut_ct    = prop_dict[mut][5]
             Adjusted_BAL_Seq_ct  = prop_dict[mut][6]
                    Chi2          = prop_dict[mut][7]
                    pv_fish       = prop_dict[mut][8]
                    log10pvFISH   = prop_dict[mut][9]
                 MP_BAL_avg_prop  = prop_dict[mut][10]
                 MP_BAL_mut_avg   = prop_dict[mut][11]
                    fold_incr     = prop_dict[mut][12]
             EPCI_BAL_avg_prop_v1 = prop_dict[mut][13]
             EPCI_BAL_mut_avg_v1  = prop_dict[mut][14]
             EPCI_BAL_avg_prop_v2 = prop_dict[mut][15]
             EPCI_BAL_mut_avg_v2  = prop_dict[mut][16]
######################################
                    Chi2_rd = round(digits=2, Chi2)
                    log10pvFISH_rd = round(digits=2, log10pvFISH)
                    Chi2_string = string(Chi2_rd)
                    Chi2_post_decimal = split(Chi2_string, ".")[2]
                    if length(Chi2_post_decimal) == 1
                        Chi2_string = Chi2_string*"0"
                    end
                    MP_BAL_avg_prop_rd = round(digits=2, MP_BAL_avg_prop)
                    MP_BAL_avg_prop_pad = rpad(MP_BAL_avg_prop_rd, 5)
                    MP_BAL_mut_avg_rd = round(digits=2, MP_BAL_mut_avg)
                    MP_BAL_mut_avg_pad = rpad(MP_BAL_mut_avg_rd, 4)
                EPCI_BAL_avg_prop_v1_rd = round(digits=2, EPCI_BAL_avg_prop_v1)
                EPCI_BAL_avg_prop_v1_pad = rpad(EPCI_BAL_avg_prop_v1_rd, 5)
                EPCI_BAL_avg_prop_v2_rd = round(digits=2, EPCI_BAL_avg_prop_v2)
                EPCI_BAL_avg_prop_v2_pad = rpad(EPCI_BAL_avg_prop_v2_rd, 5)
                EPCI_BAL_mut_avg_v1_rd = round(digits=2, EPCI_BAL_mut_avg_v1)
                EPCI_BAL_mut_avg_v1_pad = rpad(EPCI_BAL_mut_avg_v1_rd, 4)
                EPCI_BAL_mut_avg_v2_rd = round(digits=2, EPCI_BAL_mut_avg_v2)
                EPCI_BAL_mut_avg_v2_pad = rpad(EPCI_BAL_mut_avg_v2_rd, 4)
                    mutpad = rpad(mut, 13)
                    BAL_Mut_ct_pad = lpad(BAL_Mut_ct, 3)
                    EPCI_mut_ct_pad = lpad(EPCI_mut_ct, 3)
                    Adjusted_BAL_Seq_ct_pad = lpad(Adjusted_BAL_Seq_ct, 5)
                    EPCI_pct_rd = round(digits=2, EPCI_pct)
                    EPCI_pct_pad = lpad(EPCI_pct_rd, 5)
                    MP_pct_rd = round(digits=2, MP_pct)
                    MP_pct_pad = lpad(MP_pct_rd, 5)
                    log10pvFISH_rd2 = add_zeros_to_rounded(log10pvFISH_rd, 2)
                    log10pvFISH_rd_pad = lpad(log10pvFISH_rd2, 7)
                    Chi2_pad = lpad(Chi2_string, 6)
                    push!(df_mp, (mut, BAL_Mut_ct, EPCI_mut_ct, Adjusted_BAL_Seq_ct, EPCI_pct, MP_pct, non_MP_pct, pv_fish, fold_incr, MP_BAL_avg_prop, MP_BAL_mut_avg, EPCI_BAL_avg_prop_v1, EPCI_BAL_mut_avg_v1, EPCI_BAL_avg_prop_v2, EPCI_BAL_mut_avg_v2, log10pvFISH, Chi2))
##                    df_mp: Mutation,BAL_Mut_ct,EPCI_Mut_ct,Adjusted_BAL_Seq_ct, EPCI_pct, BAL_MP_pct, Fishers_Exact_Pval, Fold_Incr, MP_AvgBALProp, MP_AvgBALMut, EPCI_AvgBALProp, EPCI_AvgBALMut, log10pvFish, Chi2)                    
                println(g, "               $(mutpad)|$(BAL_Mut_ct_pad)|$(EPCI_mut_ct_pad)|$(Adjusted_BAL_Seq_ct_pad)|$(EPCI_pct_pad)%|$(MP_pct_pad)%|$(log10pvFISH_rd_pad) | $(Chi2_pad)| $(EPCI_BAL_avg_prop_v1_pad) | $(EPCI_BAL_mut_avg_v1_pad) | $(EPCI_BAL_avg_prop_v2_pad)| $(EPCI_BAL_mut_avg_v2_pad) |")
                end
            end
        end
###########################################################################################################################################################################
###########################################################################################################################################################################
        CSV.write("$(folder_name)/$(folder_name)__DataFrame_CSV.csv", df_mp)
#        CSV.write("$(folder_name)/$(folder_name)__DataFrame_TSV.tsv", df_mp, delim='\t')
        open(mut_file, "a") do g
            println("###################################################################################################")
            println("      Final Number of Mutations in $(mut_pattern_name) = $(pattern_mut_set_len) | Total BAL Seqs = $(total_qualifying_BAL_seqs)")
            println("###################################################################################################")
            println(g)
            println(g, "###################################################################################################")
            println(g, "      Final Number of Mutations in $(mut_pattern_name) = $(pattern_mut_set_len)  | Total BAL Seqs = $(total_qualifying_BAL_seqs)")
            println(g, "###################################################################################################")
            print(g, "\n"^1)
            kz = keys(new_mut_by_round_dict)
            len_kz = length(kz)
            for i in 1:tot_rounds
                length_of_losses = 0
                key_or_not = 0
                if haskey(new_mut_by_round_dict, i)
                    key_or_not += 1
                    print(g, "Round $(i) Gained = ")
                    for new_mut in new_mut_by_round_dict[i]
                        print(g, "$(new_mut) | ")
                    end
                end
                if !isempty(lost_mut_by_round_dict[i])
                    if key_or_not ≥ 1
                        println(g)
                    end
                    key_or_not += 1
                    print(g, "Round $(i) Lost   = ")
                    for j in 1:length(lost_mut_by_round_dict[i])
                        mut = lost_mut_by_round_dict[i][j]
                        mut_len = length(mut) + 3
                        print(g, "$(mut) | ")
                        length_of_losses += mut_len
                    end
                end
                if !isempty(reason_lost_by_round[i])
                    pad = 75 - length_of_losses
                    printpad_left = lpad("### Reasons: ", pad)
                    print(g, printpad_left)
                    reason_set = Set{String}()
                    for reason in reason_lost_by_round[i]
                        if !(reason in reason_set)
                            print(g, reason, "||")
                        end
                        push!(reason_set, reason)
                    end
                end
                if key_or_not ≥ 1
                    println(g)
                end
            end
            println(g, "\n"^2)
            mct = 0
            for mut in pattern_mut_set_sort
                if mct < length(pattern_mut_set_sort)
                    print("$(mut), ")
                else
                    print(mut)
                end
                mct += 1
            end     
        end
        mp_muts_sort = pattern_mut_set_sort
        mp_seqs_final = new_pattern_seq_set
        mp_seqs_sort = sort(collect(mp_seqs_final), by = x -> (length(x), x))
        println("\n"^2)
        open("$(folder_name)/$(folder_name)__final_mp_seqs.txt", "w") do g
            mp_seqs_sort_join = join(mp_seqs_sort, ", ")
            println(mp_seqs_sort_join)
            println(g, mp_seqs_sort_join)
        end
        println("\n"^2)
        open("$(folder_name)/$(folder_name)__final_mp_muts.txt", "w") do g
            mp_muts_sort_join = join(mp_muts_sort, ", ")
            println(mp_muts_sort_join)
            println(g, mp_muts_sort_join)
        end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
        df_pango_ct = DataFrame(
            pango_lineage = String[],
            pango_unaliased = String[],
            count = Int[])
        df_clade_ct = DataFrame(
            clade = String[],
            cladepango = String[],
            cladepango_unaliased = String[],
            count = Int[])
        df_cladepango_ct = DataFrame(
            cladepango = String[],
            cladepango_unaliased = String[],
            count = Int[])
        df_megaclade_ct = DataFrame(
            megaclade = String[],
            count = Int[],
            expected_ct = Float64[],
            fold_increase = Union{Float64,String}[])
        pango = ""; clade = ""; pango_unaliased = ""; cladepango = ""
        pango_ct_dict = Dict{String, Int}()
        clade_ct_dict = Dict{String, Int}()
        cladepango_ct_dict = Dict{String, Int}()
        for seq in mp_seqs_final
            pango = seq_pango[seq]
            clade = seq_clade[seq]
            pango_unaliased = pango_to_pango_unaliased_v2[pango]
            cladepango = clade_to_pango[clade]
            pango_ct_dict[pango] = get(pango_ct_dict, pango, 0) + 1
            clade_ct_dict[clade] = get(clade_ct_dict, clade, 0) + 1
            cladepango_ct_dict[cladepango] = get(cladepango_ct_dict, cladepango, 0) + 1
        end
        pango_ct_dict_sort = sort(collect(pango_ct_dict), by = x -> x[2])
        clade_ct_dict_sort = sort(collect(clade_ct_dict), by = x -> x[2])
        cladepango_ct_dict_sort = sort(collect(cladepango_ct_dict), by = x -> x[2])
###################################################################
        pango_csv_file = "$(folder_name)/BAL__PANGO_CTS_$(date_now_fx).csv"
        clade_csv_file = "$(folder_name)/BAL__CLADE_CTS_$(date_now_fx).csv"
        cladepango_csv_file = "$(folder_name)/BAL__CLADEPANGO_CTS_$(date_now_fx).csv"
        megaclade_csv_file = "$(folder_name)/BAL__MEGACLADE_CTS_$(date_now_fx).csv"
###################################################################
        mega_clade_pangos = ["pre_VOC", "B.1.1.7", "B.1.351", "B.1.1.28.1", "B.1.617", "B.1.1.529.1", "B.1.1.529.2", "B.1.1.529.4", "B.1.1.529.5", "XBB", "B.1.1.529.2.86"]
        mega_clade_dict_keys = ["pre_VOC", "B.1.1.7", "B.1.351", "P.1", "B.1.617", "BA.1", "BA.2", "BA.5", "XBB", "BA.2.86", "Omicron_Unknown", "Minor_Recombinant"]
        megaclade_ct_dict = Dict{String, Int}()
        for megaclade in mega_clade_dict_keys
            megaclade_ct_dict[megaclade] = 0
        end
        for i in 1:length(pango_ct_dict_sort)
            pango__count = pango_ct_dict_sort[i]
            pango = pango__count[1]
            count = pango__count[2]
            unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
            push!(df_pango_ct, (pango, unaliased, count))     
        end
        for i in 1:length(clade_ct_dict_sort)
            clade__count = clade_ct_dict_sort[i]
            clade = clade__count[1]
            count = clade__count[2]
            cladepango = clade_to_pango[clade]
            cladepango_unaliased = get(pango_to_pango_unaliased_v2, cladepango, cladepango)
            push!(df_clade_ct, (clade, cladepango, cladepango_unaliased, count))
        end
        for i in 1:length(cladepango_ct_dict_sort)
            cladepango__count = cladepango_ct_dict_sort[i]
            cladepango = cladepango__count[1]
            count = cladepango__count[2]
            cladepango_unaliased = get(pango_to_pango_unaliased_v2, cladepango, cladepango)
            push!(df_cladepango_ct, (cladepango, cladepango_unaliased, count))
        end
######################################################################################################################
        BA286_Xs = ["XEC", "XEK", "XFG"]
        XBB_Xs = ["XCH"]
        for i in 1:length(pango_ct_dict_sort)
            complete = 0
            pango__count = pango_ct_dict_sort[i]
            pango = pango__count[1]
            count = pango__count[2]
            unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
            if length(unaliased) ≥ 3
                if unaliased[1:3] == "XBB"
                    megaclade_ct_dict["XBB"] += count
                    complete = 1
                elseif unaliased[1:3] == "XEC" || unaliased[1:3] == "XEK" || unaliased[1:3] == "XFG"
                    megaclade_ct_dict["BA.2.86"] += count
                    complete = 1
                elseif unaliased[1:3] == "XCH"
                    megaclade_ct_dict["XBB"] += count
                    complete = 1
                end
                if length(unaliased) ≥ 7
                    if unaliased[1:7] == "B.1.1.7"
                        megaclade_ct_dict["B.1.1.7"] += count
                        complete = 1
                    elseif unaliased[1:7] == "B.1.351"
                        megaclade_ct_dict["B.1.351"] += count
                        complete = 1
                    elseif unaliased[1:7] == "B.1.617"
                        megaclade_ct_dict["B.1.617"] += count
                        complete = 1
                    end
                    if length(unaliased) ≥ 9
                        if unaliased == "B.1.1.529"
                            megaclade_ct_dict["Omicron_Unknown"] += count
                            complete = 1
                        end
                        if length(unaliased) ≥ 10 
                            if unaliased[1:10] == "B.1.1.28.1"
                                megaclade_ct_dict["P.1"] += count
                                complete = 1
                            end
                            if length(unaliased) ≥ 11
                                if unaliased[1:11] == "B.1.1.529.1"
                                    megaclade_ct_dict["BA.1"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.2"
                                    megaclade_ct_dict["BA.2"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.4"
                                    megaclade_ct_dict["BA.5"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.5"
                                    megaclade_ct_dict["BA.5"] += count
                                    complete = 1
                                end
                                if length(unaliased) ≥ 14
                                    if unaliased[1:14] == "B.1.1.529.2.86"
                                        megaclade_ct_dict["BA.2.86"] += count
                                        megaclade_ct_dict["BA.2"] = megaclade_ct_dict["BA.2"] - count
                                        complete = 1
                                    end
                                end
                            end
                        end
                    end
                end
            end
            if complete == 0
                if unaliased[1] == 'A' || unaliased[1] == 'B'
                    megaclade_ct_dict["pre_VOC"] +=count
                elseif unaliased[1] == 'X'
                    megaclade_ct_dict["Minor_Recombinant"] +=count
                else
                    println("********************** WARNING: Uncategorized PANGO **************************")
                end
            end
        end
        megaclade_ct_dict_sort = sort(collect(megaclade_ct_dict), by = x -> x[2], rev=true)
        total_MP_sequence_ct = length(mp_seqs_final)
        for i in 1:length(megaclade_ct_dict_sort)
            megaclade___count = megaclade_ct_dict_sort[i]
            megaclade = megaclade___count[1]
            count = megaclade___count[2]
    #######################
            megaclade_expected_ct = megaclade_EPCI_proportion_dict[megaclade]*total_MP_sequence_ct
            EPCI_megaclade_ct = megaclade_EPCI_ct_dict[megaclade]
            non_MP_EPCI_ct = total_EPCI_seq_ct - total_MP_sequence_ct
            non_MP_megaclade_ct = EPCI_megaclade_ct - count
            megaclade_non_MP_prop = non_MP_megaclade_ct/non_MP_EPCI_ct
            megaclade_MP_prop = count/total_MP_sequence_ct
            megaclade_fold_enrichment = megaclade_MP_prop/megaclade_non_MP_prop
            push!(df_megaclade_ct, (megaclade, count, megaclade_expected_ct, megaclade_fold_enrichment))
        end
        CSV.write(pango_csv_file, df_pango_ct)    #; CSV.write(pango_ct_file_tsv_df_RANDOM, df_pango_ct, delim='\t')
        CSV.write(clade_csv_file, df_clade_ct)
        CSV.write(cladepango_csv_file, df_cladepango_ct)
        CSV.write(megaclade_csv_file, df_megaclade_ct)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
## mp_seqs = all seqs that meet criteria   1                   2                3                 4              5                      6                      7                          8                      9                     10                      11                                12              
##      AA_mut_pattern_iterative(mut_pattern_name::String, tot_rounds::Int, plus_or_minus::Int, min_AA_ct::Int, mp_seqs::Set{String}, mp_ct_min::Float64, min_log_pv_fish::Float64, abs_min_fish::Float64, min_chi_sq::Float64, abs_chi2_min::Float64, min_chr_all_ratio::Float64, seq_factor_OFF_HALF_ON__0_1_2::Int)
        if sub_0__posonly_1 == 0
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_df_mp.jld2", "df_mp", df_mp)
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_mp_seqs_sort.jld2", "mp_seqs_sort", mp_seqs_sort) 
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_mp_muts_sort.jld2", "mp_muts_sort", mp_muts_sort)
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_prop_dict.jld2", "prop_dict", prop_dict) 
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_non_BAL_related_seqs_to_exclude.jld2", "non_BAL_related_seqs_to_exclude", non_BAL_related_seqs_to_exclude)
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_all_unique_chr_seqs_set_BAL.jld2", "all_unique_chr_seqs_set", all_unique_chr_seqs_set) 
            save("$(folder_name)/$(mut_pattern_name)_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_pattern_mut_set_by_round_dict.jld2", "pattern_mut_set_by_round_dict", pattern_mut_set_by_round_dict)
        elseif sub_0__posonly_1 == 1
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_df_mp.jld2", "df_mp", df_mp)
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_mp_seqs_sort.jld2", "mp_seqs_sort", mp_seqs_sort) 
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_mp_muts_sort.jld2", "mp_muts_sort", mp_muts_sort)
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_prop_dict.jld2", "prop_dict", prop_dict) 
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_non_BAL_related_seqs_to_exclude.jld2", "non_BAL_related_seqs_to_exclude", non_BAL_related_seqs_to_exclude)
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_all_unique_chr_seqs_set_BAL.jld2", "all_unique_chr_seqs_set", all_unique_chr_seqs_set) 
            save("$(folder_name)/$(mut_pattern_name)_POS_ONLY_$(date)_minAAct$(min_AA_ct)_mpAAmin$(mp_ct_min)_minFish$(abs_min_fish)_plusMinus$(plus_or_minus)_pattern_mut_set_by_round_dict.jld2", "pattern_mut_set_by_round_dict", pattern_mut_set_by_round_dict)
        end
    end
end
########################################################################################################################################

2026_03_04__615PM
Length of confirmed BAL seqs list = 38
ORF1a:3076, ORF1a:4205, N:210, ORF1a:1500, E:27, N:416, E:30, ORF7a:115, M:74, ORF1a:3077, ORF1a:3058, ORF1b:2537, S:1186, E:5, S:1259, M:125, ORF1a:814, E:14, N:325, M:28, S:1179, E:32, S:375, ORF1a:2972, ORF1b:1181, S:1169, S:691, S:653, 


AA_mut_pattern_iterative_2025_11_29 (generic function with 1 method)

In [371]:
### Execute AA_mut_pattern_iterative_2025_11_29
# rds=50, ±=3, minAAct=4, mp_ct_min=2.0, minLogPvFish=6.0, absMinFish=3.0, minChi^2=20.0, absChi^2min=14.0, seqFactorOnOffHalf=1
date_now_2 = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now_2)
##############################
minEPCI_pct_1 = 45
minBALprop_1 = 0.15
minAbsFsh_1 = 3.00
##############################
minEPCI_pct = minEPCI_pct_1
minBALprop = minBALprop_1
AbsFsh = minAbsFsh_1
###########################################
if sub_0__posonly_1 == 1
    minEPCI_pct = minQryPct_1 + 2
    minBALprop = minBALprop_1 + 0.02
    AbsFsh = minAbsFsh_1 + 0.2
end
#############################################################################################################################################################################    
mut_pattern_round_ct = 0                 #name |minQryPct| minBALprop|rds|±|minAA|mpSeqs|       mpMin|minFsh|absFsh|minChi|absMinChi|min_chr_all_ratio|seq_factor_OFF_HALF_ON__0_1_2
start = time()                      #     1        2           3      4   5  6         7          8     9      10     11      12            13               14
    AA_mut_pattern_iterative_2025_11_29("BAL",minEPCI_pct,minBALprop, 50, 3, 3, BAL_uniq_seq_set,2.0,  6.0,  AbsFsh, 0.0,     0.0,         0.4,               0)
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println()
println("Runtime v0 = $(runtime_rd) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
println()
#          1                    2                3                 4               5                      6
# (mut_pattern_name::String, tot_rounds::Int, plus_or_minus::Int, min_AA_ct::Int, mp_seqs::Set{String}, mp_ct_min::Float64, 
#         7                      8                      9                              10                    11     
#  min_log_pv_fish::Float64, min_chi_sq::Float64, abs_chi2_min::Float64, seq_factor_OFF_HALF_ON__0_1_2::Int, )

2026_03_04__615PM
               Total AA Subs in BAL, Round 1 = 28 | Total seqs = 443
               Total AA Subs in BAL, Round 2 = 33 | Total seqs = 516
               Total AA Subs in BAL, Round 3 = 35 | Total seqs = 555
               Total AA Subs in BAL, Round 4 = 39 | Total seqs = 615
               Total AA Subs in BAL, Round 5 = 42 | Total seqs = 641
               Total AA Subs in BAL, Round 6 = 46 | Total seqs = 684
               Total AA Subs in BAL, Round 8 = 49 | Total seqs = 744
               Total AA Subs in BAL, Round 9 = 51 | Total seqs = 752
               Total AA Subs in BAL, Round 12 = 52 | Total seqs = 765
               Total AA Subs in BAL, Round 13 = 54 | Total seqs = 773
               Total AA Subs in BAL, Round 14 = 55 | Total seqs = 777
               Total AA Subs in BAL, Round 15 = 57 | Total seqs = 783
               Total AA Subs in BAL, Round 16 = 60 | Total seqs = 788
               Total AA Subs in BAL, Round 17 = 61 | Total seqs = 796
          

In [159]:
### After downloading all sequences that contain any words that might be related to bronchoalveolar lavage, this function   
### filters out the sequences that definitely have nothing to do with BAL (sequences with an author whose name contains "lung" within it, for example. These non-BAL
### sequences were determined by manual examination of sequence metadata. 
########################################################################################################################################################################
### Lung, lavage, bronchial, BAL, etc GISAID searches + filters
###  search terms = "BAL", "bronchoalveolar", "lavage", "alveolar", "lung", "lower respiratory sample", "bronchial", "bronchoscopy", "bronchoaspiration", "organ", 
###                 "endotracheal", "tracheal", "trachea", "trach aspirate"
########################################################################################################################################################################
bronchoalveolar_seqs = Set{String}()
headers, seqs = read_fasta("Downloads/bronchoalveolar_GISAID_search_1513seq.fasta")
for head in headers
    epi = EPI_ISL(head)
    push!(bronchoalveolar_seqs, epi)
end
println("bronchoalveolar_seqs length = $(length(bronchoalveolar_seqs))"); print("\n"^1)
########################################################
lavage_seqs = stringlist_to_set("EPI_ISL_402119-402121, EPI_ISL_402123-402124, EPI_ISL_402127-402130, EPI_ISL_402132, EPI_ISL_403928-403931, EPI_ISL_403934, EPI_ISL_406592-406595, EPI_ISL_406798-406801, EPI_ISL_408483-408484, EPI_ISL_408488, EPI_ISL_412459, EPI_ISL_412862, EPI_ISL_412898, EPI_ISL_412967, EPI_ISL_414497, EPI_ISL_417445-417447, EPI_ISL_417482, EPI_ISL_417923, EPI_ISL_418660, EPI_ISL_419255, EPI_ISL_420912, EPI_ISL_421227, EPI_ISL_421229, EPI_ISL_424343-424344, EPI_ISL_434529-434532, EPI_ISL_434534, EPI_ISL_444534, EPI_ISL_444536-444537, EPI_ISL_444539, EPI_ISL_444554, EPI_ISL_444558, EPI_ISL_444583, EPI_ISL_444585, EPI_ISL_447755, EPI_ISL_447771, EPI_ISL_447774-447776, EPI_ISL_447778, EPI_ISL_447784, EPI_ISL_447791, EPI_ISL_447799, EPI_ISL_447803-447804, EPI_ISL_447807-447808, EPI_ISL_447813, EPI_ISL_460083, EPI_ISL_460086, EPI_ISL_475803, EPI_ISL_475805, EPI_ISL_479616-479618, EPI_ISL_482784, EPI_ISL_485603, EPI_ISL_527812-527814, EPI_ISL_529213-529217, EPI_ISL_548950, EPI_ISL_583571, EPI_ISL_602319, EPI_ISL_609994-609995, EPI_ISL_639998, EPI_ISL_649058-649059, EPI_ISL_672664-672668, EPI_ISL_672670-672686, EPI_ISL_722057, EPI_ISL_722080, EPI_ISL_722134-722169, EPI_ISL_745318, EPI_ISL_745326-745334, EPI_ISL_746826, EPI_ISL_765569, EPI_ISL_775898, EPI_ISL_850666, EPI_ISL_852576, EPI_ISL_852684, EPI_ISL_852710, EPI_ISL_852720, EPI_ISL_852722, EPI_ISL_877561, EPI_ISL_882883, EPI_ISL_882896, EPI_ISL_904018-904034, EPI_ISL_944634, EPI_ISL_982892, EPI_ISL_1004233-1004242, EPI_ISL_1015002, EPI_ISL_1015021-1015030, EPI_ISL_1040814, EPI_ISL_1084726, EPI_ISL_1084730, EPI_ISL_1084737-1084738, EPI_ISL_1084740, EPI_ISL_1096141-1096142, EPI_ISL_1096983, EPI_ISL_1096985, EPI_ISL_1142081-1142082, EPI_ISL_1142566, EPI_ISL_1147538-1147539, EPI_ISL_1153937, EPI_ISL_1195390, EPI_ISL_1195426, EPI_ISL_1217674, EPI_ISL_1262459, EPI_ISL_1285074, EPI_ISL_1292808, EPI_ISL_1351749, EPI_ISL_1354409, EPI_ISL_1422562, EPI_ISL_1569743, EPI_ISL_1588633, EPI_ISL_1588677, EPI_ISL_1642767, EPI_ISL_1642783, EPI_ISL_1642824, EPI_ISL_1662057, EPI_ISL_1662205, EPI_ISL_1662208, EPI_ISL_1678149, EPI_ISL_1678378, EPI_ISL_1678605-1678606, EPI_ISL_1678674-1678675, EPI_ISL_1678690, EPI_ISL_1678768, EPI_ISL_1678772, EPI_ISL_1678831, EPI_ISL_1678842, EPI_ISL_1751331, EPI_ISL_1751414, EPI_ISL_1751492, EPI_ISL_1751498, EPI_ISL_1751818, EPI_ISL_1752130, EPI_ISL_1752375, EPI_ISL_1752420, EPI_ISL_1846444, EPI_ISL_1849099, EPI_ISL_1849119, EPI_ISL_1849400, EPI_ISL_1908476, EPI_ISL_1939829, EPI_ISL_2036230, EPI_ISL_2036233, EPI_ISL_2036237, EPI_ISL_2091139, EPI_ISL_2095172, EPI_ISL_2123885, EPI_ISL_2123890, EPI_ISL_2235295, EPI_ISL_2272558, EPI_ISL_2272579, EPI_ISL_2281746, EPI_ISL_2299669, EPI_ISL_2313935, EPI_ISL_2324423, EPI_ISL_2383045, EPI_ISL_2391648, EPI_ISL_2402221, EPI_ISL_2483815, EPI_ISL_2484152, EPI_ISL_2488761-2488763, EPI_ISL_2490544, EPI_ISL_2535312, EPI_ISL_2535398, EPI_ISL_2535414, EPI_ISL_2535419, EPI_ISL_2621563, EPI_ISL_2661755, EPI_ISL_2671653, EPI_ISL_2709248, EPI_ISL_2762183, EPI_ISL_2773991, EPI_ISL_2801690, EPI_ISL_2845544, EPI_ISL_2845597, EPI_ISL_2845608, EPI_ISL_2845645, EPI_ISL_2845942-2845943, EPI_ISL_2956898, EPI_ISL_2993446, EPI_ISL_3031951, EPI_ISL_3066314, EPI_ISL_3071071, EPI_ISL_3100455, EPI_ISL_3117542, EPI_ISL_3133566, EPI_ISL_3355592, EPI_ISL_3491798, EPI_ISL_3500367, EPI_ISL_3500770, EPI_ISL_3500949, EPI_ISL_3507365, EPI_ISL_3556916, EPI_ISL_3568791, EPI_ISL_3667128-3667129, EPI_ISL_3748184, EPI_ISL_3893706, EPI_ISL_3982251, EPI_ISL_4045386, EPI_ISL_4072038, EPI_ISL_4261663, EPI_ISL_4549080, EPI_ISL_4549107-4549108, EPI_ISL_4549133, EPI_ISL_4549144, EPI_ISL_4616573, EPI_ISL_4621457, EPI_ISL_4632968, EPI_ISL_4849731, EPI_ISL_5107511, EPI_ISL_5121979, EPI_ISL_5402838, EPI_ISL_5402955, EPI_ISL_5593051, EPI_ISL_5681555, EPI_ISL_5881164, EPI_ISL_5913765, EPI_ISL_6125384, EPI_ISL_6125392, EPI_ISL_6133024-6133025, EPI_ISL_6157236, EPI_ISL_6642226, EPI_ISL_6863572, EPI_ISL_6959518, EPI_ISL_7034850, EPI_ISL_7152236, EPI_ISL_7192689, EPI_ISL_7195636, EPI_ISL_7473133-7473134, EPI_ISL_7473136, EPI_ISL_7812986, EPI_ISL_7969788, EPI_ISL_8151541, EPI_ISL_8246256, EPI_ISL_8650047, EPI_ISL_8728795, EPI_ISL_8728828, EPI_ISL_8764161, EPI_ISL_8825830, EPI_ISL_8886131, EPI_ISL_8890608, EPI_ISL_8898218, EPI_ISL_8910905, EPI_ISL_8964692, EPI_ISL_9004081, EPI_ISL_9057987, EPI_ISL_9084777, EPI_ISL_9202280-9202281, EPI_ISL_9214783, EPI_ISL_9214991, EPI_ISL_9258621, EPI_ISL_9873278, EPI_ISL_10104797, EPI_ISL_10590270, EPI_ISL_10680183, EPI_ISL_10695750, EPI_ISL_10695757, EPI_ISL_10797175, EPI_ISL_11029753-11029754, EPI_ISL_11532903, EPI_ISL_11533135, EPI_ISL_11536070, EPI_ISL_11794345, EPI_ISL_11901406, EPI_ISL_11957613, EPI_ISL_11960787, EPI_ISL_12009104, EPI_ISL_12611157-12611179, EPI_ISL_12628184-12628185, EPI_ISL_12647832-12647969, EPI_ISL_12672726, EPI_ISL_12703333-12703365, EPI_ISL_12824078, EPI_ISL_12996407, EPI_ISL_13106275, EPI_ISL_13370684, EPI_ISL_13405084, EPI_ISL_13483775-13483776, EPI_ISL_13503362, EPI_ISL_13523452, EPI_ISL_13697278, EPI_ISL_13710827, EPI_ISL_13721083, EPI_ISL_13724443, EPI_ISL_13750385, EPI_ISL_13821775, EPI_ISL_13925630, EPI_ISL_14027973-14027974, EPI_ISL_14081372, EPI_ISL_14081390, EPI_ISL_14371393, EPI_ISL_14418987, EPI_ISL_14444367, EPI_ISL_14444389, EPI_ISL_14479735, EPI_ISL_14532734, EPI_ISL_14553998, EPI_ISL_14560895, EPI_ISL_14562050, EPI_ISL_14657162, EPI_ISL_14685935, EPI_ISL_14685942, EPI_ISL_14685946, EPI_ISL_14696982, EPI_ISL_14778590, EPI_ISL_14842764, EPI_ISL_14842768, EPI_ISL_14844961, EPI_ISL_14904594-14904608, EPI_ISL_14918022-14918023, EPI_ISL_14918791, EPI_ISL_14920334, EPI_ISL_14935908, EPI_ISL_14935931, EPI_ISL_14944319-14944323, EPI_ISL_14944327-14944329, EPI_ISL_14966080-14966093, EPI_ISL_15001380, EPI_ISL_15022505, EPI_ISL_15169445, EPI_ISL_15204879, EPI_ISL_15251243, EPI_ISL_15285482, EPI_ISL_15313006-15313015, EPI_ISL_15424734-15424768, EPI_ISL_15435517, EPI_ISL_15436429, EPI_ISL_15481006, EPI_ISL_15511842, EPI_ISL_15512667, EPI_ISL_15537455, EPI_ISL_15572128, EPI_ISL_15665394-15665407, EPI_ISL_15719141, EPI_ISL_15736818, EPI_ISL_15781862-15781892, EPI_ISL_15789169, EPI_ISL_15800232, EPI_ISL_15839963, EPI_ISL_15912969, EPI_ISL_15963321, EPI_ISL_15983112-15983115, EPI_ISL_15985495, EPI_ISL_15988460, EPI_ISL_16003254-16003264, EPI_ISL_16018586, EPI_ISL_16018588, EPI_ISL_16018642, EPI_ISL_16018659, EPI_ISL_16055689, EPI_ISL_16078250-16078252, EPI_ISL_16078257, EPI_ISL_16138329, EPI_ISL_16153726, EPI_ISL_16157875, EPI_ISL_16157885, EPI_ISL_16177080, EPI_ISL_16222576, EPI_ISL_16222580, EPI_ISL_16222588, EPI_ISL_16222590, EPI_ISL_16222594-16222595, EPI_ISL_16260321, EPI_ISL_16275312, EPI_ISL_16298428, EPI_ISL_16332297-16332298, EPI_ISL_16332301-16332304, EPI_ISL_16332307, EPI_ISL_16332309, EPI_ISL_16343211-16343215, EPI_ISL_16343217-16343220, EPI_ISL_16354099-16354105, EPI_ISL_16355484-16355505, EPI_ISL_16355598, EPI_ISL_16356910, EPI_ISL_16359188, EPI_ISL_16389924, EPI_ISL_16395040, EPI_ISL_16410666, EPI_ISL_16410718, EPI_ISL_16411167, EPI_ISL_16461620, EPI_ISL_16479607-16479621, EPI_ISL_16507432, EPI_ISL_16536257, EPI_ISL_16543464-16543477, EPI_ISL_16565384, EPI_ISL_16616032, EPI_ISL_16616037, EPI_ISL_16620443, EPI_ISL_16638800, EPI_ISL_16672268-16672283, EPI_ISL_16679191-16679193, EPI_ISL_16682478, EPI_ISL_16691487, EPI_ISL_16753887, EPI_ISL_16789020, EPI_ISL_16789023, EPI_ISL_16789027, EPI_ISL_16789031, EPI_ISL_16789034, EPI_ISL_16789039, EPI_ISL_16789042, EPI_ISL_16789046, EPI_ISL_16789050, EPI_ISL_16789053, EPI_ISL_16789057, EPI_ISL_16789061, EPI_ISL_16789064, EPI_ISL_16789068, EPI_ISL_16789072, EPI_ISL_16789076, EPI_ISL_16868655-16868657, EPI_ISL_16898237, EPI_ISL_16898239-16898240, EPI_ISL_16898242-16898243, EPI_ISL_16898245, EPI_ISL_16898247, EPI_ISL_16898249-16898250, EPI_ISL_16898252, EPI_ISL_16898254, EPI_ISL_16935796-16935805, EPI_ISL_16951978, EPI_ISL_16951997, EPI_ISL_16954664, EPI_ISL_16957446, EPI_ISL_16957448, EPI_ISL_16957458, EPI_ISL_16966626, EPI_ISL_16984862, EPI_ISL_16984870, EPI_ISL_17003178, EPI_ISL_17008676, EPI_ISL_17027270-17027282, EPI_ISL_17041085-17041087, EPI_ISL_17071452, EPI_ISL_17071533, EPI_ISL_17071654, EPI_ISL_17071656, EPI_ISL_17071658, EPI_ISL_17074791-17074800, EPI_ISL_17082284-17082313, EPI_ISL_17088847, EPI_ISL_17127142, EPI_ISL_17170803-17170809, EPI_ISL_17174959, EPI_ISL_17183094-17183104, EPI_ISL_17198046, EPI_ISL_17213920, EPI_ISL_17215237-17215246, EPI_ISL_17233410, EPI_ISL_17321951-17321965, EPI_ISL_17350314, EPI_ISL_17350374, EPI_ISL_17371910-17371921, EPI_ISL_17385090, EPI_ISL_17406628, EPI_ISL_17407898, EPI_ISL_17408305, EPI_ISL_17408352, EPI_ISL_17408356, EPI_ISL_17414331, EPI_ISL_17423463-17423473, EPI_ISL_17446485-17446486, EPI_ISL_17446489-17446490, EPI_ISL_17446493, EPI_ISL_17463807, EPI_ISL_17463814, EPI_ISL_17474998, EPI_ISL_17514694, EPI_ISL_17518362, EPI_ISL_17518372, EPI_ISL_17521229, EPI_ISL_17548513, EPI_ISL_17553089, EPI_ISL_17557556, EPI_ISL_17584918-17584948, EPI_ISL_17619330-17619338, EPI_ISL_17634290-17634291, EPI_ISL_17637901, EPI_ISL_17658324, EPI_ISL_17674512-17674515, EPI_ISL_17674517-17674518, EPI_ISL_17685404, EPI_ISL_17685562-17685572, EPI_ISL_17709837-17709926, EPI_ISL_17711339-17711348, EPI_ISL_17719124, EPI_ISL_17721553, EPI_ISL_17736110, EPI_ISL_17736161, EPI_ISL_17777729, EPI_ISL_17780572-17780584, EPI_ISL_17809170-17809179, EPI_ISL_17826285, EPI_ISL_17829710, EPI_ISL_17964486, EPI_ISL_17972112, EPI_ISL_17972848, EPI_ISL_17972850, EPI_ISL_17997284, EPI_ISL_18000356, EPI_ISL_18011048-18011063, EPI_ISL_18012374, EPI_ISL_18035028, EPI_ISL_18065755-18065757, EPI_ISL_18069163, EPI_ISL_18078883, EPI_ISL_18096030, EPI_ISL_18099953-18099955, EPI_ISL_18111040-18111041, EPI_ISL_18116015, EPI_ISL_18163431, EPI_ISL_18208803, EPI_ISL_18210658-18210681, EPI_ISL_18212584, EPI_ISL_18225332, EPI_ISL_18227951, EPI_ISL_18244640, EPI_ISL_18255077-18255092, EPI_ISL_18266191-18266194, EPI_ISL_18266196-18266200, EPI_ISL_18266202-18266205, EPI_ISL_18274516, EPI_ISL_18275284, EPI_ISL_18284434-18284449, EPI_ISL_18302465, EPI_ISL_18311648-18311657, EPI_ISL_18320086-18320099, EPI_ISL_18322385, EPI_ISL_18323517-18323532, EPI_ISL_18330352-18330367, EPI_ISL_18330631-18330646, EPI_ISL_18331808, EPI_ISL_18333033-18333039, EPI_ISL_18343342, EPI_ISL_18343438, EPI_ISL_18343879, EPI_ISL_18345690, EPI_ISL_18346603-18346618, EPI_ISL_18353888, EPI_ISL_18353909, EPI_ISL_18356765, EPI_ISL_18359530-18359545, EPI_ISL_18385948, EPI_ISL_18405563-18405577, EPI_ISL_18417750-18417765, EPI_ISL_18430000, EPI_ISL_18437952-18437967, EPI_ISL_18438014-18438029, EPI_ISL_18443626-18443657, EPI_ISL_18458762-18458777, EPI_ISL_18474245, EPI_ISL_18474266-18474281, EPI_ISL_18480824-18480838, EPI_ISL_18486543-18486544, EPI_ISL_18510419-18510434, EPI_ISL_18514068-18514083, EPI_ISL_18523049-18523063, EPI_ISL_18523161, EPI_ISL_18535078-18535089, EPI_ISL_18541240, EPI_ISL_18541242, EPI_ISL_18543650, EPI_ISL_18543679-18543694, EPI_ISL_18543810-18543825, EPI_ISL_18559065, EPI_ISL_18559070, EPI_ISL_18563075-18563088, EPI_ISL_18563222, EPI_ISL_18567736, EPI_ISL_18585511-18585543, EPI_ISL_18585545-18585552, EPI_ISL_18585554-18585557, EPI_ISL_18588311-18588342, EPI_ISL_18612999-18613014, EPI_ISL_18622366-18622381, EPI_ISL_18638224-18638239, EPI_ISL_18662376, EPI_ISL_18674004-18674008, EPI_ISL_18674010-18674023, EPI_ISL_18674025-18674029, EPI_ISL_18675121, EPI_ISL_18675134, EPI_ISL_18677246, EPI_ISL_18700348-18700366, EPI_ISL_18700384, EPI_ISL_18702271-18702284, EPI_ISL_18722000, EPI_ISL_18741828, EPI_ISL_18741831, EPI_ISL_18767237, EPI_ISL_18792937, EPI_ISL_18822195-18822208, EPI_ISL_18824049, EPI_ISL_18824912-18824926, EPI_ISL_18842044, EPI_ISL_18842046-18842048, EPI_ISL_18842050-18842056, EPI_ISL_18842058, EPI_ISL_18852493-18852504, EPI_ISL_18853600, EPI_ISL_18880330, EPI_ISL_18895660, EPI_ISL_18900939, EPI_ISL_18933318, EPI_ISL_18936434, EPI_ISL_18968825, EPI_ISL_18990105-18990107, EPI_ISL_18991885, EPI_ISL_18992114, EPI_ISL_19002299, EPI_ISL_19049545, EPI_ISL_19069483, EPI_ISL_19092550, EPI_ISL_19130836, EPI_ISL_19196027, EPI_ISL_19209583, EPI_ISL_19243835, EPI_ISL_19244762, EPI_ISL_19261040-19261041, EPI_ISL_19270732, EPI_ISL_19270782, EPI_ISL_19270784, EPI_ISL_19276224, EPI_ISL_19283657, EPI_ISL_19302363, EPI_ISL_19326635, EPI_ISL_19331907, EPI_ISL_19333027, EPI_ISL_19414066-19414068, EPI_ISL_19444288-19444290, EPI_ISL_19444300-19444303, EPI_ISL_19444305, EPI_ISL_19444307-19444308, EPI_ISL_19444311-19444313, EPI_ISL_19453799, EPI_ISL_19457180, EPI_ISL_19477229, EPI_ISL_19477245, EPI_ISL_19487956, EPI_ISL_19530653, EPI_ISL_19540761, EPI_ISL_19546543, EPI_ISL_19549840, EPI_ISL_19550100, EPI_ISL_19553366-19553370, EPI_ISL_19553372-19553373, EPI_ISL_19588462, EPI_ISL_19615222, EPI_ISL_19618600, EPI_ISL_19628593, EPI_ISL_19645473-19645474, EPI_ISL_19658692, EPI_ISL_19671572, EPI_ISL_19676153, EPI_ISL_19676160, EPI_ISL_19689967, EPI_ISL_19726607, EPI_ISL_19726653, EPI_ISL_19739555, EPI_ISL_19739680, EPI_ISL_19787452, EPI_ISL_19808161, EPI_ISL_19827775, EPI_ISL_19872727, EPI_ISL_19902962, EPI_ISL_19902984-19902985, EPI_ISL_19903184-19903186, EPI_ISL_19903232, EPI_ISL_19903332, EPI_ISL_19977768, EPI_ISL_20001073, EPI_ISL_20001090, EPI_ISL_20001092, EPI_ISL_20004425, EPI_ISL_20005422, EPI_ISL_20032171, EPI_ISL_20034202, EPI_ISL_20036538, EPI_ISL_20049879, EPI_ISL_20049885, EPI_ISL_20052335-20052336, EPI_ISL_20052340, EPI_ISL_20091626, EPI_ISL_20094106, EPI_ISL_20094114, EPI_ISL_20094288, EPI_ISL_20094303, EPI_ISL_20094307, EPI_ISL_20094316-20094317, EPI_ISL_20094328, EPI_ISL_20094344, EPI_ISL_20094347, EPI_ISL_20101292, EPI_ISL_20129191, EPI_ISL_20142897, EPI_ISL_20142910-20142911, EPI_ISL_20142926, EPI_ISL_20142993-20142995, EPI_ISL_20143005, EPI_ISL_20143007-20143008, EPI_ISL_20143010-20143011, EPI_ISL_20143087, EPI_ISL_20143107, EPI_ISL_20143109, EPI_ISL_20143194-20143197, EPI_ISL_20143210, EPI_ISL_20143298, EPI_ISL_20160725, EPI_ISL_20160727, EPI_ISL_20160729, EPI_ISL_20181431, EPI_ISL_20185471, EPI_ISL_20191499-20191500, EPI_ISL_20191575, EPI_ISL_20191582, EPI_ISL_20191673, EPI_ISL_20191682-20191683, EPI_ISL_20238732, EPI_ISL_20244054")
lavage_net_seqs = setdiff(lavage_seqs, bronchoalveolar_seqs)
println("lavage_net_seqs length = $(length(lavage_net_seqs))")
lavage_net_sort = sort(collect(lavage_net_seqs), by = x -> (length(x), x))
lavage_net_sort_join = join(lavage_net_sort, ", ")
### println(lavage_net_sort_join)
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_plus_lavage = union(bronchoalveolar_seqs, lavage_seqs)
#############################################################################################################################################################################
#############################################################################################################################################################################
println("bronchoalveolar_plus_lavage length = $(length(bronchoalveolar_plus_lavage))"); print("\n"^1)
########################################################################################################################################################################
##### Turns out there are no sequences with "bronchoscopy" but not "bronchoalveolar" in the metadata.
#bronchoscopy_seqs = stringlist_to_set("EPI_ISL_14479735, EPI_ISL_14553998, EPI_ISL_14657162, EPI_ISL_14844961, EPI_ISL_14918022-14918023, EPI_ISL_14918791, EPI_ISL_16935796-16935805, EPI_ISL_16951978, EPI_ISL_16951997, EPI_ISL_16954664, EPI_ISL_17071654, EPI_ISL_17071656, EPI_ISL_17071658, EPI_ISL_17074791-17074800, EPI_ISL_17215237-17215246, EPI_ISL_17371910-17371921, EPI_ISL_17423463-17423473, EPI_ISL_17446485-17446486, EPI_ISL_17446489-17446490, EPI_ISL_17446493, EPI_ISL_18266191-18266194, EPI_ISL_18266196-18266200, EPI_ISL_18266202-18266205, EPI_ISL_18323517-18323532, EPI_ISL_18333033-18333039, EPI_ISL_18356765, EPI_ISL_18405563-18405577, EPI_ISL_18458762-18458777, EPI_ISL_18486543-18486544, EPI_ISL_18585511-18585543, EPI_ISL_18585545-18585552, EPI_ISL_18585554-18585557, EPI_ISL_18675121, EPI_ISL_18675134, EPI_ISL_18677246, EPI_ISL_18852493-18852504")
#bronchoscopy_net = setdiff(bronchoscopy_seqs, bronchoalveolar_plus_lavage);          println(length(bronchoscopy_net))
#bronchoscopy_net_sort = sort(collect(bronchoscopy_net), by = x -> (length(x), x));   println(bronchoscopy_net_sort)
########################################################################################################################################################################
lung_seqs_raw = stringlist_to_set("EPI_ISL_412982, EPI_ISL_413459, EPI_ISL_417176-417185, EPI_ISL_417187-417188, EPI_ISL_417190, EPI_ISL_417193, EPI_ISL_417195, EPI_ISL_417197, EPI_ISL_417199, EPI_ISL_418815, EPI_ISL_419213-419219, EPI_ISL_419221-419229, EPI_ISL_419231-419232, EPI_ISL_419241-419253, EPI_ISL_419663-419664, EPI_ISL_420455, EPI_ISL_428355-428356, EPI_ISL_430844, EPI_ISL_436040, EPI_ISL_437198-437200, EPI_ISL_437299-437300, EPI_ISL_447886, EPI_ISL_451935, EPI_ISL_452140, EPI_ISL_452142, EPI_ISL_452148-452152, EPI_ISL_528752, EPI_ISL_577606, EPI_ISL_582003-582016, EPI_ISL_591059-591065, EPI_ISL_591341-591346, EPI_ISL_596233-596234, EPI_ISL_653916, EPI_ISL_660111-660118, EPI_ISL_660323, EPI_ISL_672575-672607, EPI_ISL_677723, EPI_ISL_678251, EPI_ISL_678253, EPI_ISL_738084, EPI_ISL_771371, EPI_ISL_850510, EPI_ISL_853819-853821, EPI_ISL_903312-903338, EPI_ISL_909744, EPI_ISL_930802, EPI_ISL_933804, EPI_ISL_954301, EPI_ISL_1010732-1010736, EPI_ISL_1013453, EPI_ISL_1013615, EPI_ISL_1014765, EPI_ISL_1046886-1046888, EPI_ISL_1061404-1061411, EPI_ISL_1140122, EPI_ISL_1140424, EPI_ISL_1153948, EPI_ISL_1204503, EPI_ISL_1213568, EPI_ISL_1404135, EPI_ISL_1424065, EPI_ISL_1469958-1469959, EPI_ISL_1469961, EPI_ISL_1472367, EPI_ISL_1588443-1588444, EPI_ISL_1588479-1588481, EPI_ISL_1588487-1588488, EPI_ISL_1588493, EPI_ISL_1588495-1588496, EPI_ISL_1588584, EPI_ISL_1593826, EPI_ISL_1593828, EPI_ISL_1593830, EPI_ISL_1593832-1593833, EPI_ISL_1593835, EPI_ISL_1751846, EPI_ISL_1751849-1751850, EPI_ISL_1752145-1752146, EPI_ISL_1789541, EPI_ISL_2094634, EPI_ISL_2095237-2095238, EPI_ISL_2107469-2107498, EPI_ISL_2155762-2155766, EPI_ISL_2156223-2156225, EPI_ISL_2156321, EPI_ISL_2156513-2156514, EPI_ISL_2156523, EPI_ISL_2156527, EPI_ISL_2285723, EPI_ISL_2350784, EPI_ISL_2378700-2378708, EPI_ISL_2438907, EPI_ISL_2438909, EPI_ISL_2438912, EPI_ISL_2438928, EPI_ISL_2438938-2438939, EPI_ISL_2438943, EPI_ISL_2438945, EPI_ISL_2438959-2438960, EPI_ISL_2438967-2438968, EPI_ISL_2476408, EPI_ISL_2491824-2491825, EPI_ISL_2491830, EPI_ISL_2621731, EPI_ISL_2652169, EPI_ISL_2652195, EPI_ISL_2652202-2652203, EPI_ISL_2726970-2726972, EPI_ISL_2726977, EPI_ISL_2727499-2727501, EPI_ISL_2727566, EPI_ISL_2827754, EPI_ISL_2839551, EPI_ISL_2859188, EPI_ISL_2859315, EPI_ISL_3298301, EPI_ISL_3298767, EPI_ISL_3418698, EPI_ISL_3541226, EPI_ISL_3541230-3541231, EPI_ISL_3690200, EPI_ISL_4045399, EPI_ISL_4237122, EPI_ISL_4710645, EPI_ISL_4712850, EPI_ISL_4741240, EPI_ISL_4741537, EPI_ISL_4741611, EPI_ISL_4882459, EPI_ISL_4885262-4885263, EPI_ISL_4917667, EPI_ISL_4951281, EPI_ISL_5540222, EPI_ISL_5540247, EPI_ISL_5540250, EPI_ISL_5540412, EPI_ISL_5540459, EPI_ISL_5540523, EPI_ISL_5540606, EPI_ISL_5541746, EPI_ISL_5542533, EPI_ISL_5542811, EPI_ISL_5542963, EPI_ISL_5542980, EPI_ISL_5542994, EPI_ISL_5543028, EPI_ISL_5543032, EPI_ISL_5543046-5543047, EPI_ISL_5543049, EPI_ISL_5543052, EPI_ISL_5543082, EPI_ISL_5543106, EPI_ISL_5543265, EPI_ISL_5543447, EPI_ISL_5543496, EPI_ISL_5544450, EPI_ISL_5547388, EPI_ISL_5548227, EPI_ISL_5548346, EPI_ISL_5548534, EPI_ISL_5549947, EPI_ISL_5550668, EPI_ISL_5553153, EPI_ISL_5553169, EPI_ISL_5553278, EPI_ISL_5553423, EPI_ISL_5553457, EPI_ISL_5566855, EPI_ISL_5568066, EPI_ISL_5568762, EPI_ISL_5569469, EPI_ISL_5569490, EPI_ISL_5570930, EPI_ISL_5778211, EPI_ISL_5782318-5782319, EPI_ISL_5782323, EPI_ISL_5782328-5782329, EPI_ISL_5782332, EPI_ISL_5782336, EPI_ISL_5782340, EPI_ISL_5782345, EPI_ISL_5782350, EPI_ISL_5782353, EPI_ISL_5782398, EPI_ISL_5782402, EPI_ISL_5782404, EPI_ISL_5926943, EPI_ISL_5926975, EPI_ISL_7999830, EPI_ISL_7999832-7999833, EPI_ISL_7999836-7999837, EPI_ISL_7999839, EPI_ISL_8050320, EPI_ISL_8142636, EPI_ISL_8142640, EPI_ISL_8142643, EPI_ISL_8350527, EPI_ISL_9057974, EPI_ISL_9214773, EPI_ISL_9321580, EPI_ISL_9321583-9321584, EPI_ISL_9321588, EPI_ISL_9321590, EPI_ISL_9321593, EPI_ISL_9321595, EPI_ISL_9321597, EPI_ISL_9321599, EPI_ISL_9321602, EPI_ISL_9321604, EPI_ISL_9321607, EPI_ISL_9321609, EPI_ISL_9321612, EPI_ISL_9321615, EPI_ISL_9321617, EPI_ISL_9321619, EPI_ISL_9321622, EPI_ISL_9321716, EPI_ISL_9321718, EPI_ISL_9321720, EPI_ISL_9321723, EPI_ISL_9321725, EPI_ISL_9321728, EPI_ISL_9321730, EPI_ISL_9321733, EPI_ISL_9321736, EPI_ISL_9321738, EPI_ISL_9321740, EPI_ISL_9321787, EPI_ISL_9324425-9324426, EPI_ISL_9670407, EPI_ISL_10064500, EPI_ISL_10065077, EPI_ISL_10066376, EPI_ISL_10066577, EPI_ISL_10066836, EPI_ISL_10070307, EPI_ISL_10070315, EPI_ISL_10070358, EPI_ISL_10070427, EPI_ISL_10070554, EPI_ISL_10070708-10070709, EPI_ISL_10070740, EPI_ISL_10070822, EPI_ISL_10071205, EPI_ISL_10071912, EPI_ISL_10071928, EPI_ISL_10072005, EPI_ISL_10072161, EPI_ISL_10072186, EPI_ISL_10072374, EPI_ISL_10072708-10072710, EPI_ISL_10072805, EPI_ISL_10072889, EPI_ISL_10073048, EPI_ISL_10073121, EPI_ISL_10073167, EPI_ISL_10073250, EPI_ISL_10073320, EPI_ISL_10073375, EPI_ISL_10073418-10073419, EPI_ISL_10073513, EPI_ISL_10200831, EPI_ISL_10673517-10673518, EPI_ISL_10910094, EPI_ISL_11050880, EPI_ISL_11229018-11229022, EPI_ISL_11523694, EPI_ISL_11750345, EPI_ISL_11984533, EPI_ISL_11984537-11984539, EPI_ISL_12051823, EPI_ISL_12252130, EPI_ISL_12476329, EPI_ISL_12509958, EPI_ISL_12509964-12509965, EPI_ISL_12509992, EPI_ISL_12517178, EPI_ISL_12721573-12721609, EPI_ISL_12721611-12721622, EPI_ISL_13145839-13145869, EPI_ISL_13171739, EPI_ISL_13331746, EPI_ISL_13503362, EPI_ISL_13850649-13850686, EPI_ISL_13925628, EPI_ISL_14146438, EPI_ISL_14146452, EPI_ISL_14176116, EPI_ISL_14371392, EPI_ISL_14570076-14570108, EPI_ISL_14584266-14584271, EPI_ISL_14696992, EPI_ISL_14862419-14862452, EPI_ISL_14999065, EPI_ISL_14999067, EPI_ISL_14999087-14999090, EPI_ISL_15003303, EPI_ISL_15003305, EPI_ISL_15003347, EPI_ISL_15003355, EPI_ISL_15003366, EPI_ISL_15005649, EPI_ISL_15306057, EPI_ISL_15358683, EPI_ISL_15494583-15494585, EPI_ISL_15577083-15577128, EPI_ISL_15596846, EPI_ISL_15668545, EPI_ISL_15669341, EPI_ISL_15669997, EPI_ISL_15670031-15670068, EPI_ISL_15698181-15698187, EPI_ISL_15698197-15698204, EPI_ISL_15736810, EPI_ISL_15775265-15775267, EPI_ISL_15780154-15780190, EPI_ISL_15811452-15811453, EPI_ISL_15812726-15812727, EPI_ISL_15819750-15819792, EPI_ISL_15988458, EPI_ISL_16017322-16017324, EPI_ISL_16036888, EPI_ISL_16046977-16047020, EPI_ISL_16098364, EPI_ISL_16109704, EPI_ISL_16109706, EPI_ISL_16137963-16137965, EPI_ISL_16157390-16157392, EPI_ISL_16238909, EPI_ISL_16368845-16368857, EPI_ISL_16368859-16368891, EPI_ISL_16449508-16449580, EPI_ISL_16465329-16465375, EPI_ISL_16488430, EPI_ISL_16548984-16549031, EPI_ISL_16674345, EPI_ISL_16858727, EPI_ISL_16858730, EPI_ISL_16858745, EPI_ISL_16858752, EPI_ISL_17033247, EPI_ISL_17037231-17037271, EPI_ISL_17082133, EPI_ISL_17089311-17089312, EPI_ISL_17089314, EPI_ISL_17198046, EPI_ISL_17244974-17244976, EPI_ISL_17244979-17244980, EPI_ISL_17244983, EPI_ISL_17374807, EPI_ISL_17509034, EPI_ISL_17509293, EPI_ISL_17509296, EPI_ISL_17509299, EPI_ISL_17509302, EPI_ISL_17509304, EPI_ISL_17509307, EPI_ISL_17509310, EPI_ISL_17509312, EPI_ISL_17509315, EPI_ISL_17509318, EPI_ISL_17509322, EPI_ISL_17509325, EPI_ISL_17509327, EPI_ISL_17509330, EPI_ISL_17509332, EPI_ISL_17509335, EPI_ISL_17509337, EPI_ISL_17509340, EPI_ISL_17509342, EPI_ISL_17509344, EPI_ISL_17509349, EPI_ISL_17509352, EPI_ISL_17509354, EPI_ISL_17509357, EPI_ISL_17509359, EPI_ISL_17509362, EPI_ISL_17509364, EPI_ISL_17509369, EPI_ISL_17509372, EPI_ISL_17509374, EPI_ISL_17509377, EPI_ISL_17509386, EPI_ISL_17509388, EPI_ISL_17509390, EPI_ISL_17509393, EPI_ISL_17509566, EPI_ISL_17509569, EPI_ISL_17509572, EPI_ISL_17509577, EPI_ISL_17509580, EPI_ISL_17509582, EPI_ISL_17509585, EPI_ISL_17509587, EPI_ISL_17509590, EPI_ISL_17509615, EPI_ISL_17509617, EPI_ISL_17509620, EPI_ISL_17555080-17555094, EPI_ISL_17642145-17642150, EPI_ISL_17642155, EPI_ISL_17642171-17642174, EPI_ISL_17729624-17729650, EPI_ISL_17730048-17730054, EPI_ISL_17736630-17736643, EPI_ISL_17859259, EPI_ISL_17977781, EPI_ISL_17980321-17980335, EPI_ISL_17980344-17980349, EPI_ISL_18060053, EPI_ISL_18060555, EPI_ISL_18062777, EPI_ISL_18062780, EPI_ISL_18062787, EPI_ISL_18062800-18062801, EPI_ISL_18062804, EPI_ISL_18062825-18062826, EPI_ISL_18062835-18062837, EPI_ISL_18069163, EPI_ISL_18184495, EPI_ISL_18305929, EPI_ISL_18334577-18334578, EPI_ISL_18334582, EPI_ISL_18353888, EPI_ISL_18353909, EPI_ISL_18354536-18354567, EPI_ISL_18704572-18704573, EPI_ISL_18760111-18760112, EPI_ISL_18926323, EPI_ISL_18926329, EPI_ISL_19057564, EPI_ISL_19166995, EPI_ISL_19167016, EPI_ISL_19220361, EPI_ISL_19287413, EPI_ISL_19516822, EPI_ISL_19588617, EPI_ISL_19588630, EPI_ISL_19742521, EPI_ISL_19807253, EPI_ISL_19808167, EPI_ISL_19808172, EPI_ISL_19878655, EPI_ISL_19913369-19913370, EPI_ISL_19974230, EPI_ISL_20252380")
lung_center_of_philippines = stringlist_to_set("EPI_ISL_430844, EPI_ISL_1213568, EPI_ISL_2155762-2155766, EPI_ISL_2156223-2156225, EPI_ISL_2156321, EPI_ISL_2156513-2156514, EPI_ISL_2156523, EPI_ISL_2156527, EPI_ISL_2859188, EPI_ISL_2859315, EPI_ISL_4712850, EPI_ISL_4741240, EPI_ISL_4741537, EPI_ISL_4741611, EPI_ISL_5540222, EPI_ISL_5540247, EPI_ISL_5540250, EPI_ISL_5540412, EPI_ISL_5540459, EPI_ISL_5540523, EPI_ISL_5540606, EPI_ISL_5541746, EPI_ISL_5542533, EPI_ISL_5542811, EPI_ISL_5542963, EPI_ISL_5542980, EPI_ISL_5542994, EPI_ISL_5543028, EPI_ISL_5543032, EPI_ISL_5543046-5543047, EPI_ISL_5543049, EPI_ISL_5543052, EPI_ISL_5543082, EPI_ISL_5543106, EPI_ISL_5543265, EPI_ISL_5543447, EPI_ISL_5543496, EPI_ISL_5544450, EPI_ISL_5547388, EPI_ISL_5548227, EPI_ISL_5548346, EPI_ISL_5548534, EPI_ISL_5549947, EPI_ISL_5550668, EPI_ISL_5553153, EPI_ISL_5553169, EPI_ISL_5553278, EPI_ISL_5553423, EPI_ISL_5553457, EPI_ISL_5566855, EPI_ISL_5568066, EPI_ISL_5568762, EPI_ISL_5569469, EPI_ISL_5569490, EPI_ISL_5570930, EPI_ISL_9321580, EPI_ISL_9321583-9321584, EPI_ISL_9321588, EPI_ISL_9321590, EPI_ISL_9321593, EPI_ISL_9321595, EPI_ISL_9321597, EPI_ISL_9321599, EPI_ISL_9321602, EPI_ISL_9321604, EPI_ISL_9321607, EPI_ISL_9321609, EPI_ISL_9321612, EPI_ISL_9321615, EPI_ISL_9321617, EPI_ISL_9321619, EPI_ISL_9321622, EPI_ISL_9321716, EPI_ISL_9321718, EPI_ISL_9321720, EPI_ISL_9321723, EPI_ISL_9321725, EPI_ISL_9321728, EPI_ISL_9321730, EPI_ISL_9321733, EPI_ISL_9321736, EPI_ISL_9321738, EPI_ISL_9321740, EPI_ISL_9321787, EPI_ISL_9324425-9324426, EPI_ISL_13171739, EPI_ISL_15494583-15494585, EPI_ISL_15698181-15698187, EPI_ISL_15698197-15698204, EPI_ISL_15812726-15812727, EPI_ISL_16017322-16017324, EPI_ISL_16157390-16157392, EPI_ISL_16488430, EPI_ISL_17089311-17089312, EPI_ISL_17089314, EPI_ISL_17244974-17244976, EPI_ISL_17244979-17244980, EPI_ISL_17244983, EPI_ISL_17642145-17642150, EPI_ISL_17642155, EPI_ISL_17642171-17642174, EPI_ISL_17859259, EPI_ISL_18354536-18354567")
india_TN = stringlist_to_set("EPI_ISL_14999065, EPI_ISL_14999067, EPI_ISL_14999087-14999090, EPI_ISL_15003303, EPI_ISL_15003305, EPI_ISL_15003347, EPI_ISL_15003355, EPI_ISL_15003366, EPI_ISL_1500564")
leslie_thian_lung_than = stringlist_to_set("EPI_ISL_10064500, EPI_ISL_10065077, EPI_ISL_10066376, EPI_ISL_10066577, EPI_ISL_10066836, EPI_ISL_10070307, EPI_ISL_10070315, EPI_ISL_10070358, EPI_ISL_10070427, EPI_ISL_10070554, EPI_ISL_10070708-10070709, EPI_ISL_10070740, EPI_ISL_10070822, EPI_ISL_10071205, EPI_ISL_10071912, EPI_ISL_10071928, EPI_ISL_10072005, EPI_ISL_10072161, EPI_ISL_10072186, EPI_ISL_10072374, EPI_ISL_10072708-10072710, EPI_ISL_10072805, EPI_ISL_10072889, EPI_ISL_10073048, EPI_ISL_10073121, EPI_ISL_10073167, EPI_ISL_10073250, EPI_ISL_10073320, EPI_ISL_10073375, EPI_ISL_10073418-10073419, EPI_ISL_10073513, EPI_ISL_12721573-12721609, EPI_ISL_12721611-12721622, EPI_ISL_13145839-13145869, EPI_ISL_13850649-13850686, EPI_ISL_14570076-14570108, EPI_ISL_14584266-14584271, EPI_ISL_14862419-14862452, EPI_ISL_15577083-15577128, EPI_ISL_15668545, EPI_ISL_15669341, EPI_ISL_15669997, EPI_ISL_15670031-15670068, EPI_ISL_15775265-15775267, EPI_ISL_15780154-15780190, EPI_ISL_15811452-15811453, EPI_ISL_15819750-15819792, EPI_ISL_16036888, EPI_ISL_16046977-16047020, EPI_ISL_16368845-16368857, EPI_ISL_16368859-16368891, EPI_ISL_16449508-16449580, EPI_ISL_16465329-16465375, EPI_ISL_16548984-16549031, EPI_ISL_17033247, EPI_ISL_17037231-17037271, EPI_ISL_17509034, EPI_ISL_17509293, EPI_ISL_17509296, EPI_ISL_17509299, EPI_ISL_17509302, EPI_ISL_17509304, EPI_ISL_17509307, EPI_ISL_17509310, EPI_ISL_17509312, EPI_ISL_17509315, EPI_ISL_17509318, EPI_ISL_17509322, EPI_ISL_17509325, EPI_ISL_17509327, EPI_ISL_17509330, EPI_ISL_17509332, EPI_ISL_17509335, EPI_ISL_17509337, EPI_ISL_17509340, EPI_ISL_17509342, EPI_ISL_17509344, EPI_ISL_17509349, EPI_ISL_17509352, EPI_ISL_17509354, EPI_ISL_17509357, EPI_ISL_17509359, EPI_ISL_17509362, EPI_ISL_17509364, EPI_ISL_17509369, EPI_ISL_17509372, EPI_ISL_17509374, EPI_ISL_17509377, EPI_ISL_17509386, EPI_ISL_17509388, EPI_ISL_17509390, EPI_ISL_17509393, EPI_ISL_17509566, EPI_ISL_17509569, EPI_ISL_17509572, EPI_ISL_17509577, EPI_ISL_17509580, EPI_ISL_17509582, EPI_ISL_17509585, EPI_ISL_17509587, EPI_ISL_17509590, EPI_ISL_17509615, EPI_ISL_17509617, EPI_ISL_17509620, EPI_ISL_17555080-17555094, EPI_ISL_17729624-17729650, EPI_ISL_17730048-17730054, EPI_ISL_17736630-17736643, EPI_ISL_17980321-17980335, EPI_ISL_17980344-17980349")
thian_lung = stringlist_to_set("EPI_ISL_5778211, EPI_ISL_5782318-5782319, EPI_ISL_5782323, EPI_ISL_5782328-5782329, EPI_ISL_5782332, EPI_ISL_5782336, EPI_ISL_5782340, EPI_ISL_5782345, EPI_ISL_5782350, EPI_ISL_5782353, EPI_ISL_5782398, EPI_ISL_5782402, EPI_ISL_5782404, EPI_ISL_10064500, EPI_ISL_10065077, EPI_ISL_10066376, EPI_ISL_10066577, EPI_ISL_10066836, EPI_ISL_10070307, EPI_ISL_10070315, EPI_ISL_10070358, EPI_ISL_10070427, EPI_ISL_10070554, EPI_ISL_10070708-10070709, EPI_ISL_10070740, EPI_ISL_10070822, EPI_ISL_10071205, EPI_ISL_10071912, EPI_ISL_10071928, EPI_ISL_10072005, EPI_ISL_10072161, EPI_ISL_10072186, EPI_ISL_10072374, EPI_ISL_10072708-10072710, EPI_ISL_10072805, EPI_ISL_10072889, EPI_ISL_10073048, EPI_ISL_10073121, EPI_ISL_10073167, EPI_ISL_10073250, EPI_ISL_10073320, EPI_ISL_10073375, EPI_ISL_10073418-10073419, EPI_ISL_10073513, EPI_ISL_12721573-12721609, EPI_ISL_12721611-12721622, EPI_ISL_13145839-13145869, EPI_ISL_13850649-13850686, EPI_ISL_14570076-14570108, EPI_ISL_14584266-14584271, EPI_ISL_14862419-14862452, EPI_ISL_15577083-15577128, EPI_ISL_15668545, EPI_ISL_15669341, EPI_ISL_15669997, EPI_ISL_15670031-15670068, EPI_ISL_15775265-15775267, EPI_ISL_15780154-15780190, EPI_ISL_15811452-15811453, EPI_ISL_15819750-15819792, EPI_ISL_16036888, EPI_ISL_16046977-16047020, EPI_ISL_16368845-16368857, EPI_ISL_16368859-16368891, EPI_ISL_16449508-16449580, EPI_ISL_16465329-16465375, EPI_ISL_16548984-16549031, EPI_ISL_17033247, EPI_ISL_17037231-17037271, EPI_ISL_17509034, EPI_ISL_17509293, EPI_ISL_17509296, EPI_ISL_17509299, EPI_ISL_17509302, EPI_ISL_17509304, EPI_ISL_17509307, EPI_ISL_17509310, EPI_ISL_17509312, EPI_ISL_17509315, EPI_ISL_17509318, EPI_ISL_17509322, EPI_ISL_17509325, EPI_ISL_17509327, EPI_ISL_17509330, EPI_ISL_17509332, EPI_ISL_17509335, EPI_ISL_17509337, EPI_ISL_17509340, EPI_ISL_17509342, EPI_ISL_17509344, EPI_ISL_17509349, EPI_ISL_17509352, EPI_ISL_17509354, EPI_ISL_17509357, EPI_ISL_17509359, EPI_ISL_17509362, EPI_ISL_17509364, EPI_ISL_17509369, EPI_ISL_17509372, EPI_ISL_17509374, EPI_ISL_17509377, EPI_ISL_17509386, EPI_ISL_17509388, EPI_ISL_17509390, EPI_ISL_17509393, EPI_ISL_17509566, EPI_ISL_17509569, EPI_ISL_17509572, EPI_ISL_17509577, EPI_ISL_17509580, EPI_ISL_17509582, EPI_ISL_17509585, EPI_ISL_17509587, EPI_ISL_17509590, EPI_ISL_17509615, EPI_ISL_17509617, EPI_ISL_17509620, EPI_ISL_17555080-17555094, EPI_ISL_17729624-17729650, EPI_ISL_17730048-17730054, EPI_ISL_17736630-17736643, EPI_ISL_17980321-17980335, EPI_ISL_17980344-17980349")
Cheung_LUNG = stringlist_to_set("EPI_ISL_417176-417185, EPI_ISL_417187-417188, EPI_ISL_417190, EPI_ISL_417193, EPI_ISL_417195, EPI_ISL_417197, EPI_ISL_417199, EPI_ISL_418815, EPI_ISL_419213-419219, EPI_ISL_419221-419229, EPI_ISL_419231-419232, EPI_ISL_419241-419253, EPI_ISL_420455, EPI_ISL_738084, EPI_ISL_930802, EPI_ISL_1046886-1046888, EPI_ISL_1061404-1061411, EPI_ISL_2378700-2378708, EPI_ISL_16137963-16137965") 
lung_biology_center_iowa = stringlist_to_set("EPI_ISL_903312-903338, EPI_ISL_1010732-1010736")
lung_institute = stringlist_to_set("EPI_ISL_18060053, EPI_ISL_18060555, EPI_ISL_18062777, EPI_ISL_18062780, EPI_ISL_18062787, EPI_ISL_18062800-18062801, EPI_ISL_18062804, EPI_ISL_18062825-18062826, EPI_ISL_18062835-18062837, EPI_ISL_18334577-18334578, EPI_ISL_18334582, EPI_ISL_18926323, EPI_ISL_18926329")
lung_hospital = stringlist_to_set("EPI_ISL_412982, EPI_ISL_528752, EPI_ISL_4885262-4885263, EPI_ISL_4951281, EPI_ISL_8050320, EPI_ISL_10673517-10673518, EPI_ISL_11750345, EPI_ISL_11984533, EPI_ISL_11984537-11984539, EPI_ISL_12509958, EPI_ISL_12509964-12509965, EPI_ISL_12509992")
other = stringlist_to_set("EPI_ISL_1061406, EPI_ISL_16137963, EPI_ISL_16137964, EPI_ISL_16137965, EPI_ISL_18704573, EPI_ISL_18704572, EPI_ISL_577606, EPI_ISL_582003-582016, EPI_ISL_591059-591065, EPI_ISL_591341-591346, EPI_ISL_660111-660118, EPI_ISL_672575-672607, EPI_ISL_2107469-2107498, EPI_ISL_850510, EPI_ISL_8350527, EPI_ISL_930802, EPI_ISL_5926943, EPI_ISL_5926975, EPI_ISL_10910094, EPI_ISL_10200831, EPI_ISL_9670407, EPI_ISL_8142643, EPI_ISL_8142636, EPI_ISL_8142640, EPI_ISL_16238909, EPI_ISL_14176116, EPI_ISL_14146452, EPI_ISL_14146438, EPI_ISL_13331746, EPI_ISL_12476329, EPI_ISL_12252130, EPI_ISL_12051823, EPI_ISL_11523694, EPI_ISL_1061411, EPI_ISL_1061410, EPI_ISL_1061407, EPI_ISL_1061405, EPI_ISL_3298767, EPI_ISL_3298301, EPI_ISL_8050320, EPI_ISL_11750345, EPI_ISL_12509992, EPI_ISL_11984539, EPI_ISL_11984538, EPI_ISL_11984537, EPI_ISL_11984533, EPI_ISL_10673518, EPI_ISL_10673517, EPI_ISL_12509965, EPI_ISL_12509964, EPI_ISL_12509958, EPI_ISL_850510, EPI_ISL_8350527, EPI_ISL_930802, EPI_ISL_11050880, EPI_ISL_10910094, EPI_ISL_1061409, EPI_ISL_1061408, EPI_ISL_1061404, EPI_ISL_677723, EPI_ISL_2839551, EPI_ISL_1140424, EPI_ISL_1140122, EPI_ISL_1014765, EPI_ISL_1013615, EPI_ISL_1013453, EPI_ISL_954301, EPI_ISL_933804, EPI_ISL_909744, EPI_ISL_678253, EPI_ISL_678251")
lung_excluded_seqs = union(thian_lung, bronchoalveolar_plus_lavage, lung_center_of_philippines, india_TN, leslie_thian_lung_than, Cheung_LUNG, lung_biology_center_iowa, lung_institute, lung_hospital, other)
lung_seqs_net = setdiff(lung_seqs_raw, lung_excluded_seqs)
println("lung_seqs_net length = $(length(lung_seqs_net))")
lung_seqs_net_sort = sort(collect(lung_seqs_net), by = x -> (length(x), x))
lung_seqs_net_sort_join = join(lung_seqs_net_sort, ", ")
### println(lung_seqs_net_sort_join); print("\n"^2)
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_plus_lavage_lung = union(bronchoalveolar_plus_lavage, lung_seqs_net)
#############################################################################################################################################################################
#############################################################################################################################################################################
println("bronchoalveolar_plus_lavage_lung Length = $(length(bronchoalveolar_plus_lavage_lung))")
bronchoalveolar_plus_lavage_lung_sort = sort(collect(bronchoalveolar_plus_lavage_lung), by = x -> (length(x), x))
bronchoalveolar_plus_lavage_lung_sort_join = join(bronchoalveolar_plus_lavage_lung_sort, ", ")
open("bronchoalveolar_plus_lavage_lung_2442seq.txt", "w") do g
    println(g, bronchoalveolar_plus_lavage_lung_sort_join)
end
########################################################################################################################################################################
lower_resp = Set{String}()
headers, seqs = read_fasta("Downloads/lower_respiratory_sample_65seq.fasta")
for head in headers
    epi = EPI_ISL(head)
    push!(lower_resp, epi)
end
println("lower resp length = $(length(lower_resp))")
lower_respiratory_seqs_sort = sort(collect(lower_resp), by = x -> (length(x), x))
lower_respiratory_seqs_sort_join = join(lower_respiratory_seqs_sort, ", ")
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_plus_lavage_lung_lowerresp = union(bronchoalveolar_plus_lavage_lung, lower_resp)
########################################################################################################################################################################
########################################################################################################################################################################
all_organ_seqs = stringlist_to_set("EPI_ISL_18541932, EPI_ISL_18541933, EPI_ISL_18541934, EPI_ISL_18541935, EPI_ISL_18541936, EPI_ISL_1404131, EPI_ISL_1404142, EPI_ISL_1404139, EPI_ISL_1404141, EPI_ISL_1404140, EPI_ISL_1404137, EPI_ISL_1404135, EPI_ISL_1404132, EPI_ISL_1404133, EPI_ISL_1404134, EPI_ISL_626620, EPI_ISL_693675, EPI_ISL_693685, EPI_ISL_737025, EPI_ISL_626605, EPI_ISL_626619, EPI_ISL_626621, EPI_ISL_660555, EPI_ISL_693669, EPI_ISL_693683, EPI_ISL_693687, EPI_ISL_737033, EPI_ISL_792686, EPI_ISL_882950, EPI_ISL_960408, EPI_ISL_960411, EPI_ISL_960412, EPI_ISL_960413, EPI_ISL_960414, EPI_ISL_10132959, EPI_ISL_10132973, EPI_ISL_10132974, EPI_ISL_10133020, EPI_ISL_10592851, EPI_ISL_10594629, EPI_ISL_10595993, EPI_ISL_10595994, EPI_ISL_10595995, EPI_ISL_10595996, EPI_ISL_10595997, EPI_ISL_10608644, EPI_ISL_10608667, EPI_ISL_11828464, EPI_ISL_11828532, EPI_ISL_11835786, EPI_ISL_11835883, EPI_ISL_12075511, EPI_ISL_12075532, EPI_ISL_12415974, EPI_ISL_12546617, EPI_ISL_13128169, EPI_ISL_13134475, EPI_ISL_13145208, EPI_ISL_13145289, EPI_ISL_13145670, EPI_ISL_13145676, EPI_ISL_13145678, EPI_ISL_13145679, EPI_ISL_13145695, EPI_ISL_13884731, EPI_ISL_14405091, EPI_ISL_14406098, EPI_ISL_14407282, EPI_ISL_14771639, EPI_ISL_14771643, EPI_ISL_14819882, EPI_ISL_15193619, EPI_ISL_15193680, EPI_ISL_15193681, EPI_ISL_15638292, EPI_ISL_15639166, EPI_ISL_15639210, EPI_ISL_1591137, EPI_ISL_17267008, EPI_ISL_17385066, EPI_ISL_18041830, EPI_ISL_18041876, EPI_ISL_18041889, EPI_ISL_18113358, EPI_ISL_1893515, EPI_ISL_1893517, EPI_ISL_1893525, EPI_ISL_1893527, EPI_ISL_1893529, EPI_ISL_1939487, EPI_ISL_1939488, EPI_ISL_1939489, EPI_ISL_1939490, EPI_ISL_1939516, EPI_ISL_1939517, EPI_ISL_1939526, EPI_ISL_1939527, EPI_ISL_1939528, EPI_ISL_1939529, EPI_ISL_1939544, EPI_ISL_1939592, EPI_ISL_1939600, EPI_ISL_1939601, EPI_ISL_1939616, EPI_ISL_19568025, EPI_ISL_1965119, EPI_ISL_1965134, EPI_ISL_1965135, EPI_ISL_1965136, EPI_ISL_1965137, EPI_ISL_1965138, EPI_ISL_1965163, EPI_ISL_1965164, EPI_ISL_1965188, EPI_ISL_1965196, EPI_ISL_1965222, EPI_ISL_1965223, EPI_ISL_1965224, EPI_ISL_1965225, EPI_ISL_1965226, EPI_ISL_1965227, EPI_ISL_1965228, EPI_ISL_1965229, EPI_ISL_1965230, EPI_ISL_1965231, EPI_ISL_1965278, EPI_ISL_1965283, EPI_ISL_1965284, EPI_ISL_1965285, EPI_ISL_1965286, EPI_ISL_1965287, EPI_ISL_1965302, EPI_ISL_1965303, EPI_ISL_1965304, EPI_ISL_1965305, EPI_ISL_1965306, EPI_ISL_1965307, EPI_ISL_1965308, EPI_ISL_1965309, EPI_ISL_1965319, EPI_ISL_1965320, EPI_ISL_1965321, EPI_ISL_1965323, EPI_ISL_1965324, EPI_ISL_1965325, EPI_ISL_1965326, EPI_ISL_1965327, EPI_ISL_1965328, EPI_ISL_1965329, EPI_ISL_1965364, EPI_ISL_1965367, EPI_ISL_1965368, EPI_ISL_1965369, EPI_ISL_1965370, EPI_ISL_1965393, EPI_ISL_1965394, EPI_ISL_1965395, EPI_ISL_1965396, EPI_ISL_1965397, EPI_ISL_1965398, EPI_ISL_1965399, EPI_ISL_1965400, EPI_ISL_1965401, EPI_ISL_1965402, EPI_ISL_1965403, EPI_ISL_1965413, EPI_ISL_1965414, EPI_ISL_1965437, EPI_ISL_2000568, EPI_ISL_2178676, EPI_ISL_2178688, EPI_ISL_2178689, EPI_ISL_2799307, EPI_ISL_2799978, EPI_ISL_2800044, EPI_ISL_2800048, EPI_ISL_3247404, EPI_ISL_4259453, EPI_ISL_4259454, EPI_ISL_4259460, EPI_ISL_4296123, EPI_ISL_4296140, EPI_ISL_4296147, EPI_ISL_4296160, EPI_ISL_4396785, EPI_ISL_4396786, EPI_ISL_4396815, EPI_ISL_4396851, EPI_ISL_4919845, EPI_ISL_4935949, EPI_ISL_5115041, EPI_ISL_5115043, EPI_ISL_5143360, EPI_ISL_5143428, EPI_ISL_5315214, EPI_ISL_5315224, EPI_ISL_5317645, EPI_ISL_5317669, EPI_ISL_5317691, EPI_ISL_5317732, EPI_ISL_5317733, EPI_ISL_5393375, EPI_ISL_5393406, EPI_ISL_5393408, EPI_ISL_5393409, EPI_ISL_5741250, EPI_ISL_5741299, EPI_ISL_5741300, EPI_ISL_5741308, EPI_ISL_5741378, EPI_ISL_5741570, EPI_ISL_5741589, EPI_ISL_5741590, EPI_ISL_5741601, EPI_ISL_5741665, EPI_ISL_5741677, EPI_ISL_5741742, EPI_ISL_5741743, EPI_ISL_5741746, EPI_ISL_5741747, EPI_ISL_5741751, EPI_ISL_5741799, EPI_ISL_5741880, EPI_ISL_5741881, EPI_ISL_5741902, EPI_ISL_5858611, EPI_ISL_5858612, EPI_ISL_5858613, EPI_ISL_5858614, EPI_ISL_5858621, EPI_ISL_5858622, EPI_ISL_6482016, EPI_ISL_6482017, EPI_ISL_6482118, EPI_ISL_6482250, EPI_ISL_6482585, EPI_ISL_6482589, EPI_ISL_6482595, EPI_ISL_6575065, EPI_ISL_6575072, EPI_ISL_6575073, EPI_ISL_6575075, EPI_ISL_6575084, EPI_ISL_6575104, EPI_ISL_6575115, EPI_ISL_6575123, EPI_ISL_6575125, EPI_ISL_7962591, EPI_ISL_7962592, EPI_ISL_7962633, EPI_ISL_7963001, EPI_ISL_7967350, EPI_ISL_7967353, EPI_ISL_7967383, EPI_ISL_7967415, EPI_ISL_7967416, EPI_ISL_7967428, EPI_ISL_7967429, EPI_ISL_7967431, EPI_ISL_7967433, EPI_ISL_7967434, EPI_ISL_7967457, EPI_ISL_7967458, EPI_ISL_7967460, EPI_ISL_7967461, EPI_ISL_7967478, EPI_ISL_7967529, EPI_ISL_7967556, EPI_ISL_7967567, EPI_ISL_7999577, EPI_ISL_8801846, EPI_ISL_9714277, EPI_ISL_9973860, EPI_ISL_9973863, EPI_ISL_13857993, EPI_ISL_1040814, EPI_ISL_1262459, EPI_ISL_1422562, EPI_ISL_15789169, EPI_ISL_17385090, EPI_ISL_17474998, EPI_ISL_2272558, EPI_ISL_2272579, EPI_ISL_2383045, EPI_ISL_3133566, EPI_ISL_3491798, EPI_ISL_8886131, EPI_ISL_9004081, EPI_ISL_9202280, EPI_ISL_9202281, EPI_ISL_982892, EPI_ISL_16389924, EPI_ISL_19726607, EPI_ISL_19726653, EPI_ISL_19739555, EPI_ISL_19739680, EPI_ISL_19872727")
organ_seqs_to_remove = stringlist_to_set("EPI_ISL_18541932, EPI_ISL_18541933, EPI_ISL_18541934, EPI_ISL_18541935, EPI_ISL_18541936, EPI_ISL_1040814, EPI_ISL_1262459, EPI_ISL_1422562, EPI_ISL_15789169, EPI_ISL_17385090, EPI_ISL_17474998, EPI_ISL_2272558, EPI_ISL_2272579, EPI_ISL_2383045, EPI_ISL_3133566, EPI_ISL_3491798, EPI_ISL_8886131, EPI_ISL_9004081, EPI_ISL_9202280, EPI_ISL_9202281, EPI_ISL_982892, EPI_ISL_16389924, EPI_ISL_19726607, EPI_ISL_19726653, EPI_ISL_19739555, EPI_ISL_19739680, EPI_ISL_19872727, EPI_ISL_13857993, EPI_ISL_1404131, EPI_ISL_1404142, EPI_ISL_1404139, EPI_ISL_1404141, EPI_ISL_1404140, EPI_ISL_1404137, EPI_ISL_1404135, EPI_ISL_1404132, EPI_ISL_1404133, EPI_ISL_1404134")
###############################################################
#println(length(all_organ_seqs))
#println(length(organ_seqs_to_remove))
###############################################################
net_organ_seqs = setdiff(all_organ_seqs, organ_seqs_to_remove)
net_organ_seqs_sort = sort(collect(net_organ_seqs), by = x -> (length(x), x))
net_organ_seqs_sort_join = join(net_organ_seqs_sort, ", ")
###############################################################
println("length_net_organ_seqs = $(length(net_organ_seqs))"); print("\n"^1)
#println(net_organ_seqs_sort_join)
open("BAL_organ_search_net_results_comma_list.txt", "w") do g
    println(g, net_organ_seqs_sort_join)
end
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_plus_lavage_lung_lowerresp_organ = union(bronchoalveolar_plus_lavage_lung_lowerresp, net_organ_seqs)
#############################################################################################################################################################################
#############################################################################################################################################################################
df_cerbaliance = CSV.read("Downloads/Cerbaliance_seqs_to_filter_from_BAL_search.csv", DataFrame)
df_antonin_bal = CSV.read("Downloads/Antonin_BAL_seqs_to_filter_from_BAL_search.csv", DataFrame)
df_BAL_search = CSV.read("Downloads/all_BAL_search_seqs.csv", DataFrame)
###############################################################
cerbaliance_seqs = df_cerbaliance.seqs
antonin_bal_seqs = df_antonin_bal.seqs
df_BAL_search_seqs = df_BAL_search.seqs
###############################################################
cristobal_seqs = stringlist_to_set("EPI_ISL_13766054-13766250, EPI_ISL_14901733-14901922, EPI_ISL_15932510-15932690")
other_BAL_search_exclude = stringlist_to_set("EPI_ISL_10016159, EPI_ISL_10661002, EPI_ISL_1922892, EPI_ISL_454858-454867, EPI_ISL_576117-576126, EPI_ISL_8764350, EPI_ISL_8881510-8881511")
###############################################################
print("\n"^1)
println("total cervaliance_seqs = $(length(cerbaliance_seqs))")
println("total antonin_bal_seqs = $(length(antonin_bal_seqs))")
println("total cristobal_seqs = $(length(cristobal_seqs))")
println("total other_BAL_search_exclude = $(length(other_BAL_search_exclude))")
println("total df_BAL_search_seqs = $(length(df_BAL_search_seqs))"); print("\n"^2)
###############################################################
cerbaliance_antonin_bal_seqs = union(cerbaliance_seqs, antonin_bal_seqs)
cerbaliance_antonin_bal_cristobal_other_bronchoalveolar_plus_lavage_lung_seqs = union(bronchoalveolar_plus_lavage_lung, cerbaliance_antonin_bal_seqs, cristobal_seqs, other_BAL_search_exclude)
###############################################################
net_BAL_search_seqs = setdiff(df_BAL_search_seqs, cerbaliance_antonin_bal_cristobal_other_bronchoalveolar_plus_lavage_lung_seqs)
tot_net_BAL_search_seqs = length(net_BAL_search_seqs)
println("Total Net BAL search seqs = $(tot_net_BAL_search_seqs)"); print("\n"^1)
net_BAL_search_seqs_sort = sort(collect(net_BAL_search_seqs), by = x -> (length(x), x))
net_BAL_search_seqs_sort_join = join(net_BAL_search_seqs_sort, ", ")
###############################################################
open("net_BAL_search_seqs_v4.txt", "w") do g
    println(g, net_BAL_search_seqs_sort_join)
end
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_lavage_lung_lowerresp_organ_BAL = union(bronchoalveolar_plus_lavage_lung_lowerresp_organ, net_BAL_search_seqs) 
#############################################################################################################################################################################
#############################################################################################################################################################################
###################################### Tracheal stuff ##########################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
date = Dates.format(today(), "yyyy_mm_dd")
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
################################################################################################
endotracheal_search_seqs = Set{String}()
hedz, sekz = read_fasta("Downloads/endotracheal_GISAID_search_109seq.fasta")
for hed in hedz
    epi = EPI_ISL(hed)
    push!(endotracheal_search_seqs, epi)
end
endotracheal_len = length(endotracheal_search_seqs)
println("endotracheal seqs = $(endotracheal_len)"); print("\n"^1)
################################################################################################
tracheal_search_seqs = Set{String}()
heads, seqs = read_fasta("Downloads/tracheal_GISAID_search_594seq.fasta")
for head in heads
    epi = EPI_ISL(head)
    push!(tracheal_search_seqs, epi)
end
tracheal_len = length(tracheal_search_seqs)
println("tracheal seqs = $(tracheal_len)"); print("\n"^1)
################################################################################################
trachea_search_seqs = Set{String}()
heeds, seeqs = read_fasta("Downloads/trachea_GISAID_search_185seq.fasta")
for heed in heeds
    epi = EPI_ISL(heed)
    push!(trachea_search_seqs, epi)
end
trachea_len = length(trachea_search_seqs)
println("trachea seqs = $(trachea_len)"); print("\n"^1)
################################################################################################
trach_aspirate_search_seqs = Set{String}()
hidz, siqs = read_fasta("Downloads/trach_aspirate_GISAID_search_11seq.fasta")
for hid in hidz
    epi = EPI_ISL(hid)
    push!(trach_aspirate_search_seqs, epi)
end
trach_aspirate_len = length(trach_aspirate_search_seqs)
println("trach aspirate seqs = $(trach_aspirate_len)"); print("\n"^1)
##########################################################################################################################################################
tracheal_endotracheal_trachea_trachAspirate_set = union(tracheal_search_seqs, endotracheal_search_seqs, trachea_search_seqs, trach_aspirate_search_seqs)
trachea_sum_length = length(tracheal_endotracheal_trachea_trachAspirate_set)
println("length all trachea-sample seqs = $(trachea_sum_length)"); print("\n"^1)
##########################################################################################################################################################
tracheal_excluded = stringlist_to_set("EPI_ISL_16389924, EPI_ISL_1040814, EPI_ISL_1262459, EPI_ISL_1422562, EPI_ISL_15789169, EPI_ISL_17385090, EPI_ISL_17474998, EPI_ISL_2272558, EPI_ISL_2272579, EPI_ISL_2383045, EPI_ISL_3133566, EPI_ISL_3491798, EPI_ISL_8886131, EPI_ISL_9004081, EPI_ISL_9202280, EPI_ISL_9202281, EPI_ISL_982892")
net_tracheal_seqs = setdiff(tracheal_endotracheal_trachea_trachAspirate_set, tracheal_excluded)
net_tracheal_len = length(net_tracheal_seqs)
println("length net trachea-sample seqs = $(net_tracheal_len)"); print("\n"^2)
##########################################################################################################################################################
net_tracheal_seqs_sort = sort(collect(net_tracheal_seqs), by = x -> (length(x), x))
net_tracheal_seqs_sort_join = join(net_tracheal_seqs_sort, ", ")
## println(net_tracheal_seqs_sort_join); print("\n"^2)
open("net_tracheal_seq_epi_list_$(date_now).txt", "w") do g
    println(g, net_tracheal_seqs_sort_join); print("\n"^2)
end
#############################################################################################################################################################################
#############################################################################################################################################################################
bronchoalveolar_lavage_lung_lowerresp_organ_BAL_tracheal = union(bronchoalveolar_lavage_lung_lowerresp_organ_BAL, net_tracheal_seqs)
excluded_2019_seqs = stringlist_to_set("EPI_ISL_403930, EPI_ISL_406799, EPI_ISL_402121, EPI_ISL_406798, EPI_ISL_402123, EPI_ISL_402132, EPI_ISL_529213, EPI_ISL_529217, EPI_ISL_402128, EPI_ISL_402119, EPI_ISL_529215, EPI_ISL_402129, EPI_ISL_402127, EPI_ISL_402130, EPI_ISL_403931, EPI_ISL_529214, EPI_ISL_529216, EPI_ISL_434534, EPI_ISL_412898, EPI_ISL_403929, EPI_ISL_402124")
final_BAL_start_list = setdiff(bronchoalveolar_lavage_lung_lowerresp_organ_BAL_tracheal, excluded_2019_seqs)
##############################################################
final_BAL_start_list_len = length(final_BAL_start_list)
println("final_BAL_start_list_len = $(final_BAL_start_list_len)"); print("\n"^2)
##############################################################
final_BAL_start_list_sort = sort(collect(final_BAL_start_list), by = x -> (length(x), x))
final_BAL_start_list_sort_join = join(final_BAL_start_list_sort, ", ")
open("final_BAL_start_list_$(date_now).txt", "w") do g
    println(g, final_BAL_start_list_sort_join)
end
#############################################################################################################################################################################
#############################################################################################################################################################################

bronchoalveolar_seqs length = 1513

lavage_net_seqs length = 755
bronchoalveolar_plus_lavage length = 2268

lung_seqs_net length = 159
bronchoalveolar_plus_lavage_lung Length = 2427
lower resp length = 65
length_net_organ_seqs = 259


total cervaliance_seqs = 1290
total antonin_bal_seqs = 134099
total cristobal_seqs = 568
total other_BAL_search_exclude = 26
total df_BAL_search_seqs = 136042


Total Net BAL search seqs = 258

2025-12-18__834PM

endotracheal seqs = 109

tracheal seqs = 594

trachea seqs = 185

trach aspirate seqs = 11

length all trachea-sample seqs = 899

length net trachea-sample seqs = 882




final_BAL_start_list_len = 3866




In [155]:
##### Code to remove 2019 BAL sequences (which the main-function is not designed to deal with)
exclude_2019 = Set{String}()
heads, seqs = read_fasta("bronchoalveolar_plus_lavage_lung_lower_resp_2498seq_v2.fasta")
for head in heads
    if length(string(split(head, "|")[2])) > 3
        epi = EPI_ISL(head)
        if length(string(split(head, "|")[3])) > 3
            date = sequence_date(head)
            if date[1:4] == "2019"
                push!(exclude_2019, epi)
                println(epi)
            end
        end
    else
        println("Bad Name = $(head)")
    end
end; print("\n"^2)
#### bronchoalveolar_plus_lavage_lung_lower_resp_2507seq.ndjson
########################################################################################################################################################################
########################################################################################################################################################################

EPI_ISL_403930
EPI_ISL_406799
EPI_ISL_402121
EPI_ISL_406798
EPI_ISL_402123
EPI_ISL_402132
EPI_ISL_529213
EPI_ISL_529217
EPI_ISL_402128
EPI_ISL_402119
EPI_ISL_529215
EPI_ISL_402129
EPI_ISL_402127
EPI_ISL_402130
EPI_ISL_403931
EPI_ISL_529214
EPI_ISL_529216
EPI_ISL_434534
EPI_ISL_412898
EPI_ISL_403929
EPI_ISL_402124




In [376]:
##### Finding ORF7a stop codon mutation sequences
##################################################
function list_to_strings_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
###################################################
BAL_muts_106 = list_to_strings_set("ORF1a:T2183I, ORF1a:S2193F, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3023V, ORF1a:T3032I, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:V3077A, ORF1a:F3089L, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:K3365R, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:K4176R, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:A1302V, ORF1b:T1424I, ORF1b:I2147T, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:T274I, S:P330S, S:N354D, S:V367F, S:Y369H, S:A372-, S:A372T, S:F374L, S:F375-, S:A376V, S:N405S, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659A, S:S659P, S:A668V, S:S691F, S:I693V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:I1183T, S:L1186F, E:V5A, E:V5F, E:S6L, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:L19I, E:L27S, E:T30I, E:A32V, E:A36V, E:Y42C, E:I46V, E:L51F, E:S68F, E:P71L, M:A2V, M:N3I, M:N3K, M:S4P, M:L17F, M:F28S, M:T77N, M:G78S, M:S94R, M:S94N, M:H125Y, M:H148R, M:S173P, M:A188T, M:G189S, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*")
ORF7a_stop_seqs = ["EPI_ISL_16630261", "EPI_ISL_19369553", "EPI_ISL_10128185", "EPI_ISL_1372288", "EPI_ISL_19819753", "EPI_ISL_14766363", "EPI_ISL_19621085", "EPI_ISL_18222367", "EPI_ISL_3358574", "EPI_ISL_3045809", "EPI_ISL_17347577", "EPI_ISL_3982251", "EPI_ISL_18118264", "EPI_ISL_19198259"]
ORF7a_stop_seqs_len = length(ORF7a_stop_seqs)
ORF7a_stop_seqs_BAL_ct = 0
for seq in ORF7a_stop_seqs
    for mut in seq_AA_muts[seq]
        if mut in BAL_muts_106
            ORF7a_stop_seqs_BAL_ct += 1
        end
    end
end
avg_BAL_muts_ORF7a_stop_seqs = round(digits=2, ORF7a_stop_seqs_BAL_ct/ORF7a_stop_seqs_len)
println("Avg BAL muts in ORF7a Stop-Codon-Mutated Seqs = $(avg_BAL_muts_ORF7a_stop_seqs)"); println()
all_seqs_BAL_muts_ct = 0
all_unique_chr_seqs_length = length(all_unique_chr_seqs)
for seq in all_unique_chr_seqs
    for mut in seq_AA_muts[seq]
        if mut in BAL_muts_106
            all_seqs_BAL_muts_ct += 1
        end
    end
end
Avg_BAL_muts_in_all_unique_chr_seqs = round(digits=2, all_seqs_BAL_muts_ct/all_unique_chr_seqs_length)
println("Avg BAL muts in all_unique_chr_seqs = $(Avg_BAL_muts_in_all_unique_chr_seqs)"); println()
#############################################################################################################################################################################
#############################################################################################################################################################################

Avg BAL muts in ORF7a Stop-Codon-Mutated Seqs = 3.79

Avg BAL muts in all_unique_chr_seqs = 1.06



In [34]:
######## Finding ORF7a stop-mutation sequences in EPCI sequences
ORF7a_stop_seqs = ["EPI_ISL_16630261", "EPI_ISL_19369553", "EPI_ISL_10128185", "EPI_ISL_1372288", "EPI_ISL_19819753", "EPI_ISL_14766363", "EPI_ISL_19621085", "EPI_ISL_18222367", "EPI_ISL_3358574", "EPI_ISL_3045809", "EPI_ISL_17347577", "EPI_ISL_3982251", "EPI_ISL_18118264", "EPI_ISL_19198259"]
stops7a_ct_unique = 0
for seq in ORF7a_stop_seqs
    if seq in all_unique_chr_seqs
        stops7a_ct_unique += 1
    end
end
println("Total ORF7a stops seqs in all_unique_chr_seqs = $(stops7a_ct_unique)")
#############################################################################################################################################################################
#############################################################################################################################################################################

Total ORF7a stops seqs in all_unique_chr_seqs = 14
